# GPT-2 from scratch: a shape-first implementation

**What this notebook is for.** You already know the transformer maths. This notebook attacks the *other* gap: the ability to look at a formula and immediately see the tensor that implements it — its shape, its memory layout, its cost.

**The method, used everywhere below, is always three beats:**

> **1. Trace it by hand.** Write the shapes, and where it's small enough, the actual numbers.
> **2. Write the code** that must follow from that trace.
> **3. Assert** that PyTorch agrees with the hand calculation.

Step 3 is what wires the intuition in. Guessing a shape and having it confirmed by an `assert` teaches more than reading ten explanations.

**A rule I've tried to hold to:** nothing is *named* without being *explained*. If you see a term like `scatter_add` or "log-sum-exp trick" or "arithmetic intensity", the text right around it derives what it means and why it's there. Naming a concept is not explaining it. Where I catch myself having asserted something instead of deriving it, there's a cell that proves it.

**How to run it.** Top to bottom, one cell at a time. Cells share state. Read the markdown, *predict the shape before you run the cell*, then run it and check. The predicting is the exercise.

| Part | Content |
|---|---|
| 0 | The shape contract, notation, config |
| 1 | Data: what a `(B, T)` batch really is |
| 2 | Embeddings: lookup tables, and their sparse gradient |
| 3 | LayerNorm: which axis, and why not the others |
| 4 | **Attention**: full hand trace, `view`/`transpose`, masking, batched matmul |
| 5 | MLP: why 4x, what GELU actually is |
| 6 | Block: the residual stream, and why pre-LN trains deep |
| 7 | The whole model: weight tying, init, vocab padding |
| 8 | Loss: `cross_entropy`, and the `ln(V)` sanity check |
| 9 | Training: AdamW, cosine LR, bf16, grad accumulation, the `6N` rule |
| 10 | Inference: sampling, and the KV cache |
| 11 | Cheat sheet + exercises |

**Requirements:** `torch>=2.0`, `numpy`, `tiktoken`, `datasets`, `tqdm`, `matplotlib`. Parts 0–8 run on CPU. Part 9 (training) wants one A100; Part 10 works either way.

---
# Part 0 — The shape contract

## 0.1 Notation

Six letters carry this entire notebook. Learn them now — every comment in every code cell uses them.

| Symbol | Stands for | Meaning | Teaching value | Training value |
|---|---|---|---|---|
| `B` | batch | how many independent sequences at once | 1–2 | 64 |
| `T` | time | how many tokens in each sequence | 3–4 | 256 |
| `C` | channels | how many numbers represent one token (`n_embd`) | 4 | 384 |
| `nh` | num heads | how many attention heads | 2 | 6 |
| `hs` | head size | `C // nh` — channels per head | 2 | 64 |
| `V` | vocab | how many distinct tokens exist | 8 | 50304 |
| `L` | layers | how many transformer blocks stacked | 1 | 6 |

Two of these are derived, not chosen: `hs = C // nh` always, and `C = nh * hs` always. If you remember one relationship, remember that the heads **split the channels** — they don't add any.

## 0.2 The one habit that fixes everything

Every line that touches a tensor gets a shape comment. Not most lines. Every line.

```python
qkv = self.c_attn(x)                                # (B, T, 3C)
q, k, v = qkv.split(C, dim=2)                       # 3 x (B, T, C)
q = q.view(B, T, nh, hs).transpose(1, 2)            # (B, nh, T, hs)
att = q @ k.transpose(-2, -1)                       # (B, nh, T, T)
```

This isn't documentation for a reader. It's a running proof for **you**. Here's the trick: if you can't write a line's shape comment without stopping to think, that is precisely the line that has the bug. The comment forces the trace to happen *before* the code — which is the entire skill this notebook is trying to build.

## 0.3 The three shapes that matter

Almost every tensor in a transformer is one of these three. Hold them and you can rebuild GPT-2 from memory.

**1. `(B, T, C)` — the residual stream.** The main bus. Every block reads it and writes back into it. Its shape *never changes* from the embedding output to the final LayerNorm. Every parameter in the model exists to compute something to add into this tensor.

**2. `(B, nh, T, T)` — the attention matrix.** The only tensor that is **quadratic in `T`**. It's where the memory goes, and the entire reason FlashAttention exists.

**3. `(B, T, V)` — the logits.** Only at the very end. With `V = 50304` this is by far the largest tensor in the forward pass — bigger than the model's parameters and activations combined.

Everything else is a `Linear` or a reshape.

In [ ]:
import math
import os
import time
import inspect
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)
np.random.seed(1337)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device_type = 'cuda' if 'cuda' in device else 'cpu'

print(f"torch   : {torch.__version__}")
print(f"device  : {device}")
if device_type == 'cuda':
    print(f"gpu     : {torch.cuda.get_device_name(0)}")
    print(f"memory  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"bf16 ok : {torch.cuda.is_bf16_supported()}")

# The helper we lean on constantly. Print shapes; never guess them.
def sh(name, t):
    print(f"{name:<28} {tuple(t.shape)}")

print("\n[ready]  sh(name, tensor) prints a shape. Use it obsessively.")

torch   : 2.11.0+cpu
device  : cpu

[ready]  sh(name, tensor) prints a shape. Use it obsessively.


In [ ]:
@dataclass
class GPTConfig:
    block_size: int = 256      # T_max: the longest sequence the model can ever see
    vocab_size: int = 50304    # V: 50257 rounded up to a multiple of 64 (Part 7 explains why)
    n_layer: int = 6           # L: how many blocks
    n_head: int = 6            # nh: how many attention heads
    n_embd: int = 384          # C: the residual stream width -> hs = 384 // 6 = 64
    dropout: float = 0.0
    bias: bool = False         # bias terms in Linear/LayerNorm. GPT-2 had them; we don't.

# MICRO exists only so we can print whole tensors and check them against paper.
MICRO = GPTConfig(block_size=8, vocab_size=8, n_layer=1, n_head=2, n_embd=4,
                  dropout=0.0, bias=False)
REAL = GPTConfig()

for name, cfg in [("MICRO", MICRO), ("REAL ", REAL)]:
    hs = cfg.n_embd // cfg.n_head
    print(f"{name}: C={cfg.n_embd:<4} nh={cfg.n_head}  ->  hs = C // nh = "
          f"{cfg.n_embd} // {cfg.n_head} = {hs}")
    print(f"        check: nh * hs = {cfg.n_head} * {hs} = {cfg.n_head*hs} = C  "
          f"{'[OK]' if cfg.n_head*hs == cfg.n_embd else '[MISMATCH]'}")

print("\nThe heads SPLIT the channels. They never add any. That constraint is why")
print("n_embd must be divisible by n_head, and it is asserted in the module.")

MICRO: C=4    nh=2  ->  hs = C // nh = 4 // 2 = 2
        check: nh * hs = 2 * 2 = 4 = C  [OK]
REAL : C=384  nh=6  ->  hs = C // nh = 384 // 6 = 64
        check: nh * hs = 6 * 64 = 384 = C  [OK]

The heads SPLIT the channels. They never add any. That constraint is why
n_embd must be divisible by n_head, and it is asserted in the module.


---
# Part 1 — Data: what a `(B, T)` batch actually is

## 1.1 From a string to `x` and `y`

Take the string `"Once upon a time there"`. GPT-2's BPE tokenizer turns it into integers:

```
"Once upon a time there"
        |
        v  tiktoken BPE
[7454, 2402, 257, 640, 612]
```

Now the part that trips everyone up. We do **not** build one training example from this. We build **five** — one per position — and one forward pass computes all five at once.

```
tokens :  7454   2402    257    640    612    11
           |      |       |      |      |
x      : [7454,  2402,   257,   640]              <- shape (T,) = (4,)
y      : [       2402,   257,   640,   612]       <- shape (T,) = (4,), x shifted left by 1
```

Read that as four independent (input → target) pairs:

```
  position 0 sees [7454]                    -> must predict 2402
  position 1 sees [7454, 2402]              -> must predict 257
  position 2 sees [7454, 2402, 257]         -> must predict 640
  position 3 sees [7454, 2402, 257, 640]    -> must predict 612
```

So `x` and `y` **have the same shape**. `y[i]` is the answer for the prefix ending at `x[i]`. That's the entire reason the code is `y = data[i+1 : i+1+T]` and not something cleverer.

Stack `B` of these rows and you have the batch:

```
x : (B, T)   int64
y : (B, T)   int64
```

### Why this matters more than it looks

A batch of shape `(64, 256)` is not 64 training examples. It is **16,384** of them — every position in every row is a separate supervised prediction, and all of them get gradients from one forward pass.

That density is why next-token prediction beats every other self-supervised objective on compute efficiency. And it is **purchased entirely by the causal mask** in Part 4. If the mask were broken, position 0 would be able to see the answer sitting at position 1, and 16,383 of those examples would be free lunch. Your loss would look wonderful and your model would be worthless.

Hold onto that. It's why Part 8's `ln(V)` check exists.

In [ ]:
!pip install tiktoken datasets tqdm -q
import tiktoken

enc = tiktoken.get_encoding("gpt2")
text = "Once upon a time there"
ids = enc.encode(text)

print(f"text        : {text!r}")
print(f"ids         : {ids}")
print(f"piece by piece:")
for i in ids:
    print(f"   {i:>6}  ->  {enc.decode([i])!r}")
print(f"\nvocab size  : {enc.n_vocab:,}")

# Build x and y exactly as traced above
t = torch.tensor(ids + [11], dtype=torch.long)   # append the token for '.'
T = 4
x = t[0:T]
y = t[1:T+1]
print(f"\nfull tokens : {t.tolist()}")
print(f"x = t[0:{T}]   : {x.tolist()}   shape {tuple(x.shape)}")
print(f"y = t[1:{T+1}]   : {y.tolist()}   shape {tuple(y.shape)}   <- SAME shape, shifted by 1")
print()
for i in range(T):
    ctx = x[:i+1].tolist()
    print(f"  position {i}: sees {str(ctx):<28} -> must predict {y[i].item():<6}"
          f" ({enc.decode(ctx)!r} -> {enc.decode([y[i].item()])!r})")
print(f"\nOne sequence of {T} tokens = {T} training examples, not 1.")

text        : 'Once upon a time there'
ids         : [7454, 2402, 257, 640, 612]
piece by piece:
     7454  ->  'Once'
     2402  ->  ' upon'
      257  ->  ' a'
      640  ->  ' time'
      612  ->  ' there'

vocab size  : 50,257

full tokens : [7454, 2402, 257, 640, 612, 11]
x = t[0:4]   : [7454, 2402, 257, 640]   shape (4,)
y = t[1:5]   : [2402, 257, 640, 612]   shape (4,)   <- SAME shape, shifted by 1

  position 0: sees [7454]                       -> must predict 2402   ('Once' -> ' upon')
  position 1: sees [7454, 2402]                 -> must predict 257    ('Once upon' -> ' a')
  position 2: sees [7454, 2402, 257]            -> must predict 640    ('Once upon a' -> ' time')
  position 3: sees [7454, 2402, 257, 640]       -> must predict 612    ('Once upon a time' -> ' there')

One sequence of 4 tokens = 4 training examples, not 1.


## 1.2 The dataset: TinyStories

TinyStories (Eldan & Li, 2023) is ~2.1M short stories written by GPT-3.5/4 using only vocabulary a 3–4 year old would know.

It's the right choice here for one specific reason: **a 30M-parameter model trained on it produces fluent, grammatical, coherent English.** Train that same 30M model on OpenWebText and you get plausible-looking garbage. Because TinyStories' distribution is narrow, real capability shows up at a scale that fits in one A100-hour — so you get an actual feedback loop instead of a vibe.

- ~2.1M stories, ~470 MB of text
- ~373M GPT-2 BPE tokens
- One epoch of our config: ~25 minutes on an A100

## 1.3 The storage format, and the words in it

Pretraining data is stored as **one flat array of token ids**. No per-example structure. No padding. No attention masks. Documents are concatenated end-to-end with an `<|endoftext|>` separator, and batches are cut out at random offsets.

```
train.bin:  [ story 1 ][50256][ story 2 ][50256][ story 3 ][50256] ...
             ^-- one contiguous array of np.uint16, ~373M entries, ~746 MB on disk
```

Three deliberate choices. Let me actually explain each one rather than just listing it.

### `uint16`, not `int32`

`uint16` means "unsigned 16-bit integer" — 2 bytes, holding any value from `0` to `65535`.

GPT-2's largest token id is `50256`. That's less than `65535`, so **every token id fits in 2 bytes.** Using `int32` (4 bytes) would double the file size and double the memory traffic for zero benefit.

This is worth noticing as a real constraint, not a trick: the moment your vocabulary exceeds 65,536 — which many multilingual tokenizers do — this breaks and you must go to `uint32`. The file doubles. That's a genuine cost of large vocabularies.

### `np.memmap`, not `np.load` — and what "paging" means

`np.load` reads the entire file into RAM. A 746 MB file means 746 MB of RAM. A 200 GB file means you're out of luck.

`np.memmap` does something different. It asks the operating system to **map the file into your process's address space** — meaning the OS hands you what looks like a normal array, but no data has actually been read yet.

Here's the mechanism, because it's worth understanding rather than accepting:

> Memory is managed in fixed-size chunks called **pages**, usually 4 KB. When you touch `data[5_000_000]`, the CPU discovers that the page holding that element isn't in physical RAM. It raises a **page fault**. The OS catches it, reads *just that 4 KB page* from disk, drops it in RAM, and lets your read continue. You never see any of this happen.

So the OS loads only the pages you actually touch, on demand. Reading a `block_size`-long slice touches a handful of pages. Your data can be 200 GB and your RAM 16 GB and it simply works.

The consequence: **no DataLoader, no worker processes, no `__getitem__`, no collate function.** The OS's virtual memory system is your data pipeline, and it's written in C and has been tuned for forty years.

### No padding, no document boundaries

A `(B, T)` window may straddle two stories — start in the middle of one and end in the next. This is fine and intentional:

- It's a fraction of a percent of windows.
- The `<|endoftext|>` token teaches the model to reset when it sees one.
- In exchange, **every single token in the batch is a real supervised target.**

Padding would mean burning compute on positions that teach nothing. At pretraining scale that's an unacceptable waste, so nobody does it.

In [ ]:
# ============================================================================
# Data prep. Run once (~5-10 min with 8 procs). Writes train.bin / val.bin.
# Set MAX_DOCS to something like 50_000 for a fast smoke test first.
# ============================================================================
DATA_DIR = "data/tinystories"
MAX_DOCS = None          # None = full dataset (~373M tokens). 50_000 -> ~9M tokens.
NUM_PROC = 8

def prepare_tinystories():
    import numpy as np
    from datasets import load_dataset
    from tqdm.auto import tqdm

    os.makedirs(DATA_DIR, exist_ok=True)
    if all(os.path.exists(f"{DATA_DIR}/{s}.bin") for s in ("train", "val")):
        print("bins already exist, skipping")
        return

    ds = load_dataset("roneneldan/TinyStories", num_proc=NUM_PROC)
    if MAX_DOCS is not None:
        ds["train"] = ds["train"].select(range(min(MAX_DOCS, len(ds["train"]))))
        ds["validation"] = ds["validation"].select(
            range(min(MAX_DOCS // 100 + 1, len(ds["validation"]))))
    print(ds)

    enc = tiktoken.get_encoding("gpt2")

    def tokenize(example):
        ids = enc.encode_ordinary(example["text"])   # ordinary = no special tokens
        ids.append(enc.eot_token)                    # 50256, the document separator
        return {"ids": ids, "len": len(ids)}

    tokenized = ds.map(tokenize, remove_columns=["text"], desc="tokenizing",
                       num_proc=NUM_PROC)

    for split, dset in tokenized.items():
        name = "val" if split == "validation" else "train"
        arr_len = np.sum(dset["len"], dtype=np.uint64)
        filename = f"{DATA_DIR}/{name}.bin"
        # mode='w+' creates the file and maps it. We write THROUGH the mapping.
        arr = np.memmap(filename, dtype=np.uint16, mode="w+", shape=(arr_len,))
        total_batches = min(1024, len(dset))
        idx = 0
        for i in tqdm(range(total_batches), desc=f"writing {filename}"):
            batch = dset.shard(num_shards=total_batches, index=i,
                               contiguous=True).with_format("numpy")
            arr_batch = np.concatenate(batch["ids"])
            arr[idx: idx + len(arr_batch)] = arr_batch
            idx += len(arr_batch)
        arr.flush()      # force the OS to write dirty pages back to disk
        print(f"{filename}: {arr_len:,} tokens, {arr_len*2/1e6:.0f} MB")

prepare_tinystories()

bins already exist, skipping


In [ ]:
# Why uint16 is safe, and where it would break -- verify, don't assume.
print(f"GPT-2 largest token id : {enc.n_vocab - 1:,}")
print(f"uint16 maximum value   : {np.iinfo(np.uint16).max:,}")
print(f"fits?                  : {enc.n_vocab - 1 <= np.iinfo(np.uint16).max}\n")

n_tokens = 373_000_000
for dt in [np.uint16, np.int32, np.int64]:
    size = n_tokens * np.dtype(dt).itemsize
    print(f"  {np.dtype(dt).name:<8} {np.dtype(dt).itemsize} bytes/token "
          f"-> {size/1e9:.2f} GB for {n_tokens/1e6:.0f}M tokens")
print("\nuint16 halves the file and halves the memory traffic, for free.")
print("It breaks the moment vocab > 65,536 -- which many multilingual tokenizers exceed.")

GPT-2 largest token id : 50,256
uint16 maximum value   : 65,535
fits?                  : True

  uint16   2 bytes/token -> 0.75 GB for 373M tokens
  int32    4 bytes/token -> 1.49 GB for 373M tokens
  int64    8 bytes/token -> 2.98 GB for 373M tokens

uint16 halves the file and halves the memory traffic, for free.
It breaks the moment vocab > 65,536 -- which many multilingual tokenizers exceed.


## 1.4 `get_batch`: the shape trace

```
data                     np.memmap, shape (N,)            N ~ 373,000,000
   |
   |  ix = randint(0, N - T, size=(B,))     B random starting offsets
   v
[ data[i : i+T]   for i in ix ]             B separate arrays, each shape (T,)
   |
   |  torch.stack(..., dim=0)               <- stack ADDS a new axis
   v
x                        (B, T)  int64
y                        (B, T)  int64      same thing, offsets shifted by +1
```

### `torch.stack` vs `torch.cat` — nail this down

These two get confused constantly, so let's be precise.

```python
a = torch.zeros(3)      # shape (3,)
b = torch.zeros(3)      # shape (3,)

torch.cat([a, b], dim=0).shape      # (6,)     -> joins along an EXISTING axis
torch.stack([a, b], dim=0).shape    # (2, 3)   -> creates a NEW axis
```

- **`cat`** glues tensors together along an axis that already exists. It needs every shape to match *except* along `dim`. Two `(3,)` tensors → one `(6,)` tensor. No new axes.
- **`stack`** creates a brand-new axis and lines the tensors up along it. It needs *all* shapes to match exactly, and inserts an axis of size `len(list)` at position `dim`. Two `(3,)` tensors → one `(2, 3)` tensor.

Here we have `B` tensors of shape `(T,)` and we want `(B, T)`. That's a **new** axis. So: `stack`.

### `.astype(np.int64)` — why the cast

The memmap holds `uint16`. But `nn.Embedding` requires `int64` (`torch.long`) indices, and PyTorch has no `uint16` dtype at all. So we cast on the numpy side, on the CPU, before the transfer to GPU.

### `pin_memory()` + `non_blocking=True` — what "pinned" means

This one needs unpacking rather than naming.

Normally, the OS is free to move your process's memory around — swap it to disk, relocate it, whatever. That memory is called **pageable**.

The GPU's **copy engine** (a dedicated bit of hardware that moves data over PCIe without involving the CPU) cannot work with pageable memory. It needs a *fixed physical address* that won't move mid-transfer.

So:

- If you copy from **pageable** memory, CUDA must first copy your data into a hidden staging buffer that *is* fixed, then DMA from there. That extra copy is synchronous — `non_blocking=True` silently does nothing.
- If you call `.pin_memory()` first, you're asking the OS to **page-lock** that memory: pin it to a physical address and promise never to move it. Now the copy engine can DMA straight out of it, and `non_blocking=True` genuinely returns immediately while the transfer overlaps with whatever the GPU is currently computing.

> **DMA** = Direct Memory Access = hardware moving bytes without the CPU supervising each one.

On a small model like ours the H2D transfer is a real fraction of step time, so those two calls matter.

To understand `non_blocking=False`, we first need to define exactly what the CPU is doing.

### The Role of the CPU

Think of the CPU as a **Head Chef** running a kitchen (your Python script). The GPU is a massive oven, and the data (your batch of tokens) are the ingredients.

The CPU's job is to read your Python code line by line and issue commands. It doesn't actually do the heavy lifting of training the neural network; it just tells the system what to do and when to do it.

### Scenario 1: `non_blocking=False` (The Micro-Manager)

This is the default setting in PyTorch. It is completely **synchronous**.

If you do `x.to(device, non_blocking=False)`, here is what happens:

1. The CPU reaches the line of code to move the batch of data to the GPU.
2. The CPU initiates the transfer over the PCIe cable.
3. **The CPU completely freezes.** It will not read the next line of your Python script. It stands there acting like a micro-manager, staring at the transfer process until every single byte has safely landed inside the GPU's memory.
4. Only when the transfer is 100% finished does the CPU move to the next line of code (which is usually the command to run the forward pass: `logits = model(x)`).

**The Problem:** While the CPU is frozen watching the data transfer, it isn't doing anything else. Worse, the GPU's massive math cores are sitting completely idle waiting for the data to arrive. Time is being wasted.

### Scenario 2: `non_blocking=True` (The Delegator)

This setting is **asynchronous**. It allows the CPU to multitask.

If you do `x.to(device, non_blocking=True)` (and the memory is pinned), here is what happens:

1. The CPU reaches the line of code.
2. The CPU tells the GPU's Direct Memory Access (DMA) hardware: *"Hey, this data is pinned. It's safe. Start copying it over to the GPU."*
3. **The CPU immediately moves to the next line of Python code.** It does not wait for the transfer to finish.

### Why this makes training faster

Because the CPU immediately moved on, it can tell the GPU to start calculating the math for the *previous* batch of data, while the DMA hardware is busy transferring the *next* batch of data in the background.

To go back to the kitchen analogy:

* **`non_blocking=False`**: The Chef hands the ingredients to the oven loader, and stands there staring until the food is fully inside the oven before they go back to chopping vegetables.
* **`non_blocking=True`**: The Chef says "Load this," and immediately turns around to start chopping the next batch of vegetables while the loading happens behind them.

The CPU's role is just to issue the commands. By setting `non_blocking=True`, you stop the CPU from babysitting the data transfer, allowing your system to overlap the boring work (moving data) with the hard work (doing GPU math).

In [ ]:
BLOCK_SIZE = REAL.block_size

def get_batch(split, batch_size=64, block_size=BLOCK_SIZE):
    # Re-open the memmap every call. Looks wasteful; it is not. A long-lived memmap
    # keeps every page it has ever touched alive -- a documented leak. Re-opening is free.
    filename = f"{DATA_DIR}/{'train' if split == 'train' else 'val'}.bin"
    data = np.memmap(filename, dtype=np.uint16, mode='r')          # (N,)

    ix = torch.randint(len(data) - block_size - 1, (batch_size,))  # (B,) start offsets
    # B arrays of shape (T,)  ->  stack  ->  (B, T)
    x = torch.stack([torch.from_numpy(data[i:i+block_size].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i+1:i+1+block_size].astype(np.int64)) for i in ix])

    if device_type == 'cuda':
        # pin -> page-locked -> the copy engine can DMA it -> non_blocking really overlaps
        x = x.pin_memory().to(device, non_blocking=True)
        y = y.pin_memory().to(device, non_blocking=True)
    else:
        x, y = x.to(device), y.to(device)
    return x, y

# --- stack vs cat, since this is the #1 place people reach for the wrong one ---
a, b = torch.zeros(3), torch.ones(3)
print("two tensors, each shape (3,):\n")
print(f"  cat(  [a,b], dim=0) -> {tuple(torch.cat([a,b], 0).shape)}     "
      f"joins an EXISTING axis. No new axis.")
print(f"  stack([a,b], dim=0) -> {tuple(torch.stack([a,b], 0).shape)}   "
      f"creates a NEW axis at position 0.")
print(f"  stack([a,b], dim=1) -> {tuple(torch.stack([a,b], 1).shape)}   "
      f"creates a NEW axis at position 1.")
print(f"\n  cat   values: {torch.cat([a,b], 0).tolist()}")
print(f"  stack values: {torch.stack([a,b], 0).tolist()}")
print("\nWe have B tensors of shape (T,) and want (B, T) -- a NEW axis. So: stack.")

two tensors, each shape (3,):

  cat(  [a,b], dim=0) -> (6,)     joins an EXISTING axis. No new axis.
  stack([a,b], dim=0) -> (2, 3)   creates a NEW axis at position 0.
  stack([a,b], dim=1) -> (3, 2)   creates a NEW axis at position 1.

  cat   values: [0.0, 0.0, 0.0, 1.0, 1.0, 1.0]
  stack values: [[0.0, 0.0, 0.0], [1.0, 1.0, 1.0]]

We have B tensors of shape (T,) and want (B, T) -- a NEW axis. So: stack.


In [ ]:
# Sanity check the batcher (needs the .bin files from 1.3)
if os.path.exists(f"{DATA_DIR}/train.bin"):
    xb, yb = get_batch('train', batch_size=4, block_size=16)
    sh("x", xb); sh("y", yb)
    print(f"dtype     : {xb.dtype}   <- int64, because nn.Embedding demands it")
    print(f"\nrow 0 x   : {xb[0].tolist()}")
    print(f"row 0 y   : {yb[0].tolist()}")
    print("            ^ look: y is x, slid left by exactly one position")

    # THE defining invariant of causal LM data
    assert torch.equal(xb[0, 1:], yb[0, :-1]), "y must be x shifted by 1!"
    print("\n[OK] y[:, :-1] == x[:, 1:]  -- the shift invariant holds")
    print(f"\ndecoded x[0]: {enc.decode(xb[0].tolist())!r}")
    print(f"\nA (64, 256) batch = {64*256:,} supervised examples in ONE forward pass.")
    print("All of that density is bought by the causal mask in Part 4.")
else:
    print("run prepare_tinystories() first")

x                            (4, 16)
y                            (4, 16)
dtype     : torch.int64   <- int64, because nn.Embedding demands it

row 0 x   : [679, 3073, 3772, 13, 198, 198, 1, 6090, 356, 711, 351, 534, 3290, 1701, 5045, 7893]
row 0 y   : [3073, 3772, 13, 198, 198, 1, 6090, 356, 711, 351, 534, 3290, 1701, 5045, 7893, 13]
            ^ look: y is x, slid left by exactly one position

[OK] y[:, :-1] == x[:, 1:]  -- the shift invariant holds

decoded x[0]: ' He looks happy.\n\n"Can we play with your dog?" Tim asks'

A (64, 256) batch = 16,384 supervised examples in ONE forward pass.
All of that density is bought by the causal mask in Part 4.


In [ ]:
xb

tensor([[  679,  3073,  3772,    13,   198,   198,     1,  6090,   356,   711,
           351,   534,  3290,  1701,  5045,  7893],
        [  465,   670,   290,  2227,   284,  2648,   340,   351,  2506,    13,
           679,  3751,   340,   284,   257,   513],
        [  867, 12163,   290,   257,  1263,  7604,   319,  1353,    13,  5045,
           373,   845, 33440,   290,  2227,   284],
        [  502,   262,  1994,    11, 20037,  2474,  4186, 17293,   422,  2354,
            13,   366,  2504,   318,   407,  3148]])

In [ ]:
yb

tensor([[ 3073,  3772,    13,   198,   198,     1,  6090,   356,   711,   351,
           534,  3290,  1701,  5045,  7893,    13],
        [  670,   290,  2227,   284,  2648,   340,   351,  2506,    13,   679,
          3751,   340,   284,   257,   513,   614],
        [12163,   290,   257,  1263,  7604,   319,  1353,    13,  5045,   373,
           845, 33440,   290,  2227,   284,   766],
        [  262,  1994,    11, 20037,  2474,  4186, 17293,   422,  2354,    13,
           366,  2504,   318,   407,  3148,    13]])

---
# Part 2 — Embeddings

## 2.1 What the module actually contains

`nn.Embedding(V, C)` takes two numbers:

- **`V`** = vocabulary size = how many different tokens exist in the world. GPT-2: 50257.
- **`C`** = embedding dimension = how many numbers we use to represent one token. Our model: 384.

And it gives you back a module that holds exactly **one** thing: a tensor called `.weight`, with shape `(V, C)`.

That tensor is a **table**. It has `V` rows and `C` columns. Row `i` holds the `C` numbers that represent token `i`.

There is nothing else in the module. No bias. No activation. No matrix multiply. Just a table.

Let's use a tiny one so we can see all of it at once: `V = 4` (only 4 possible tokens) and `C = 3` (3 numbers per token).

```
W = embedding.weight            shape (4, 3)

          col 0   col 1   col 2
row 0  [   0.1,    0.2,    0.3  ]     <- these 3 numbers ARE token 0
row 1  [   0.4,    0.5,    0.6  ]     <- these 3 numbers ARE token 1
row 2  [   0.7,    0.8,    0.9  ]     <- these 3 numbers ARE token 2
row 3  [   1.0,    1.1,    1.2  ]     <- these 3 numbers ARE token 3
```

Someone hands you the number `2`. You walk to row 2. You read off `[0.7, 0.8, 0.9]`. You hand it back.

**That is the entire forward pass of an embedding layer.** Everything below is just doing that four times at once and then keeping track of the shape.

## 2.2 The lookup, one integer at a time

Now feed it a real batch. Two sequences, two tokens each:

```
x = [[2, 0],          <- sequence 0 is: token 2, then token 0
     [1, 3]]          <- sequence 1 is: token 1, then token 3

shape (B, T) = (2, 2).  Four integers in total.
```

Here is the thing to hold onto: **the module treats all four integers as completely independent.** It does not know these are sequences. It does not know position 0 comes before position 1. It sees four integers and does four lookups.

Let's do all four, one at a time:

```
x[0][0] = 2   ->   walk to row 2 of W   ->   [0.7, 0.8, 0.9]   ->   put it at out[0][0]
x[0][1] = 0   ->   walk to row 0 of W   ->   [0.1, 0.2, 0.3]   ->   put it at out[0][1]
x[1][0] = 1   ->   walk to row 1 of W   ->   [0.4, 0.5, 0.6]   ->   put it at out[1][0]
x[1][1] = 3   ->   walk to row 3 of W   ->   [1.0, 1.1, 1.2]   ->   put it at out[1][1]
```

Now assemble those four results into one tensor, keeping them where they landed:

```
out = [ [ [0.7, 0.8, 0.9] ,  [0.1, 0.2, 0.3] ] ,      <- sequence 0
        [ [0.4, 0.5, 0.6] ,  [1.0, 1.1, 1.2] ] ]      <- sequence 1
```

Count the nesting. Going from outside in:

- the outermost bracket holds **2** things — the sequences. That's axis 0, size 2 = `B`.
- each sequence holds **2** things — the positions. That's axis 1, size 2 = `T`.
- each position holds **3** numbers. That's axis 2, size 3 = `C`.

So `out` has shape `(2, 2, 3)` = `(B, T, C)`.

### Where did the third axis come from?

This is the only question that matters here, so let's be explicit.

**Before:** at slot `(b, t)` there was a *single number* — a token id like `2`.

**After:** at slot `(b, t)` there is a *list of 3 numbers* — `[0.7, 0.8, 0.9]`.

Replacing one number with a list of `C` numbers, at every slot, adds exactly one axis, and it gets added **at the end**. `(2, 2)` becomes `(2, 2, 3)`.

The `B` and `T` axes are untouched. Nothing moved. Nothing got reordered. We only grew a new axis on the right.

In [ ]:
V_demo, C_demo = 4, 3
emb = nn.Embedding(V_demo, C_demo)

# Set the table to the exact numbers from the walkthrough above.
emb.weight.data = torch.tensor([[0.1, 0.2, 0.3],      # token 0
                                [0.4, 0.5, 0.6],      # token 1
                                [0.7, 0.8, 0.9],      # token 2
                                [1.0, 1.1, 1.2]])     # token 3

print("The table W (this is the ENTIRE module):")
print(emb.weight.data)
print(f"shape: {tuple(emb.weight.shape)} = (V, C) = ({V_demo}, {C_demo})\n")

# A single token, on its own
print("Look up token 2 by hand   : W[2] =", emb.weight.data[2].tolist())
print("Look up token 2 via module:", emb(torch.tensor(2)).tolist())
print("  ...the module does nothing more than index into its own table.\n")

x = torch.tensor([[2, 0],
                  [1, 3]])                       # (B, T) = (2, 2)
out = emb(x)                                     # (B, T, C) = (2, 2, 3)

print("Now the batch, four lookups:")
for b in range(2):
    for t in range(2):
        tok = x[b, t].item()
        print(f"  x[{b}][{t}] = {tok}  ->  W[{tok}] = {out[b, t].tolist()}  ->  out[{b}][{t}]")

print()
sh("x   (B, T)", x)
sh("out (B, T, C)", out)
print("\nout =")
print(out.data)
print("\nThe (B, T) grid is unchanged. Each SCALAR became a LIST OF 3.")
print("That is where the third axis came from, and why it is on the right.")

The table W (this is the ENTIRE module):
tensor([[0.1000, 0.2000, 0.3000],
        [0.4000, 0.5000, 0.6000],
        [0.7000, 0.8000, 0.9000],
        [1.0000, 1.1000, 1.2000]])
shape: (4, 3) = (V, C) = (4, 3)

Look up token 2 by hand   : W[2] = [0.699999988079071, 0.800000011920929, 0.8999999761581421]
Look up token 2 via module: [0.699999988079071, 0.800000011920929, 0.8999999761581421]
  ...the module does nothing more than index into its own table.

Now the batch, four lookups:
  x[0][0] = 2  ->  W[2] = [0.699999988079071, 0.800000011920929, 0.8999999761581421]  ->  out[0][0]
  x[0][1] = 0  ->  W[0] = [0.10000000149011612, 0.20000000298023224, 0.30000001192092896]  ->  out[0][1]
  x[1][0] = 1  ->  W[1] = [0.4000000059604645, 0.5, 0.6000000238418579]  ->  out[1][0]
  x[1][1] = 3  ->  W[3] = [1.0, 1.100000023841858, 1.2000000476837158]  ->  out[1][1]

x   (B, T)                   (2, 2)
out (B, T, C)                (2, 2, 3)

out =
tensor([[[0.7000, 0.8000, 0.9000],
         [0.1000,

## 2.3 The shape rule, stated for any input

The rule generalises to any input shape at all. Whatever shape of integers goes in, you get that **same shape with `C` stuck on the end**:

| input shape | what it means | output shape |
|---|---|---|
| `(2, 2)` | 2 sequences x 2 tokens | `(2, 2, 3)` |
| `(B, T)` | our normal batch | `(B, T, C)` |
| `(B,)` | one token per sequence | `(B, C)` |
| `(5,)` | just 5 loose tokens | `(5, C)` |
| `(2, 3, 4)` | some 3-D pile of tokens | `(2, 3, 4, C)` |
| `()` (a single number) | one token | `(C,)` |

Earlier I wrote this rule as "shape `S` returns shape `S + (C,)`". That was needlessly cryptic, and I should say what it meant: the `+` there is **Python tuple joining**, not addition.

```python
(2, 3) + (384,)   ->   (2, 3, 384)        # joins the tuples
```

It is *not* arithmetic. Nothing is being added to anything. It just means "stick `C` on the end", which is exactly what the table above shows.

Why does this hold for any shape? Because the module never looks at your axes. It flattens your integers into one long list, does one lookup per integer, and then folds the results back into your original shape — with each scalar now `C` numbers wide.

In [ ]:
emb384 = nn.Embedding(50304, 384)
print("Whatever shape goes in, C=384 gets stuck on the end:\n")
for shape, meaning in [((2, 2), "2 seqs x 2 tokens"),
                       ((64, 256), "a real training batch"),
                       ((8,), "8 loose tokens"),
                       ((2, 3, 4), "some 3-D pile of tokens"),
                       ((), "one single token")]:
    idx_demo = torch.randint(0, 50304, shape)
    out_demo = emb384(idx_demo)
    print(f"  in {str(tuple(idx_demo.shape)):<12} ({meaning:<22}) "
          f"-> out {tuple(out_demo.shape)}")

print("\nAnd the '+' in 'S + (C,)' is Python TUPLE JOINING, not addition:")
print(f"  (2, 3) + (384,) = {(2, 3) + (384,)}     <- joins them into one tuple")
print("  It just means: stick C on the end. No arithmetic is happening.")

Whatever shape goes in, C=384 gets stuck on the end:

  in (2, 2)       (2 seqs x 2 tokens     ) -> out (2, 2, 384)
  in (64, 256)    (a real training batch ) -> out (64, 256, 384)
  in (8,)         (8 loose tokens        ) -> out (8, 384)
  in (2, 3, 4)    (some 3-D pile of tokens) -> out (2, 3, 4, 384)
  in ()           (one single token      ) -> out (384,)

And the '+' in 'S + (C,)' is Python TUPLE JOINING, not addition:
  (2, 3) + (384,) = (2, 3, 384)     <- joins them into one tuple
  It just means: stick C on the end. No arithmetic is happening.


## 2.4 Why this is secretly a matrix multiply

Everything so far was "fetch a row". Now the reframe. It looks like a detour, but it is the thing that makes the backward pass obvious instead of mysterious, so it's worth the five minutes.

### One-hot encoding

The **one-hot** of token `2`, when `V = 4`, is:

```
[0, 0, 1, 0]
       ^
       a single 1 at index 2, zeros everywhere else. Length V.
```

That's all one-hot means: a length-`V` vector that is all zeros except a single `1`, placed at the index of the thing you're encoding.

### Multiply the one-hot by the table

```
[0, 0, 1, 0]   @   [[0.1, 0.2, 0.3],
   (1, 4)           [0.4, 0.5, 0.6],
                    [0.7, 0.8, 0.9],
                    [1.0, 1.1, 1.2]]
                          (4, 3)
```

Let's do this multiplication by hand, slowly. In a matrix multiply, output element `j` is the dot product of the left vector with **column `j`** of the right matrix.

**Output element 0** — dot the one-hot with column 0 of W, which is `[0.1, 0.4, 0.7, 1.0]`:

```
   0 * 0.1   +   0 * 0.4   +   1 * 0.7   +   0 * 1.0    =   0.7
   ^^^^^^^       ^^^^^^^       ^^^^^^^       ^^^^^^^
   = 0           = 0           = 0.7         = 0
   killed        killed        SURVIVES      killed
```

**Output element 1** — dot with column 1, `[0.2, 0.5, 0.8, 1.1]`:

```
   0 * 0.2   +   0 * 0.5   +   1 * 0.8   +   0 * 1.1    =   0.8
```

**Output element 2** — dot with column 2, `[0.3, 0.6, 0.9, 1.2]`:

```
   0 * 0.3   +   0 * 0.6   +   1 * 0.9   +   0 * 1.2    =   0.9
```

Result: `[0.7, 0.8, 0.9]`.

**That is exactly `W[2]`.** Row 2 of the table.

### Why it worked

Look at what the zeros did. In every one of those three sums, three of the four terms were multiplied by `0` and vanished. Only the term multiplied by the single `1` survived — and that term was always an element of row 2.

**A one-hot vector times a matrix is a row selection.** The zeros annihilate every row you don't want; the lone `1` lets exactly one row through untouched.

So the two things are the same operation:

```
nn.Embedding(x)         ==         one_hot(x) @ W
   "fetch row 2"                   "multiply by a mostly-zero vector"
```

Both give `(B, T, C)`. Same numbers. Always.

In [ ]:
W = emb.weight.data                              # our (4, 3) table
onehot_2 = F.one_hot(torch.tensor(2), num_classes=4).float()
print(f"one_hot(2), V=4 : {onehot_2.tolist()}      <- a single 1 at index 2\n")

print("The matmul, term by term (0 = killed by the one-hot's zeros):")
for j in range(3):
    col = W[:, j]
    terms = [f"{onehot_2[i].item():.0f}*{col[i].item():.1f}" for i in range(4)]
    vals  = [onehot_2[i].item() * col[i].item() for i in range(4)]
    print(f"  out[{j}] = " + "  +  ".join(terms) + f"  =  {sum(vals):.1f}")

print(f"\n  result   = {(onehot_2 @ W).tolist()}")
print(f"  W[2]     = {W[2].tolist()}      <- identical. It selected row 2.")

# --- and on the full batch ---
print("\n--- the same thing on the whole (B, T) batch ---")
oh = F.one_hot(x, num_classes=4).float()         # (B, T, V) = (2, 2, 4)
sh("one_hot(x)  (B, T, V)", oh)
print("one_hot(x) =")
print(oh)
out2 = oh @ emb.weight                           # (B,T,V) @ (V,C) -> (B,T,C)
sh("\none_hot(x) @ W  (B, T, C)", out2)
assert torch.allclose(out, out2)
print("\n[OK] nn.Embedding(x) and one_hot(x) @ W give EXACTLY the same numbers.")

one_hot(2), V=4 : [0.0, 0.0, 1.0, 0.0]      <- a single 1 at index 2

The matmul, term by term (0 = killed by the one-hot's zeros):
  out[0] = 0*0.1  +  0*0.4  +  1*0.7  +  0*1.0  =  0.7
  out[1] = 0*0.2  +  0*0.5  +  1*0.8  +  0*1.1  =  0.8
  out[2] = 0*0.3  +  0*0.6  +  1*0.9  +  0*1.2  =  0.9

  result   = [0.699999988079071, 0.800000011920929, 0.8999999761581421]
  W[2]     = [0.699999988079071, 0.800000011920929, 0.8999999761581421]      <- identical. It selected row 2.

--- the same thing on the whole (B, T) batch ---
one_hot(x)  (B, T, V)        (2, 2, 4)
one_hot(x) =
tensor([[[0., 0., 1., 0.],
         [1., 0., 0., 0.]],

        [[0., 1., 0., 0.],
         [0., 0., 0., 1.]]])

one_hot(x) @ W  (B, T, C)   (2, 2, 3)

[OK] nn.Embedding(x) and one_hot(x) @ W give EXACTLY the same numbers.


## 2.5 So why doesn't PyTorch do the matmul?

Because it would be absurd. Count the work for looking up **one single token**, at real GPT-2 sizes (`V = 50257`, `C = 384`):

| | work |
|---|---|
| the matmul way | `V x C` = 50257 x 384 = **19.3 million** multiply-adds |
| the lookup way | copy **384** numbers. Zero arithmetic. |

19.3 million operations to move 384 numbers into place. And **50,256 out of 50,257** of those multiplications are multiplying by zero. It's all waste.

So PyTorch implements it as a **gather**.

> **Gather** = "here is a list of row indices; go fetch me those rows."
>
> There is no arithmetic in a gather at all. It computes a memory address and copies bytes. `W[2]` means "start of `W`, plus 2 rows' worth of bytes, read `C` numbers." That's it.

(You'll meet `torch.gather` by name again in Part 10, doing the same job for top-p sampling. Indices in, selected values out.)

**So why did we bother with the one-hot story?** Because the one-hot view is not how it's *computed* — it's how you should *think* about it. And the payoff is right now: it makes the backward pass fall out of ordinary matmul calculus, instead of being a rule you memorise.

In [ ]:
V_real, C_real = 50257, 384
matmul_flops = V_real * C_real
print(f"Looking up ONE token at V={V_real:,}, C={C_real}:\n")
print(f"  as a matmul : {matmul_flops:,} multiply-adds")
print(f"  as a lookup : {C_real} numbers copied, 0 arithmetic")
print(f"  ratio       : {matmul_flops // C_real:,}x more work for the same answer")
print(f"  and {(V_real-1)/V_real*100:.3f}% of those multiplications are  x 0.\n")

if device_type == 'cuda':
    Wbig = torch.randn(V_real, C_real, device=device)
    ids = torch.randint(0, V_real, (4096,), device=device)
    def t(fn, n=20):
        for _ in range(3): fn()
        torch.cuda.synchronize(); t0 = time.time()
        for _ in range(n): fn()
        torch.cuda.synchronize(); return (time.time()-t0)/n*1000
    t_gather = t(lambda: Wbig[ids])
    t_matmul = t(lambda: F.one_hot(ids, V_real).float() @ Wbig)
    print(f"  measured, 4096 tokens:")
    print(f"    gather (W[ids])         : {t_gather:.3f} ms")
    print(f"    one_hot(ids).float() @ W: {t_matmul:.3f} ms   <- {t_matmul/t_gather:.0f}x slower")
else:
    print("  (run on CUDA to see the gather measured against the matmul)")

Looking up ONE token at V=50,257, C=384:

  as a matmul : 19,298,688 multiply-adds
  as a lookup : 384 numbers copied, 0 arithmetic
  ratio       : 50,257x more work for the same answer
  and 99.998% of those multiplications are  x 0.

  (run on CUDA to see the gather measured against the matmul)


## 2.6 The backward pass, derived

This is the part I previously fired off in one line. Let's actually derive it, because it's short and it explains everything that follows.

### Set up

Use the one-hot view, and flatten the batch so there's only one axis of tokens. Let `N = B*T` = how many tokens we looked up in total.

```
H = one_hot(x)          (N, V)      one row per token; each row is a one-hot
y = H @ W               (N, C)      the embedding vectors we fetched
```

### Apply the standard matmul backward

For any `y = H @ W`, calculus gives you:

```
dL/dW  =  H^T @ dL/dy
```

That's not special to embeddings — it's the ordinary gradient of a matrix product. Sanity-check the shapes:

```
H^T      is  (V, N)
dL/dy    is  (N, C)
-------------------------
H^T @ dL/dy  is  (V, C)      <- same shape as W. Good, gradients always match their tensor.
```

### Now unpack what `H^T @ dL/dy` actually does

`H^T` has shape `(V, N)`. Its entry `H^T[v][n]` is `1` if position `n` used token `v`, and `0` otherwise.

So **row `v` of `H^T`** is a length-`N` checklist: *"which of my N positions used token `v`?"*

Row `v` of the result is that checklist dotted against the gradients:

```
dL/dW[v]  =  sum over n of   H^T[v][n] * dL/dy[n]
```

Every `n` where token `v` did **not** appear contributes `0 * dL/dy[n]` = nothing. Every `n` where it **did** appear contributes `1 * dL/dy[n]` = the full gradient. So:

```
dL/dW[v]  =  the SUM of dL/dy[n]  over every position n where x[n] == v
```

### Read that out loud

> **The gradient for token `v`'s row is the sum of the incoming gradients from every position where token `v` showed up.**

Two things drop straight out of that sentence:

**1. A token that never appeared gets exactly zero.** The sum is empty. Not "small" — *exactly* `0.0`. That row of the gradient is untouched.

**2. A token that appeared 3 times gets the sum of 3 gradient vectors.**

That second point is not an implementation quirk. It is **the chain rule**. When one parameter affects the loss through several different paths, the gradients from those paths **add up**. If token 2 sits at position 0 and again at position 7, then `W[2]` fed into the loss twice, by two routes. Both routes send a gradient back, and both count.

### A worked example

`V = 4`, `C = 2`. Three tokens: `x = [2, 0, 2]`. **Notice token 2 appears twice.**

Suppose the gradient arriving from above is:

```
dL/dy = [[1, 1],        <- position 0  (which used token 2)
         [2, 2],        <- position 1  (which used token 0)
         [3, 3]]        <- position 2  (which used token 2)
```

Apply the rule — for each token, sum the gradients from its positions:

```
dL/dW[0]  =  dL/dy[1]                =  [2, 2]     token 0 appeared only at position 1
dL/dW[1]  =  (nothing)               =  [0, 0]     token 1 never appeared -> exactly zero
dL/dW[2]  =  dL/dy[0] + dL/dy[2]
          =  [1,1]  +  [3,3]         =  [4, 4]     token 2 appeared at 0 AND 2 -> SUMMED
dL/dW[3]  =  (nothing)               =  [0, 0]     token 3 never appeared -> exactly zero
```

Let's check that against PyTorch.

In [ ]:
V_g, C_g = 4, 2
emb_g = nn.Embedding(V_g, C_g)
emb_g.weight.data = torch.arange(V_g * C_g).float().reshape(V_g, C_g)

x_g = torch.tensor([2, 0, 2])                    # token 2 appears TWICE
y_g = emb_g(x_g)                                 # (3, 2)

# Hand-inject a gradient from above so we know exactly what should come back.
grad_from_above = torch.tensor([[1., 1.],        # position 0 used token 2
                                [2., 2.],        # position 1 used token 0
                                [3., 3.]])       # position 2 used token 2
y_g.backward(grad_from_above)

print(f"x      = {x_g.tolist()}        <- token 2 at positions 0 and 2")
print(f"dL/dy  =\n{grad_from_above}\n")
print("Hand calculation:")
print("  dL/dW[0] = dL/dy[1]              = [2, 2]     (token 0: position 1 only)")
print("  dL/dW[1] = nothing               = [0, 0]     (token 1: never appeared)")
print("  dL/dW[2] = dL/dy[0] + dL/dy[2]")
print("           = [1,1] + [3,3]         = [4, 4]     (token 2: TWO positions, SUMMED)")
print("  dL/dW[3] = nothing               = [0, 0]     (token 3: never appeared)")
print(f"\nPyTorch says dL/dW =\n{emb_g.weight.grad}")

expected = torch.tensor([[2., 2.], [0., 0.], [4., 4.], [0., 0.]])
assert torch.equal(emb_g.weight.grad, expected)
print("\n[OK] matches the hand calculation exactly.")
print("     Row 2 is [4,4], not [1,1] or [3,3] -- the two occurrences ADDED.")
print("     That is the chain rule: one weight, two paths to the loss, gradients sum.")

x      = [2, 0, 2]        <- token 2 at positions 0 and 2
dL/dy  =
tensor([[1., 1.],
        [2., 2.],
        [3., 3.]])

Hand calculation:
  dL/dW[0] = dL/dy[1]              = [2, 2]     (token 0: position 1 only)
  dL/dW[1] = nothing               = [0, 0]     (token 1: never appeared)
  dL/dW[2] = dL/dy[0] + dL/dy[2]
           = [1,1] + [3,3]         = [4, 4]     (token 2: TWO positions, SUMMED)
  dL/dW[3] = nothing               = [0, 0]     (token 3: never appeared)

PyTorch says dL/dW =
tensor([[2., 2.],
        [0., 0.],
        [4., 4.],
        [0., 0.]])

[OK] matches the hand calculation exactly.
     Row 2 is [4,4], not [1,1] or [3,3] -- the two occurrences ADDED.
     That is the chain rule: one weight, two paths to the loss, gradients sum.


In [ ]:
# And the one-hot view gives the SAME gradient -- because it is the same function.
W_oh = torch.arange(V_g * C_g).float().reshape(V_g, C_g).requires_grad_(True)
H = F.one_hot(x_g, num_classes=V_g).float()      # (N, V) = (3, 4)

print("H = one_hot(x), shape (N, V):")
print(H)
print("\nH^T, shape (V, N) -- row v answers 'which positions used token v?':")
print(H.T)
print("  row 0 -> [0,1,0]  : token 0 was used at position 1")
print("  row 1 -> [0,0,0]  : token 1 was never used")
print("  row 2 -> [1,0,1]  : token 2 was used at positions 0 AND 2")
print("  row 3 -> [0,0,0]  : token 3 was never used")

print(f"\nH^T @ dL/dy  =  (V,N) @ (N,C)  ->  (V,C):")
print(H.T @ grad_from_above)

(H @ W_oh).backward(grad_from_above)
assert torch.equal(W_oh.grad, expected)
print("\n[OK] identical to the embedding's gradient. Same function, same derivative.")
print("     Row 2 = [1,0,1] @ dL/dy = dL/dy[0] + dL/dy[2]. The SUM is literally the matmul.")

H = one_hot(x), shape (N, V):
tensor([[0., 0., 1., 0.],
        [1., 0., 0., 0.],
        [0., 0., 1., 0.]])

H^T, shape (V, N) -- row v answers 'which positions used token v?':
tensor([[0., 1., 0.],
        [0., 0., 0.],
        [1., 0., 1.],
        [0., 0., 0.]])
  row 0 -> [0,1,0]  : token 0 was used at position 1
  row 1 -> [0,0,0]  : token 1 was never used
  row 2 -> [1,0,1]  : token 2 was used at positions 0 AND 2
  row 3 -> [0,0,0]  : token 3 was never used

H^T @ dL/dy  =  (V,N) @ (N,C)  ->  (V,C):
tensor([[2., 2.],
        [0., 0.],
        [4., 4.],
        [0., 0.]])

[OK] identical to the embedding's gradient. Same function, same derivative.
     Row 2 = [1,0,1] @ dL/dy = dL/dy[0] + dL/dy[2]. The SUM is literally the matmul.


Here is the rewritten derivation, completely overhauled to respect your "Paths of Influence" intuition. By recognizing structurally how the data flows, we can bypass the blind, textbook multivariate calculus and write the chain rule exactly as it behaves in reality.

---

### 1. The Local Forward Pass (Scalar Level)

First, we define our variables and flatten the batch into a single sequence of length $N = B \times T$.

* Let $x_n$ be the integer token ID at position $n \in \{0, 1, \dots, N-1\}$.
* Let $W \in \mathbb{R}^{V \times C}$ be the embedding weight matrix, where $V$ is the vocabulary size and $C$ is the embedding channel dimension.
* Let $y \in \mathbb{R}^{N \times C}$ be the output tensor containing the fetched embeddings.

The forward pass is a simple index lookup. For a specific position $n$ and a specific channel $c$, the output scalar is fetched directly from the $c$-th column of the row corresponding to the token ID $x_n$:

$$y_{n, c} = W_{x_n, c}$$

### 2. The Chain Rule (Paths of Influence)

To compute the backward pass for a specific weight parameter $W_{v, c}$, we ask: *Where was this weight reused, and what did it influence?*

Because an embedding layer does no mathematical mixing across channels, the weight $W_{v, c}$ travels straight across to influence **only** the exact same channel $c$ in the output. It is never broadcast across the $C$ dimension. It is *only* reused across the sequence dimension $N$ (whenever the token $v$ appears).

Following the rule to sum only over the dimensions where the weight was reused, our chain rule immediately scopes down to a single summation over $N$:

$$\frac{\partial L}{\partial W_{v, c}} = \sum_{n=0}^{N-1} \frac{\partial L}{\partial y_{n, c}} \frac{\partial y_{n, c}}{\partial W_{v, c}}$$

### 3. The Local Gradient (Calculus Level)

Now we evaluate the local derivative: *How does a tiny change in $W_{v, c}$ affect $y_{n, c}$?*

Because $y_{n, c}$ is a direct copy of the weight, the derivative is exactly $1$ if the token at position $n$ is indeed $v$ ($x_n = v$). If the token is anything else, the weight wasn't used there, and the derivative is $0$.

We express this on/off switch using the Kronecker delta ($\delta$):

$$\frac{\partial y_{n, c}}{\partial W_{v, c}} = \delta_{x_n, v}$$

Substitute this local gradient back into our chain rule equation:

$$\frac{\partial L}{\partial W_{v, c}} = \sum_{n=0}^{N-1} \frac{\partial L}{\partial y_{n, c}} \delta_{x_n, v}$$

**The Plain-Language Translation:** The gradient for the weight at row $v$ and channel $c$ is the sum of the incoming gradients strictly across the positions $n$ where the token ID equals $v$. If a token never appeared, $\delta_{x_n, v}$ remains $0$, and the gradient is exactly $0.0$.

### 4. The Tensor Form (Matrix Level)

To translate this structurally aware scalar derivation into a vectorized tensor operation, we need to construct the full gradient matrix $\frac{\partial L}{\partial W}$ of shape $(V, C)$.

Let's look closely at $\delta_{x_n, v}$. This is a function that returns $1$ if the token at position $n$ is $v$, and $0$ otherwise.

If we construct a one-hot encoded matrix $H \in \mathbb{R}^{N \times V}$, its entries are defined precisely as $H_{n, v} = \delta_{x_n, v}$. We can seamlessly swap the Kronecker delta for our one-hot matrix entry:

$$\frac{\partial L}{\partial W_{v, c}} = \sum_{n=0}^{N-1} H_{n, v} \frac{\partial L}{\partial y_{n, c}}$$

Since $H_{n, v}$ is equivalent to the transposed entry $H^T_{v, n}$, we can rewrite the sum as:

$$\frac{\partial L}{\partial W_{v, c}} = \sum_{n=0}^{N-1} H^T_{v, n} \frac{\partial L}{\partial y_{n, c}}$$

This is the exact mathematical definition of a matrix multiplication between a matrix $H^T$ of shape $(V, N)$ and a matrix $\frac{\partial L}{\partial y}$ of shape $(N, C)$.

Dropping the subscripts, we arrive at the final tensor form:

$$\frac{\partial L}{\partial W} = H^T \frac{\partial L}{\partial y}$$

### 5. The Hardware Reality

While the tensor derivation elegantly proves that the backward pass is mathematically identical to a matrix multiplication ($H^T @ dL/dy$), executing it that way in a low-level C++ or CUDA environment would be highly inefficient.

Because $H^T$ is a sparse matrix containing mostly zeros, a dense matrix multiplication would waste massive amounts of FLOPs multiplying zeros. Instead, the actual backend implementation discards the one-hot matrix $H$ entirely. It loops through the scalar chain rule equation we derived in Step 3—reading the token indices $x_n$ and dynamically accumulating (using an atomic `scatter_add`) the dense $C$-dimensional vector $\frac{\partial L}{\partial y_n}$ directly into the corresponding $x_n$-th row of $\frac{\partial L}{\partial W}$.

---

## 2.7 What `scatter_add` means

Now the jargon word can be defined, because you already understand the thing it names.

Three related operations:

| operation | meaning | example |
|---|---|---|
| **gather** | *read* from positions given by an index list | `W[[2, 0, 2]]` -> fetch rows 2, 0, 2 |
| **scatter** | *write* to positions given by an index list, **overwriting** | "put this value into row 2" |
| **scatter_add** | *write* to those positions, **adding** into whatever is there | "add this value into row 2" |

And now the symmetry:

```
FORWARD  =  gather        : take rows out of W, at the indices in x
BACKWARD =  scatter_add   : push gradients back into those same rows of W
```

They are mirror images. The forward reads rows at indices `x`; the backward writes gradients back to rows at indices `x`.

**Why `_add` and not plain `scatter`?** Because of exactly what we just saw. Token 2 appeared at positions 0 and 2, so two gradients need to land in row 2. If it were plain `scatter`, the second write would **overwrite** the first, and you'd end up with `[3, 3]` instead of `[4, 4]` — silently throwing away half the gradient. The `_add` is there because duplicates must sum, and duplicates must sum because that's the chain rule.

That's the whole story. `scatter_add` isn't an exotic op — it's the mirror of "fetch some rows", with addition where the duplicates collide.

In [ ]:
# gather / scatter_add, side by side, on the actual example
Wt = torch.arange(8).float().reshape(4, 2)
ids = torch.tensor([2, 0, 2])
print("W:\n", Wt)
print(f"\nGATHER (the forward): W[{ids.tolist()}] -> fetch those rows")
print(Wt[ids])

print(f"\nSCATTER_ADD (the backward): push these gradients back to rows {ids.tolist()}")
g = torch.tensor([[1., 1.], [2., 2.], [3., 3.]])
dest = torch.zeros(4, 2)
dest.index_add_(0, ids, g)               # index_add_ IS scatter_add along axis 0
print(dest)
print("  row 2 = [4,4] : positions 0 and 2 both targeted row 2, so [1,1] + [3,3] ADDED.")

print("\nIf it were plain scatter (overwrite) instead of scatter_add:")
dest_bad = torch.zeros(4, 2)
dest_bad[ids] = g                        # plain overwrite -- last write wins
print(dest_bad)
print("  row 2 = [3,3] : the second write CLOBBERED the first. Half the gradient, gone.")
print("  This is why the op is scatter_ADD. Duplicates must sum -- that's the chain rule.")

W:
 tensor([[0., 1.],
        [2., 3.],
        [4., 5.],
        [6., 7.]])

GATHER (the forward): W[[2, 0, 2]] -> fetch those rows
tensor([[4., 5.],
        [0., 1.],
        [4., 5.]])

SCATTER_ADD (the backward): push these gradients back to rows [2, 0, 2]
tensor([[2., 2.],
        [0., 0.],
        [4., 4.],
        [0., 0.]])
  row 2 = [4,4] : positions 0 and 2 both targeted row 2, so [1,1] + [3,3] ADDED.

If it were plain scatter (overwrite) instead of scatter_add:
tensor([[2., 2.],
        [0., 0.],
        [3., 3.],
        [0., 0.]])
  row 2 = [3,3] : the second write CLOBBERED the first. Half the gradient, gone.
  This is why the op is scatter_ADD. Duplicates must sum -- that's the chain rule.


## 2.8 Why any of this matters: three consequences

These are the three bullets I previously listed without explaining. Here they are properly.

### (a) The gradient tensor is huge and almost entirely zeros

`wte.weight.grad` has the same shape as `wte.weight`: `(50304, 384)`. That's 19.3M numbers — **77 MB in fp32**. It's allocated, written, and read **every single step**.

But how many of those 50,304 rows are actually nonzero? Only the rows for tokens that appeared in this batch.

A batch is `B*T = 64*256 = 16,384` tokens. If every one were a *different* token, you'd touch 16,384 rows out of 50,304 — 33%. But real text is [Zipfian](https://en.wikipedia.org/wiki/Zipf%27s_law): `"the"` and `"a"` appear constantly, most of the vocab appears never. In practice a batch touches roughly **5,000-6,000 distinct tokens**.

So around **89% of that 77 MB is exactly zero** — computed, stored, and read every step for no reason. The gradient is *shaped* dense but is *semantically* sparse.

### (b) The optimizer state is 3 more copies of the same thing

AdamW keeps two running statistics per parameter: `exp_avg` (the momentum, `m`) and `exp_avg_sq` (the squared-gradient average, `v`). Both are fp32, both the exact size of the parameter. Plus the gradient buffer itself.

For GPT-2 XL's embedding table (`V=50257, C=1600` = 80.4M parameters):

| | fp32 size |
|---|---|
| the weight | 322 MB |
| the gradient | 322 MB |
| `exp_avg` (m) | 322 MB |
| `exp_avg_sq` (v) | 322 MB |
| **grad + m + v** | **0.96 GB** |
| **all four** | **1.29 GB** |

*(Correction: the earlier draft of this section said "1.6GB". That was wrong — the real figure is 0.96 GB for grad + m + v, or 1.29 GB counting the weight itself. The cell below computes it.)*

And here's the sting: **Adam updates every row, every step, including the ~89% that got zero gradient.** A row with zero gradient still has its `m` decayed toward zero, still has `v` decayed, and — if you're weight-decaying `wte`, which we are, since it's 2-D — still gets shrunk. Tokens that never appear in your data slowly decay toward zero anyway. That's not a bug, but it *is* surprising the first time you notice it.

### (c) `sparse=True`, and why you probably won't use it

`nn.Embedding(V, C, sparse=True)` makes the backward produce a **sparse gradient** — just the touched indices and their values, instead of a full `(V, C)` dense tensor. That would turn 77 MB into ~9 MB.

So why doesn't everyone? Two reasons:

1. **Most optimizers don't support it.** Only `SparseAdam`, `Adagrad`, and plain `SGD` accept sparse gradients. `AdamW` — the one every LLM uses — will refuse.
2. **Weight tying kills it anyway.** In our model `wte.weight` *is* `lm_head.weight`. The `lm_head` side is a full `(B*T, C) @ (C, V)` matmul that produces a genuinely **dense** gradient over every row. So even if the lookup side were sparse, the sum of the two is dense. There's nothing to save.

This is why real systems attack the problem from a different angle entirely: **vocabulary parallelism**, where you shard the `(V, C)` table across GPUs by rows, and each GPU only ever holds — and only ever optimizes — its slice. Same problem, solved by splitting the tensor instead of sparsifying it.

**The through-line:** a large vocabulary is not just "more parameters". It's a *dense* `(V, C)` gradient and *three more* `(V, C)` optimizer buffers, all touched every step, mostly to process zeros. That's why vocab size is a systems decision and not just a modelling one.

In [ ]:
# --- (a) how sparse is a real batch's embedding gradient? ---
V_r, C_r, B_r, T_r = 50304, 384, 64, 256

# Sample tokens Zipf-style, which is roughly how real English behaves
ranks = torch.arange(1, V_r + 1).float()
p = 1.0 / ranks
p = p / p.sum()
tokens = torch.multinomial(p, B_r * T_r, replacement=True)
n_distinct = tokens.unique().numel()

grad_mb = V_r * C_r * 4 / 1e6
print(f"A batch of B*T = {B_r*T_r:,} tokens, Zipf-distributed over V = {V_r:,}\n")
print(f"  wte.weight.grad shape : ({V_r:,}, {C_r})  =  {V_r*C_r/1e6:.1f}M numbers  =  {grad_mb:.1f} MB fp32")
print(f"  distinct tokens in it : {n_distinct:,}")
print(f"  rows actually nonzero : {n_distinct/V_r*100:.1f}%")
print(f"  rows that are zero    : {(1-n_distinct/V_r)*100:.1f}%  ({grad_mb*(1-n_distinct/V_r):.0f} MB of zeros, every step)")
print(f"\n  (upper bound if every token were distinct: {B_r*T_r/V_r*100:.0f}%.")
print(f"   Zipf's law is why the real number is far lower.)")

A batch of B*T = 16,384 tokens, Zipf-distributed over V = 50,304

  wte.weight.grad shape : (50,304, 384)  =  19.3M numbers  =  77.3 MB fp32
  distinct tokens in it : 5,589
  rows actually nonzero : 11.1%
  rows that are zero    : 88.9%  (69 MB of zeros, every step)

  (upper bound if every token were distinct: 33%.
   Zipf's law is why the real number is far lower.)


In [ ]:
# --- (b) the optimizer state, computed (this is the number I got wrong before) ---
def state_table(name, V_, C_):
    n = V_ * C_
    mb = n * 4 / 1e6
    print(f"\n{name}:  V={V_:,}  C={C_}  ->  {n/1e6:.1f}M params in the table")
    for label in ["weight", "gradient", "exp_avg (m)", "exp_avg_sq (v)"]:
        print(f"    {label:<16} {mb:>7.1f} MB")
    print(f"    {'-'*24}")
    print(f"    {'grad + m + v':<16} {mb*3:>7.1f} MB  = {mb*3/1000:.2f} GB   <- 3 x V x C")
    print(f"    {'all four':<16} {mb*4:>7.1f} MB  = {mb*4/1000:.2f} GB   <- 4 x V x C")
    return mb

state_table("our TinyStories GPT", 50304, 384)
mb_xl = state_table("GPT-2 XL", 50257, 1600)
print(f"\n  NOTE: an earlier draft of this section claimed 1.6 GB for GPT-2 XL.")
print(f"  That was wrong. The correct figures are {mb_xl*3/1000:.2f} GB (grad+m+v) "
      f"or {mb_xl*4/1000:.2f} GB (with the weight).")

# --- (c) AdamW refuses sparse gradients ---
print("\n--- (c) why sparse=True usually doesn't help ---")
e_sparse = nn.Embedding(100, 8, sparse=True)
e_sparse(torch.tensor([3, 7])).sum().backward()
print(f"  sparse=True gradient type : {type(e_sparse.weight.grad).__name__}, "
      f"is_sparse = {e_sparse.weight.grad.is_sparse}")
try:
    torch.optim.AdamW(e_sparse.parameters(), lr=1e-3).step()
except RuntimeError as err:
    print(f"  AdamW.step() -> RuntimeError: {str(err)[:58]}...")
print("  Only SparseAdam / Adagrad / SGD take sparse grads. AdamW -- what every LLM")
print("  uses -- does not. And with weight tying, lm_head's gradient is dense anyway,")
print("  so the sum is dense regardless. Real systems shard the (V,C) table instead.")


our TinyStories GPT:  V=50,304  C=384  ->  19.3M params in the table
    weight              77.3 MB
    gradient            77.3 MB
    exp_avg (m)         77.3 MB
    exp_avg_sq (v)      77.3 MB
    ------------------------
    grad + m + v       231.8 MB  = 0.23 GB   <- 3 x V x C
    all four           309.1 MB  = 0.31 GB   <- 4 x V x C

GPT-2 XL:  V=50,257  C=1600  ->  80.4M params in the table
    weight             321.6 MB
    gradient           321.6 MB
    exp_avg (m)        321.6 MB
    exp_avg_sq (v)     321.6 MB
    ------------------------
    grad + m + v       964.9 MB  = 0.96 GB   <- 3 x V x C
    all four          1286.6 MB  = 1.29 GB   <- 4 x V x C

  NOTE: an earlier draft of this section claimed 1.6 GB for GPT-2 XL.
  That was wrong. The correct figures are 0.96 GB (grad+m+v) or 1.29 GB (with the weight).

--- (c) why sparse=True usually doesn't help ---
  sparse=True gradient type : Tensor, is_sparse = True
  AdamW.step() -> RuntimeError: Adam does not support spa

## 2.9 Position embeddings and the broadcast

### Why we need them at all

Here's a fact about attention that's worth sitting with: **it has no idea what order your tokens are in.**

Attention computes a weighted sum over the other tokens. A sum doesn't care about order — `a + b + c` is the same as `c + a + b`. So if you shuffle the input tokens, the outputs shuffle identically and nothing else changes. The technical word is **permutation-equivariant**: permute the input, and you permute the output, but no *new* information appears.

That means `"the cat sat on the mat"` and `"mat the on sat cat the"` would produce the same set of output vectors, just reordered. Useless for language.

So position has to be injected by hand. GPT-2 does it the crudest possible way: **a second lookup table, indexed by position instead of by token id**, added on top.

```
wte  = nn.Embedding(V, C)            <- "what token is this?"     indexed by token id
wpe  = nn.Embedding(block_size, C)   <- "where in the sequence?"  indexed by 0, 1, 2, ...
```

`wpe` row 5 is a learned vector meaning "I am at position 5". Add it to the token's vector and now the token knows where it is.

```
tok_emb = wte(idx)                   (B, T, C)
pos_emb = wpe(arange(T))                (T, C)      <- NO batch axis
x = tok_emb + pos_emb                (B, T, C)      <- broadcast
```

### `torch.arange(T)`

`torch.arange(T)` builds `[0, 1, 2, ..., T-1]` — shape `(T,)`. That's literally the list of positions. Feed it to `wpe` and (by the shape rule from 2.3) you get `(T,) -> (T, C)`: one vector per position.

Always pass `device=` to `arange`, or you allocate on the CPU and eat a silent host-to-device sync on every single forward pass.

### The broadcast, properly

`tok_emb` is `(B, T, C)`. `pos_emb` is only `(T, C)` — it has no batch axis. How does adding them work?

**Broadcasting** is PyTorch's rule for elementwise ops on mismatched shapes. It works in two steps.

**Step 1 — right-align the shapes and pad the left with 1s:**

```
tok_emb :  (B, T, C)
pos_emb :     (T, C)
           ^^^ line them up at the RIGHT edge, then pad the shorter one with 1s on the left

tok_emb :  (B, T, C)
pos_emb :  (1, T, C)      <- the 1 is inserted for you, virtually
```

**Step 2 — walk the axes. For each one, exactly one of two things must be true:**

1. the sizes are **equal**, or
2. one of them is **1** — in which case that operand is *virtually repeated* along that axis.

```
axis -1:  C vs C  ->  equal.  OK.
axis -2:  T vs T  ->  equal.  OK.
axis -3:  B vs 1  ->  one is 1. Repeat pos_emb across the batch.
result :  (B, T, C)
```

If neither condition holds on some axis, you get a `RuntimeError`.

**What this means semantically:** every sequence in the batch gets the *same* position vectors. Position 5 means the same thing in row 0 as in row 63. That's obviously what we want, and right-alignment makes it fall out for free.

**And "virtually repeated" is literal.** Broadcasting allocates nothing. It sets the **stride** of the broadcast axis to `0`, which means "when you step along this axis, don't move in memory at all." Reading `pos_emb[0]`, `pos_emb[1]`, ..., `pos_emb[63]` all read the same bytes. Free.

### The cases worth memorising

```
(B, T, C) + (T, C)      -> (B, T, C)     works. This is our case.
(B, T, C) + (C,)        -> (B, T, C)     works: a per-channel bias
(B, T, C) + (B, T)      -> ERROR         right-aligns as C vs T. Mismatch.
(B, T, C) + (B, T, 1)   -> (B, T, C)     works: one scalar per token. Add the axis yourself.
(B, T, C) + (B, 1, C)   -> (B, T, C)     works: same vector at every position
(4, 1)    + (1, 3)      -> (4, 3)        WORKS. And this is the dangerous one.
```

That last line is where real bugs live. A `(B, 1)` and a `(1, B)` will happily broadcast into `(B, B)` instead of erroring — you silently get a tensor `B` times bigger than you wanted, and a loss that looks *almost* right. Rule of thumb: **if a tensor is mysteriously larger than expected, look for a missing `squeeze` or `unsqueeze`.**

In [ ]:
B_, T_, C_ = 2, 3, 4
tok_emb = torch.zeros(B_, T_, C_)
pos_emb = torch.arange(T_ * C_).float().reshape(T_, C_)

print("arange(T) is literally the list of positions:")
print(f"  torch.arange({T_}) = {torch.arange(T_).tolist()}   shape {tuple(torch.arange(T_).shape)}")
print(f"  wpe(arange(T)) would give shape (T, C) = ({T_}, {C_})\n")

sh("tok_emb (B, T, C)", tok_emb)
sh("pos_emb    (T, C)", pos_emb)
print("\nright-align and pad with 1s:")
print(f"  tok_emb : (B, T, C) = ({B_}, {T_}, {C_})")
print(f"  pos_emb :    (T, C) = (   {T_}, {C_})  ->  (1, {T_}, {C_})  <- the 1 is virtual")

x = tok_emb + pos_emb
sh("\nsum     (B, T, C)", x)
print("\nBoth batch rows got the SAME position vectors -- that is the entire point:")
print(x)
assert torch.equal(x[0], x[1])
print("[OK] x[0] == x[1]. Position 5 means the same thing in every sequence.")

arange(T) is literally the list of positions:
  torch.arange(3) = [0, 1, 2]   shape (3,)
  wpe(arange(T)) would give shape (T, C) = (3, 4)

tok_emb (B, T, C)            (2, 3, 4)
pos_emb    (T, C)            (3, 4)

right-align and pad with 1s:
  tok_emb : (B, T, C) = (2, 3, 4)
  pos_emb :    (T, C) = (   3, 4)  ->  (1, 3, 4)  <- the 1 is virtual

sum     (B, T, C)           (2, 3, 4)

Both batch rows got the SAME position vectors -- that is the entire point:
tensor([[[ 0.,  1.,  2.,  3.],
         [ 4.,  5.,  6.,  7.],
         [ 8.,  9., 10., 11.]],

        [[ 0.,  1.,  2.,  3.],
         [ 4.,  5.,  6.,  7.],
         [ 8.,  9., 10., 11.]]])
[OK] x[0] == x[1]. Position 5 means the same thing in every sequence.


In [ ]:
print("--- what broadcasts, what doesn't ---\n")
cases = [((2,3,4), (3,4),   "our case: (B,T,C) + (T,C)"),
         ((2,3,4), (4,),    "per-channel bias"),
         ((2,3,4), (2,3),   "right-aligns C vs T -> mismatch"),
         ((2,3,4), (2,3,1), "one scalar per token"),
         ((2,3,4), (2,1,4), "same vector at every position"),
         ((4,1),   (1,3),   "THE DANGEROUS ONE")]
for s1, s2, note in cases:
    try:
        r = tuple((torch.zeros(s1) + torch.zeros(s2)).shape)
        flag = "  <-- SILENT blowup! (4,1)+(1,3) -> (4,3), no error" if r == (4,3) else ""
        print(f"  {str(s1):<10} + {str(s2):<8} -> {str(r):<11} {note}{flag}")
    except RuntimeError:
        print(f"  {str(s1):<10} + {str(s2):<8} -> {'ERROR':<11} {note}")

# broadcasting allocates NOTHING -- it sets stride 0
e = torch.zeros(3, 1).expand(3, 4)
print(f"\nexpand((3,1) -> (3,4)):")
print(f"  shape  : {tuple(e.shape)}")
print(f"  stride : {e.stride()}      <- the 0 means 'stepping along this axis moves 0 bytes'")
print(f"  storage: {e.untyped_storage().nbytes()//e.element_size()} elements backing a {e.numel()}-element tensor")
print("\nBroadcasting is free. 'Virtually repeated' is literal: stride 0, no copy.")

--- what broadcasts, what doesn't ---

  (2, 3, 4)  + (3, 4)   -> (2, 3, 4)   our case: (B,T,C) + (T,C)
  (2, 3, 4)  + (4,)     -> (2, 3, 4)   per-channel bias
  (2, 3, 4)  + (2, 3)   -> ERROR       right-aligns C vs T -> mismatch
  (2, 3, 4)  + (2, 3, 1) -> (2, 3, 4)   one scalar per token
  (2, 3, 4)  + (2, 1, 4) -> (2, 3, 4)   same vector at every position
  (4, 1)     + (1, 3)   -> (4, 3)      THE DANGEROUS ONE  <-- SILENT blowup! (4,1)+(1,3) -> (4,3), no error

expand((3,1) -> (3,4)):
  shape  : (3, 4)
  stride : (1, 0)      <- the 0 means 'stepping along this axis moves 0 bytes'
  storage: 3 elements backing a 12-element tensor

Broadcasting is free. 'Virtually repeated' is literal: stride 0, no copy.


---
# Part 3 — LayerNorm: the axis question

## 3.1 What normalisation means here

LayerNorm does one thing: **it takes each token's `C`-vector and rescales it so it has mean 0 and standard deviation 1**, using only that vector's own numbers. Then it applies two learned parameters.

The formula:

```
        x - mean(x)
xhat = ---------------
       sqrt(var(x) + eps)

out = gamma * xhat + beta
```

That's it. But three details in that formula are load-bearing and each one is a real bug if you get it wrong.

## 3.2 Hand calculation

Take one token with `C = 8`:

```
x = [2, 4, 4, 4, 5, 5, 7, 9]
```

**Step 1 — the mean:**

```
mean = (2 + 4 + 4 + 4 + 5 + 5 + 7 + 9) / 8
     = 40 / 8
     = 5
```

**Step 2 — the variance.** Take each deviation from the mean, square it, average:

```
deviations   : [2-5, 4-5, 4-5, 4-5, 5-5, 5-5, 7-5, 9-5]
             = [ -3,  -1,  -1,  -1,   0,   0,   2,   4]
squared      : [  9,   1,   1,   1,   0,   0,   4,  16]
sum          : 32
var = 32 / 8 = 4                     <- divide by N (=8), NOT by N-1
```

**Step 3 — the standard deviation:**

```
std = sqrt(4) = 2
```

**Step 4 — normalise:**

```
xhat = (x - 5) / 2
     = [(2-5)/2, (4-5)/2, (4-5)/2, (4-5)/2, (5-5)/2, (5-5)/2, (7-5)/2, (9-5)/2]
     = [ -1.5,    -0.5,    -0.5,    -0.5,     0,       0,       1,       2   ]
```

Check: those 8 numbers have mean 0 and std 1. That's the whole operation.

**Step 5 — scale and shift by the learned parameters:**

```
out = gamma * xhat + beta            gamma and beta are both shape (C,) = (8,)
```

At initialisation `gamma = 1` and `beta = 0`, so `out == xhat`. The model can learn to undo the normalisation if it wants to — that's what `gamma` and `beta` are for.

## 3.3 The three details that are bugs if you get them wrong

### Detail 1: the variance is **biased** (divide by N, not N-1)

`nn.LayerNorm` divides by `N`. But PyTorch's `x.var()` defaults to `unbiased=True`, which divides by `N-1`.

Why do these two conventions exist at all? `N-1` (Bessel's correction) is for when you have a *sample* from a larger population and want an unbiased estimate of the population's variance. But LayerNorm isn't estimating anything — the `C` numbers *are* the whole population. There's no larger set to infer about. So `N` is correct.

The practical consequence: hand-roll a LayerNorm with `x.var(dim=-1)` and you'll be off by a factor of `sqrt(C/(C-1))`. At `C=8` that's 6.9% — big enough to break, small enough to look like a mystery. Use `unbiased=False` (or `correction=0`).

### Detail 2: `eps` goes **inside** the sqrt

```
CORRECT:  (x - mu) / sqrt(var + eps)
WRONG:    (x - mu) / (sqrt(var) + eps)
```

`eps` (default `1e-5`) exists to stop division by zero when a token's vector happens to be constant (`var = 0`). Inside the sqrt it does that job cleanly. Outside, it adds a small bias to every normalisation, not just the degenerate ones.

### Detail 3: `keepdim=True`

`x.mean(dim=-1)` on a `(B, T, C)` tensor gives `(B, T)` — **the axis is gone**. Now try `x - x.mean(dim=-1)`: broadcasting right-aligns `(B,T,C)` against `(B,T)`, compares `C` vs `T`, and errors (or worse, if `C == T`, silently does something insane).

`keepdim=True` gives `(B, T, 1)` instead — the axis survives with size 1, which broadcasts against `(B, T, C)` exactly as you intended.

## 3.4 Which axis — and why not the others

This is the conceptual heart of the section.

LayerNorm normalises over **`dim=-1`, the channel axis. Only that one.** Here's what the alternatives would actually mean:

| Normalise over | Stats shape | What it would mean | Verdict |
|---|---|---|---|
| **`C`** (LayerNorm) | `(B, T, 1)` | each token normalises using its own numbers | **correct** |
| `B` (BatchNorm) | `(1, T, C)` | token 5 of story 3 depends on token 5 of story 40 | breaks: batch-dependent, and inference has no batch |
| `T` | `(B, 1, C)` | position 0's output depends on position 200 | **catastrophic — leaks the future** |

That last row is the deep reason autoregressive language models use LayerNorm and nothing else.

**A causal model must guarantee that the output at position `t` depends only on inputs at positions `<= t`.** Normalising over `T` would compute a mean across *all* positions — including future ones — and mix it into every output. Your training loss would look fantastic (the model can see the answers). Your generation would be garbage (at sampling time, the future doesn't exist yet, so the statistics the model learned to rely on are simply gone).

And nothing would ever raise an exception.

So here's the rule that governs every operation in a GPT:

> **Every op must either be strictly per-token (LayerNorm, MLP), or explicitly masked (attention). There is no third option.**

Check it against the architecture: LayerNorm is per-token. The MLP is per-token (`nn.Linear` only touches the last axis). Attention mixes across `T` — and is therefore masked. That's the complete list. Nothing else in the model moves information between positions.

## 3.5 The parameter count is almost nothing

`gamma` is `(C,)`. `beta` is `(C,)`. Two vectors of length `C`, **shared across all `B*T` tokens**.

At `C=384` that's 768 parameters per LayerNorm — a rounding error in a 30M model. Yet each call performs `B*T = 16,384` completely independent normalisations. Tiny parameter count, enormous amount of work. That asymmetry is why LayerNorm shows up as a real chunk of your kernel time despite being invisible in the parameter table.

In [ ]:
x = torch.tensor([2., 4., 4., 4., 5., 5., 7., 9.])
C_ln = 8

print(f"x = {x.tolist()}\n")
mean = x.mean()
print(f"Step 1  mean = sum / N = {x.sum().item():.0f} / {C_ln} = {mean.item()}")

dev = x - mean
sq = dev ** 2
print(f"\nStep 2  deviations : {dev.tolist()}")
print(f"        squared    : {sq.tolist()}")
print(f"        sum        : {sq.sum().item():.0f}")
var_biased = x.var(unbiased=False)
var_unbiased = x.var(unbiased=True)
print(f"        var = {sq.sum().item():.0f} / {C_ln} = {var_biased.item()}   <- divide by N")
print(f"\n        for comparison, /(N-1) gives {var_unbiased.item():.4f}  <- torch's DEFAULT. Wrong here.")
print(f"        the ratio is sqrt(C/(C-1)) = {math.sqrt(C_ln/(C_ln-1)):.4f} = "
      f"{(math.sqrt(C_ln/(C_ln-1))-1)*100:.1f}% error if you use the default")

std = torch.sqrt(var_biased)
print(f"\nStep 3  std = sqrt({var_biased.item()}) = {std.item()}")

xhat = (x - mean) / torch.sqrt(var_biased + 1e-5)
print(f"\nStep 4  xhat = (x - {mean.item():.0f}) / {std.item():.0f}")
print(f"        = {[round(v, 4) for v in xhat.tolist()]}")
print(f"        check: mean = {xhat.mean().item():.2e}  std = {xhat.std(unbiased=False).item():.4f}")

ln = nn.LayerNorm(C_ln)                 # gamma=1, beta=0 at init
print(f"\ntorch nn.LayerNorm gives:")
print(f"        {[round(v, 4) for v in ln(x).tolist()]}")
assert torch.allclose(xhat, ln(x), atol=1e-6)
print("\n[OK] hand calculation == nn.LayerNorm, exactly")
print(f"     gamma shape: {tuple(ln.weight.shape)}   beta shape: {tuple(ln.bias.shape)}")
print(f"     at init gamma=1, beta=0, so out == xhat. The model can learn otherwise.")

x = [2.0, 4.0, 4.0, 4.0, 5.0, 5.0, 7.0, 9.0]

Step 1  mean = sum / N = 40 / 8 = 5.0

Step 2  deviations : [-3.0, -1.0, -1.0, -1.0, 0.0, 0.0, 2.0, 4.0]
        squared    : [9.0, 1.0, 1.0, 1.0, 0.0, 0.0, 4.0, 16.0]
        sum        : 32
        var = 32 / 8 = 4.0   <- divide by N

        for comparison, /(N-1) gives 4.5714  <- torch's DEFAULT. Wrong here.
        the ratio is sqrt(C/(C-1)) = 1.0690 = 6.9% error if you use the default

Step 3  std = sqrt(4.0) = 2.0

Step 4  xhat = (x - 5) / 2
        = [-1.5, -0.5, -0.5, -0.5, 0.0, 0.0, 1.0, 2.0]
        check: mean = -2.98e-08  std = 1.0000

torch nn.LayerNorm gives:
        [-1.5, -0.5, -0.5, -0.5, 0.0, 0.0, 1.0, 2.0]

[OK] hand calculation == nn.LayerNorm, exactly
     gamma shape: (8,)   beta shape: (8,)
     at init gamma=1, beta=0, so out == xhat. The model can learn otherwise.


In [ ]:
# --- Detail 2: eps inside vs outside the sqrt ---
print("--- eps placement ---")
x_const = torch.tensor([3., 3., 3., 3.])       # var = 0. The degenerate case eps exists for.
mu, v = x_const.mean(), x_const.var(unbiased=False)
print(f"a constant vector {x_const.tolist()}: var = {v.item()}")
print(f"  correct  (x-mu)/sqrt(var+eps) = "
      f"{((x_const-mu)/torch.sqrt(v+1e-5)).tolist()}   <- finite. Good.")
print(f"  no eps   (x-mu)/sqrt(var)     = "
      f"{((x_const-mu)/torch.sqrt(v)).tolist()}   <- 0/0 = nan")

# --- Detail 3: keepdim ---
print("\n--- keepdim: the one that bites ---")
x3 = torch.randn(2, 3, 4)                       # (B, T, C)
sh("x", x3)
sh("x.mean(dim=-1)              ", x3.mean(dim=-1))
sh("x.mean(dim=-1, keepdim=True)", x3.mean(dim=-1, keepdim=True))
print("   ^ the first LOST the axis. The second kept it, as size 1.")
try:
    _ = x3 - x3.mean(dim=-1)
except RuntimeError as e:
    print(f"\n  x - x.mean(dim=-1)            -> RuntimeError: {str(e)[:52]}...")
    print(f"     (broadcasting right-aligned (2,3,4) vs (2,3): compared C=4 to T=3)")
print(f"  x - x.mean(dim=-1, keepdim=True) -> OK, shape "
      f"{tuple((x3 - x3.mean(dim=-1, keepdim=True)).shape)}")

--- eps placement ---
a constant vector [3.0, 3.0, 3.0, 3.0]: var = 0.0
  correct  (x-mu)/sqrt(var+eps) = [0.0, 0.0, 0.0, 0.0]   <- finite. Good.
  no eps   (x-mu)/sqrt(var)     = [nan, nan, nan, nan]   <- 0/0 = nan

--- keepdim: the one that bites ---
x                            (2, 3, 4)
x.mean(dim=-1)               (2, 3)
x.mean(dim=-1, keepdim=True) (2, 3, 1)
   ^ the first LOST the axis. The second kept it, as size 1.

  x - x.mean(dim=-1)            -> RuntimeError: The size of tensor a (4) must match the size of tens...
     (broadcasting right-aligned (2,3,4) vs (2,3): compared C=4 to T=3)
  x - x.mean(dim=-1, keepdim=True) -> OK, shape (2, 3, 4)


In [ ]:
# --- 3.4 verified: LayerNorm is strictly PER-TOKEN, so it cannot leak the future ---
ln4 = nn.LayerNorm(4)
x = torch.randn(1, 5, 4)                       # one sequence, 5 tokens, C=4
out_full = ln4(x)

# Now change ONLY the last token and re-run.
x_changed = x.clone()
x_changed[0, 4] = torch.randn(4) * 100         # wreck the FUTURE token
out_changed = ln4(x_changed)

print("Change token 4 (the future). Do earlier tokens' outputs move?\n")
for t in range(5):
    d = (out_full[0, t] - out_changed[0, t]).abs().max().item()
    tag = "<- the one we changed" if t == 4 else ("SAME" if d == 0 else "LEAKED!")
    print(f"  token {t}: max change = {d:.3e}   {tag}")
assert torch.equal(out_full[0, :4], out_changed[0, :4])
print("\n[OK] tokens 0-3 are bit-identical. LayerNorm cannot see the future,")
print("     because it only ever looks at ONE token's own C numbers.")
print("\nNow imagine normalising over T instead: the mean would include token 4,")
print("so ALL FIVE outputs would move. Training loss would improve. Generation")
print("would collapse. And nothing would raise an exception.")

print(f"\nB*T independent normalisations per call, {ln4.weight.numel()+ln4.bias.numel()} "
      f"parameters total.")

Change token 4 (the future). Do earlier tokens' outputs move?

  token 0: max change = 0.000e+00   SAME
  token 1: max change = 0.000e+00   SAME
  token 2: max change = 0.000e+00   SAME
  token 3: max change = 0.000e+00   SAME
  token 4: max change = 1.511e+00   <- the one we changed

[OK] tokens 0-3 are bit-identical. LayerNorm cannot see the future,
     because it only ever looks at ONE token's own C numbers.

Now imagine normalising over T instead: the mean would include token 4,
so ALL FIVE outputs would move. Training loss would improve. Generation
would collapse. And nothing would raise an exception.

B*T independent normalisations per call, 8 parameters total.


In [ ]:
class LayerNorm(nn.Module):
    # nn.LayerNorm, but with an optional-bias switch, which torch does not expose.
    # GPT-2 used bias=True. Modern models (LLaMA etc.) drop it: it buys nothing
    # measurable and costs a kernel argument.
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))                     # gamma (C,)
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None    # beta  (C,)

    def forward(self, x):
        # x: (B, T, C) -> (B, T, C). Normalises over the LAST axis only.
        return F.layer_norm(x, self.weight.shape, self.weight, self.bias, 1e-5)

_ln = LayerNorm(4, bias=True)
sh("LayerNorm((2,3,4))", _ln(torch.randn(2, 3, 4)))
print("Shape in == shape out. LayerNorm never changes the residual stream's shape.")
print("It is one of only two op types in a GPT: strictly per-token. (The other is")
print("attention, which mixes across T and is therefore masked.)")

LayerNorm((2,3,4))           (2, 3, 4)
Shape in == shape out. LayerNorm never changes the residual stream's shape.
It is one of only two op types in a GPT: strictly per-token. (The other is
attention, which mixes across T and is therefore masked.)


---
# Part 4 — Attention

This is the part worth slowing down for. Everything else in a transformer is a `Linear` and a nonlinearity. Take it in five passes: the shape map, a full hand computation, the `view`/`transpose` trap, the mask, and the cost.

## 4.1 The shape map, walked slowly

Here is the whole module. Don't try to absorb it in one go — we'll walk each arrow.

```
x                                             (B, T, C)
 |
 |  c_attn = Linear(C, 3C)          one matmul produces Q, K and V at once
 v
qkv                                           (B, T, 3C)
 |
 |  .split(C, dim=2)
 v
q, k, v                                 each  (B, T, C)
 |
 |  .view(B, T, nh, hs).transpose(1, 2)      <-- THE step people get wrong
 v
q, k, v                                 each  (B, nh, T, hs)
 |
 |  q @ k.transpose(-2, -1)                  (B,nh,T,hs) @ (B,nh,hs,T)
 v
att                                           (B, nh, T, T)      <-- quadratic in T
 |
 |  / sqrt(hs)  ->  masked_fill(-inf)  ->  softmax(dim=-1)
 v
att                                           (B, nh, T, T)      rows sum to 1
 |
 |  att @ v                                  (B,nh,T,T) @ (B,nh,T,hs)
 v
y                                             (B, nh, T, hs)
 |
 |  .transpose(1, 2).contiguous().view(B, T, C)
 v
y                                             (B, T, C)
 |
 |  c_proj = Linear(C, C)
 v
out                                           (B, T, C)          same shape we started with
```

### The arrows, one at a time

**`x (B,T,C) -> c_attn -> qkv (B,T,3C)`.** A `Linear(C, 3C)` maps each token's `C` numbers to `3C` numbers. Why `3C`? Because we need three different things per token — a query, a key, and a value — each of size `C`. One matmul makes all three.

**`qkv (B,T,3C) -> split -> 3 x (B,T,C)`.** `split(C, dim=2)` chops axis 2 into chunks of size `C`. `3C / C = 3` chunks. Channels `0:C` become `q`, `C:2C` become `k`, `2C:3C` become `v`. This is a free view — no copy.

**`(B,T,C) -> view + transpose -> (B,nh,T,hs)`.** Split each token's `C` channels into `nh` groups of `hs`, then move the head axis up next to the batch axis. Section 4.4 is entirely about why this is two steps and not one.

**`q @ k^T -> (B,nh,T,T)`.** For each head, every query token dots against every key token. `T` queries x `T` keys = a `(T, T)` grid of scores. This is the only tensor that grows as `T²`.

**`softmax -> att @ v -> (B,nh,T,hs)`.** Each row of `att` is a probability distribution over the keys. Multiplying by `v` takes a weighted average of value vectors. Output: one `hs`-vector per query token, per head. Same shape we fed in.

**`transpose + view -> (B,T,C)`, then `c_proj`.** Glue the heads back together and mix them with a final `Linear(C, C)`.

### Two things to carry away

**In `(B, T, C)`, out `(B, T, C)`.** Attention is shape-preserving. It has to be — it writes into the residual stream.

**`(B, nh, T, T)` is the only tensor whose size depends on `T` squared.** Everything else is linear in `T`. Every inference optimisation you have ever heard of is about that one tensor.

### Why one `Linear(C, 3C)` instead of three `Linear(C, C)`?

Mathematically identical. `c_attn.weight` is `(3C, C)`; slice it into three `(C, C)` blocks and you recover `W_q`, `W_k`, `W_v`.

But one matmul of `(B*T, C) @ (C, 3C)` launches **one** GPU kernel with three times the arithmetic intensity, instead of three skinny ones. Three separate matmuls means three kernel launches, three trips to fetch weights from memory, three chances to be bottlenecked on launch latency rather than math.

At small `B*T` — which is exactly the situation during autoregressive decoding, where `B*T` might be **1** — that difference is most of your runtime. The fused QKV projection is the first and cheapest kernel fusion in the stack, and every serious implementation does it.

## 4.2 The hand calculation

Small enough to do entirely on paper. **`B=1, T=3, C=4, nh=2, hs=2`**, and we set `W_q = W_k = W_v = I` (the identity) and `W_o = I`, so that `q = k = v = x` and the arithmetic is visible.

```
x = [[1, 0, 1, 0],     <- token 0
     [0, 1, 0, 1],     <- token 1
     [1, 1, 0, 0]]     <- token 2          (T, C) = (3, 4)
```

### Step 1 — split into heads

`hs = C // nh = 4 // 2 = 2`. Head 0 takes channels `[0:2]`. Head 1 takes channels `[2:4]`.

**The split is along `C`, not along `T`.** Each head sees **all 3 tokens**, but only **half of each token's vector**. Burn that in — it's the whole content of section 4.4.

```
head 0 (channels 0:2)          head 1 (channels 2:4)
q0 = k0 = v0 =                 q1 = k1 = v1 =
 [[1, 0],                       [[1, 0],
  [0, 1],                        [0, 1],
  [1, 1]]                        [0, 0]]     <- token 2 is all-zero in head 1's channels
```

### Step 2 — scores, `q @ k^T`, for head 0

Entry `(i, j)` = how much query `i` wants key `j` = the dot product of row `i` of `q0` with row `j` of `k0`.

```
             k0[0]=[1,0]  k0[1]=[0,1]  k0[2]=[1,1]
q0[0]=[1,0]       1            0            1
q0[1]=[0,1]       0            1            1
q0[2]=[1,1]       1            1            2

scores_0 = [[1, 0, 1],
            [0, 1, 1],
            [1, 1, 2]]          (T, T) = (3, 3)
```

Check one by hand: `q0[2] · k0[2]` = `[1,1] · [1,1]` = `1*1 + 1*1` = `2`. ✓

### Step 3 — scale by `1/sqrt(hs) = 1/sqrt(2) = 0.7071`

```
scores_0 / sqrt(2) = [[0.7071, 0.0000, 0.7071],
                      [0.0000, 0.7071, 0.7071],
                      [0.7071, 0.7071, 1.4142]]
```

(Section 4.7 derives why this factor is here.)

### Step 4 — the causal mask

Row `i` may only attend to columns `j <= i`. Everything strictly above the diagonal becomes `-inf`:

```
[[0.7071,  -inf,   -inf ],      <- token 0 sees only itself
 [0.0000, 0.7071,  -inf ],      <- token 1 sees tokens 0, 1
 [0.7071, 0.7071, 1.4142]]      <- token 2 sees tokens 0, 1, 2
```

### Step 5 — softmax over the last axis (over `j`, the keys)

```
row 0: only one live entry             -> [1.0000, 0.0000, 0.0000]

row 1: exp(0) = 1,  exp(0.7071) = 2.0281,  sum = 3.0281
       -> [1/3.0281, 2.0281/3.0281, 0]    = [0.3302, 0.6698, 0.0000]

row 2: exp(0.7071)=2.0281, exp(0.7071)=2.0281, exp(1.4142)=4.1133,  sum = 8.1695
       -> [0.2483, 0.2483, 0.5035]

att_0 = [[1.0000, 0.0000, 0.0000],
         [0.3302, 0.6698, 0.0000],
         [0.2483, 0.2483, 0.5035]]      every row sums to 1; upper triangle is EXACTLY 0
```

### Step 6 — `att_0 @ v0`: a weighted average of value vectors

```
y0[0] = 1.0000*[1,0]                                      = [1.0000, 0.0000]
y0[1] = 0.3302*[1,0] + 0.6698*[0,1]                       = [0.3302, 0.6698]
y0[2] = 0.2483*[1,0] + 0.2483*[0,1] + 0.5035*[1,1]        = [0.7517, 0.7517]
```

### Head 1, same procedure

`q1[2] = [0, 0]`, so its scores against every key are `0`. After masking, row 2 is `[0, 0, 0]`, and softmax of all-equal values is **uniform**: `[1/3, 1/3, 1/3]`.

```
att_1 = [[1.0000, 0.0000, 0.0000],
         [0.3302, 0.6698, 0.0000],
         [0.3333, 0.3333, 0.3333]]

y1[2] = (1/3)([1,0] + [0,1] + [0,0]) = [0.3333, 0.3333]
```

### Step 7 — concatenate the heads back along `C`

```
      head 0 out    head 1 out
y = [[1.0000, 0.0000, 1.0000, 0.0000],
     [0.3302, 0.6698, 0.3302, 0.6698],
     [0.7517, 0.7517, 0.3333, 0.3333]]      (T, C) = (3, 4)
```

**Look at row 2.** Head 0 produced `[0.7517, 0.7517]` and head 1 produced `[0.3333, 0.3333]`. Same token, same position, **genuinely different attention distributions** — head 0 put 50% of its mass on token 2, head 1 spread it uniformly.

That divergence is the entire argument for multi-head: `nh` separate `(T, T)` relational patterns computed in parallel, at **exactly the same FLOP cost** as one head of size `C`, because the head axis is just a free reshape.

Now let's check every number of it.

In [ ]:
torch.set_printoptions(precision=4, sci_mode=False)

B_, T_, C_, nh_ = 1, 3, 4, 2
hs_ = C_ // nh_

x = torch.tensor([[[1., 0., 1., 0.],
                   [0., 1., 0., 1.],
                   [1., 1., 0., 0.]]])                       # (B, T, C) = (1, 3, 4)
sh("x", x)

# --- Step 1: split into heads ---------------------------------------------
q = k = v = x
qh = q.view(B_, T_, nh_, hs_).transpose(1, 2)                 # (B, nh, T, hs)
kh = k.view(B_, T_, nh_, hs_).transpose(1, 2)
vh = v.view(B_, T_, nh_, hs_).transpose(1, 2)
sh("q after head-split", qh)
print(f"\nhead 0 q (channels 0:2):\n{qh[0, 0]}")
print(f"head 1 q (channels 2:4):\n{qh[0, 1]}       <- token 2 is [0,0] here")
print("\nBoth heads see ALL 3 tokens. Each sees only HALF of each token's vector.")

# --- Steps 2-3: scores, scaled --------------------------------------------
raw = qh @ kh.transpose(-2, -1)
print(f"\nraw scores head 0 (q @ k^T):\n{raw[0,0]}")
att = raw * (1.0 / math.sqrt(hs_))                            # (B, nh, T, T)
sh("\natt = q @ k^T / sqrt(hs)", att)
print(f"\nscaled by 1/sqrt({hs_}) = {1/math.sqrt(hs_):.4f}:\n{att[0, 0]}")

# --- Step 4: causal mask ---------------------------------------------------
mask = torch.tril(torch.ones(T_, T_)).view(1, 1, T_, T_)      # (1, 1, T, T)
att = att.masked_fill(mask == 0, float('-inf'))
print(f"\nafter the causal mask, head 0:\n{att[0, 0]}")

# --- Step 5: softmax -------------------------------------------------------
att = F.softmax(att, dim=-1)
print(f"\nsoftmax head 0:\n{att[0, 0]}")
print(f"softmax head 1:\n{att[0, 1]}    <- row 2 is uniform, exactly as traced")
print(f"\nrow sums (must all be 1): {att.sum(dim=-1).flatten().tolist()}")

# --- Steps 6-7: weighted sum, merge heads ---------------------------------
y = att @ vh                                                  # (B, nh, T, hs)
sh("\ny = att @ v", y)
y = y.transpose(1, 2).contiguous().view(B_, T_, C_)           # (B, T, C)
sh("y merged", y)
print(f"\n{y[0]}")

x                            (1, 3, 4)
q after head-split           (1, 2, 3, 2)

head 0 q (channels 0:2):
tensor([[1., 0.],
        [0., 1.],
        [1., 1.]])
head 1 q (channels 2:4):
tensor([[1., 0.],
        [0., 1.],
        [0., 0.]])       <- token 2 is [0,0] here

Both heads see ALL 3 tokens. Each sees only HALF of each token's vector.

raw scores head 0 (q @ k^T):
tensor([[1., 0., 1.],
        [0., 1., 1.],
        [1., 1., 2.]])

att = q @ k^T / sqrt(hs)    (1, 2, 3, 3)

scaled by 1/sqrt(2) = 0.7071:
tensor([[0.7071, 0.0000, 0.7071],
        [0.0000, 0.7071, 0.7071],
        [0.7071, 0.7071, 1.4142]])

after the causal mask, head 0:
tensor([[0.7071,   -inf,   -inf],
        [0.0000, 0.7071,   -inf],
        [0.7071, 0.7071, 1.4142]])

softmax head 0:
tensor([[1.0000, 0.0000, 0.0000],
        [0.3302, 0.6698, 0.0000],
        [0.2483, 0.2483, 0.5035]])
softmax head 1:
tensor([[1.0000, 0.0000, 0.0000],
        [0.3302, 0.6698, 0.0000],
        [0.3333, 0.3333, 0.3333]])    <- 

In [ ]:
# The moment of truth: does PyTorch agree with the paper trace?
expected = torch.tensor([[1.0000, 0.0000, 1.0000, 0.0000],
                         [0.3302, 0.6698, 0.3302, 0.6698],
                         [0.7517, 0.7517, 0.3333, 0.3333]])

assert torch.allclose(y[0], expected, atol=1e-4), f"\ngot\n{y[0]}\nwant\n{expected}"
print("[OK] PyTorch output matches the hand calculation to 4 decimal places.\n")
print("Row 2 of the output:")
print(f"  head 0 said : {[round(v,4) for v in y[0, 2, :2].tolist()]}")
print(f"  head 1 said : {[round(v,4) for v in y[0, 2, 2:].tolist()]}")
print("\nSame token. Same position. Two different views of the context.")
print("Head 0 put 50% of its mass on token 2; head 1 spread it uniformly.")
print("That divergence IS multi-head attention -- and it cost zero extra FLOPs")
print("over one big head, because the head axis is a free reshape.")

[OK] PyTorch output matches the hand calculation to 4 decimal places.

Row 2 of the output:
  head 0 said : [0.7517, 0.7517]
  head 1 said : [0.3333, 0.3333]

Same token. Same position. Two different views of the context.
Head 0 put 50% of its mass on token 2; head 1 spread it uniformly.
That divergence IS multi-head attention -- and it cost zero extra FLOPs
over one big head, because the head axis is a free reshape.


## 4.3 `view` vs `reshape` vs `transpose` vs `permute` vs `contiguous`

This is the highest-value section in the notebook. Nearly every shape bug in transformer code lives here.

### Strides: what a tensor actually is

A tensor is **a flat 1-D block of memory, plus a `(shape, stride)` interpretation.**

That's the whole idea. The memory is always flat — a straight line of numbers. The `shape` says how to *pretend* it has axes. The `stride` says how far to jump in that flat line to move one step along each axis.

```python
t = torch.arange(24).reshape(2, 3, 4)
t.stride()  ->  (12, 4, 1)
```

Read it out loud: *"to move one step along axis 0, jump 12 elements. Along axis 1, jump 4. Along axis 2, jump 1."*

Why those numbers? A `(2, 3, 4)` tensor is 2 slabs of 3 rows of 4 numbers. Moving one slab = skipping `3*4 = 12` numbers. Moving one row = skipping `4`. Moving one element = skipping `1`.

So element `(i, j, k)` lives at flat position `12i + 4j + k`.

**Contiguous** means exactly this: the strides are the natural descending products, and memory order matches index order. Walk the tensor in index order and you walk memory in a straight line.

### `transpose` / `permute` do not move data

```python
t.transpose(1, 2).stride()  ->  (12, 1, 4)
```

Look at what happened: the strides **swapped**. Same memory, different reading instructions. Zero copy, O(1), instant.

But now the strides are `(12, 1, 4)` — **not** descending. The tensor is **not contiguous**. Its logical order no longer matches its memory order.

### `view` requires contiguity — and that's a feature

`view` is a pure metadata change: keep the memory, reinterpret shape and strides. It can only do that if reading the flat buffer in order happens to produce the elements in the new logical order.

On a transposed tensor it can't. So it **raises**.

```python
t.transpose(1,2).view(2, 12)                  # RuntimeError
t.transpose(1,2).contiguous().view(2, 12)     # works -- contiguous() COPIES to fix the layout
t.transpose(1,2).reshape(2, 12)               # works -- reshape = view if possible, else contiguous().view()
```

That error is PyTorch protecting you. Silently returning wrong values would be far worse than an exception.

### So why not always use `reshape`?

Because **`reshape` hides a memory copy behind a name that sounds free.**

`view` fails loudly and tells you a copy is about to happen. `reshape` just does it. In a training loop running a million times, you want to *know* where your copies are.

`contiguous().view()` says, explicitly: *"I am paying for a copy here, deliberately, and I know it."* That's why nanoGPT and every reference implementation writes it out longhand.

In [ ]:
t = torch.arange(24).reshape(2, 3, 4)
print(f"t.shape           {tuple(t.shape)}")
print(f"t.stride()        {t.stride()}")
print(f"  read it: 'one step along axis 0 = jump 12 elements (a whole 3x4 slab)'")
print(f"           'one step along axis 1 = jump 4  elements (a row)'")
print(f"           'one step along axis 2 = jump 1  element'")
print(f"t.is_contiguous() {t.is_contiguous()}   <- strides descend naturally: 12 > 4 > 1")
print(f"\nelement (1,2,3) is at flat offset 12*1 + 4*2 + 3 = {12*1+4*2+3}")
print(f"  t[1,2,3]          = {t[1,2,3].item()}")
print(f"  t.flatten()[{12*1+4*2+3}]   = {t.flatten()[23].item()}    <- same number. The formula IS the tensor.")

tt = t.transpose(1, 2)
print(f"\nafter .transpose(1, 2):")
print(f"  shape           {tuple(tt.shape)}")
print(f"  stride()        {tt.stride()}      <- the strides SWAPPED. Not descending any more.")
print(f"  is_contiguous() {tt.is_contiguous()}")
print(f"  data_ptr same?  {tt.data_ptr() == t.data_ptr()}          <- no copy happened at all")

print()
try:
    tt.view(2, 12)
except RuntimeError as e:
    print(f"  tt.view(2, 12)                -> RuntimeError: {str(e)[:52]}...")
print(f"  tt.contiguous().view(2, 12)   -> OK  {tuple(tt.contiguous().view(2,12).shape)}")
print(f"  tt.reshape(2, 12)             -> OK  {tuple(tt.reshape(2,12).shape)}   (copy hidden)")
print(f"\n  tt.contiguous().data_ptr() != t.data_ptr()  ->  "
      f"{tt.contiguous().data_ptr() != t.data_ptr()}   <- contiguous() DID copy")

print("\n--- permute is transpose for >2 axes at once ---")
print(f"  t.transpose(0,2).shape    : {tuple(t.transpose(0,2).shape)}   (swap exactly two)")
print(f"  t.permute(2, 0, 1).shape  : {tuple(t.permute(2,0,1).shape)}   (reorder them all)")
print(f"  t.permute(0,2,1) == t.transpose(1,2)? "
      f"{torch.equal(t.permute(0,2,1), t.transpose(1,2))}   <- same thing for 2 axes")

t.shape           (2, 3, 4)
t.stride()        (12, 4, 1)
  read it: 'one step along axis 0 = jump 12 elements (a whole 3x4 slab)'
           'one step along axis 1 = jump 4  elements (a row)'
           'one step along axis 2 = jump 1  element'
t.is_contiguous() True   <- strides descend naturally: 12 > 4 > 1

element (1,2,3) is at flat offset 12*1 + 4*2 + 3 = 23
  t[1,2,3]          = 23
  t.flatten()[23]   = 23    <- same number. The formula IS the tensor.

after .transpose(1, 2):
  shape           (2, 4, 3)
  stride()        (12, 1, 4)      <- the strides SWAPPED. Not descending any more.
  is_contiguous() False
  data_ptr same?  True          <- no copy happened at all

  tt.view(2, 12)                -> RuntimeError: view size is not compatible with input tensor's size...
  tt.contiguous().view(2, 12)   -> OK  (2, 12)
  tt.reshape(2, 12)             -> OK  (2, 12)   (copy hidden)

  tt.contiguous().data_ptr() != t.data_ptr()  ->  True   <- contiguous() DID copy

--- permute is tran

## 4.4 THE bug: `.view(B, T, nh, hs).transpose(1, 2)` vs `.view(B, nh, T, hs)`

Both produce shape `(B, nh, T, hs)`. **They contain completely different numbers.** One is multi-head attention. The other is nonsense that trains to a mediocre loss and never raises an exception.

Here's why.

Take one token's `C=4` vector, `[a0, a1, a2, a3]`, with `nh=2, hs=2`. We want head 0 to get `[a0, a1]` and head 1 to get `[a2, a3]` — **split the channel axis**.

Now the key fact: **`view` never reorders memory.** It only re-cuts the flat buffer. So what it does depends entirely on *where you put the new axes*.

```
Memory layout of a (B=1, T=2, C=4) tensor, flat:
   [ t0c0, t0c1, t0c2, t0c3,  t1c0, t1c1, t1c2, t1c3 ]

RIGHT:  .view(B, T, nh, hs)  =  .view(1, 2, 2, 2)
   The T axis is untouched; only the LAST axis (C) gets cut into (nh, hs).
   [ [ [t0c0, t0c1], [t0c2, t0c3] ],      <- token 0: head0=[c0,c1], head1=[c2,c3]
     [ [t1c0, t1c1], [t1c2, t1c3] ] ]     <- token 1: head0=[c0,c1], head1=[c2,c3]
   then .transpose(1, 2) swaps T and nh  ->  (B, nh, T, hs). Correct.

WRONG:  .view(B, nh, T, hs)  =  .view(1, 2, 2, 2)
   Same flat buffer, but now the SECOND axis is nh, so the cut lands somewhere else:
   [ [ [t0c0, t0c1], [t0c2, t0c3] ],      <- "head 0" = ALL of token 0
     [ [t1c0, t1c1], [t1c2, t1c3] ] ]     <- "head 1" = ALL of token 1
   You just split the SEQUENCE across heads instead of splitting the CHANNELS.
```

**The disaster:** head 0 now holds tokens `0..T/2` and head 1 holds tokens `T/2..T`. Each head attends over half the sequence. The causal mask gets applied to the wrong index space. And the shape is `(B, nh, T, hs)` either way — so no exception, no warning, just a model that quietly learns a worse function.

### The rule that prevents this forever

> **`view` may only split or merge *adjacent* axes. It cannot move an axis.**

`C -> (nh, hs)` splits the last axis in place. Legal, free.

Getting `nh` into position 1 requires **moving** an axis — and only `transpose`/`permute` move axes. Hence the two-step: `view` to split, `transpose` to move.

**Any time you see a `view` that appears to reposition an axis, it's a bug.**

And symmetrically on the way out:
1. `.transpose(1, 2)` — move `nh` back next to `hs`
2. `.contiguous()` — the transpose broke the layout
3. `.view(B, T, C)` — merge the now-adjacent `(nh, hs)` back into `C`

That order is forced. Not stylistic.

In [ ]:
# Build a tensor where every element announces its (token, channel).
B_, T_, C_, nh_ = 1, 4, 4, 2
hs_ = C_ // nh_
x = torch.arange(B_ * T_ * C_).float().reshape(B_, T_, C_)
print("x  (B=1, T=4, C=4):")
print(x[0])
print("  row t = token t, column c = channel c. So x[0,2,1] = channel 1 of token 2 = 9.\n")

right = x.view(B_, T_, nh_, hs_).transpose(1, 2)   # (B, nh, T, hs)   CORRECT
wrong = x.view(B_, nh_, T_, hs_)                   # (B, nh, T, hs)   WRONG

print(f"right.shape = {tuple(right.shape)}    wrong.shape = {tuple(wrong.shape)}")
print("Identical shapes. No error raised. Now look at the CONTENTS.\n")

print("RIGHT -- head 0 should be channels 0:2 of ALL 4 tokens:")
print(right[0, 0])
print("  -> [0,1] / [4,5] / [8,9] / [12,13]")
print("  -> channels 0:2 of tokens 0, 1, 2, 3.  Correct: the head sees every token.\n")

print("WRONG -- 'head 0' turns out to be ALL channels of tokens 0:2:")
print(wrong[0, 0])
print("  -> [0,1] / [2,3] / [4,5] / [6,7]")
print("  -> token 0's channels 0:2 AND 2:4, then token 1's. This 'head' sees only")
print("     2 of the 4 tokens, and half its numbers belong to the other head.\n")

assert not torch.equal(right, wrong)
print("[!] Same shape. Different data. No exception is EVER raised.")
print("    RULE: view() may only split or merge ADJACENT axes. Only transpose moves axes.")

x  (B=1, T=4, C=4):
tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.],
        [12., 13., 14., 15.]])
  row t = token t, column c = channel c. So x[0,2,1] = channel 1 of token 2 = 9.

right.shape = (1, 2, 4, 2)    wrong.shape = (1, 2, 4, 2)
Identical shapes. No error raised. Now look at the CONTENTS.

RIGHT -- head 0 should be channels 0:2 of ALL 4 tokens:
tensor([[ 0.,  1.],
        [ 4.,  5.],
        [ 8.,  9.],
        [12., 13.]])
  -> [0,1] / [4,5] / [8,9] / [12,13]
  -> channels 0:2 of tokens 0, 1, 2, 3.  Correct: the head sees every token.

WRONG -- 'head 0' turns out to be ALL channels of tokens 0:2:
tensor([[0., 1.],
        [2., 3.],
        [4., 5.],
        [6., 7.]])
  -> [0,1] / [2,3] / [4,5] / [6,7]
  -> token 0's channels 0:2 AND 2:4, then token 1's. This 'head' sees only
     2 of the 4 tokens, and half its numbers belong to the other head.

[!] Same shape. Different data. No exception is EVER raised.
    RULE: view() may only sp

In [ ]:
# The reverse trip: (B, nh, T, hs) -> (B, T, C). Order matters just as much.
# NOTE: in the real module, y comes out of `att @ v` -- a FRESH, contiguous tensor.
# `right` above is a transposed VIEW, so transposing it back would silently restore
# the original layout and hide the very trap we want to show. .contiguous() here
# makes `y` behave exactly like the real att @ v output does.
y = right.contiguous()                                     # (B, nh, T, hs), fresh
sh("y (B, nh, T, hs)", y)
print(f"  is_contiguous(): {y.is_contiguous()}   <- as att @ v would hand it to us")

y_t = y.transpose(1, 2)                                    # (B, T, nh, hs)
sh("\nstep 1: transpose(1,2)", y_t)
print(f"  is_contiguous(): {y_t.is_contiguous()}   <- the transpose BROKE the layout")

try:
    y_t.view(B_, T_, C_)
except RuntimeError as e:
    print(f"\n  step 3 attempted early: y_t.view(B,T,C)")
    print(f"    -> RuntimeError: {str(e)[:50]}...")
    print(f"    PyTorch is protecting you. It cannot merge these axes without a copy.")

y_c = y_t.contiguous()
print(f"\nstep 2: contiguous()   is_contiguous(): {y_c.is_contiguous()}   <- copied, layout fixed")
y_final = y_c.view(B_, T_, C_)                             # (B, T, C)
sh("step 3: view(B, T, C)", y_final)

assert torch.equal(y_final, x), "round trip must be the identity"
print("\n[OK] x -> split heads -> merge heads -> x. Exact round trip.")
print("     The order is FORCED, not stylistic:")
print("       transpose (move the axis) -> contiguous (fix layout) -> view (merge)")

y (B, nh, T, hs)             (1, 2, 4, 2)
  is_contiguous(): True   <- as att @ v would hand it to us

step 1: transpose(1,2)      (1, 4, 2, 2)
  is_contiguous(): False   <- the transpose BROKE the layout

  step 3 attempted early: y_t.view(B,T,C)
    -> RuntimeError: view size is not compatible with input tensor's si...
    PyTorch is protecting you. It cannot merge these axes without a copy.

step 2: contiguous()   is_contiguous(): True   <- copied, layout fixed
step 3: view(B, T, C)        (1, 4, 4)

[OK] x -> split heads -> merge heads -> x. Exact round trip.
     The order is FORCED, not stylistic:
       transpose (move the axis) -> contiguous (fix layout) -> view (merge)


## 4.5 The batched matmul

```python
att = q @ k.transpose(-2, -1)     # (B, nh, T, hs) @ (B, nh, hs, T) -> (B, nh, T, T)
```

### What `@` does with more than 2 axes

`torch.matmul` (`@`) has a rule for high-dimensional tensors:

> **The last two axes are "the matrix". Every axis before them is "batch".**

So it takes `B*nh` independent `(T, hs) @ (hs, T)` matrix multiplications and runs them all in parallel.

```
(B, nh, T, hs) @ (B, nh, hs, T)
 ^^^^^^          ^^^^^^            batch axes: must match (or broadcast)
        ^^^^^^^         ^^^^^^     matrix axes: inner dims must agree (hs == hs)
->  (B, nh, T, T)
```

The batch axes are along for the ride. Slice `b=1, h=2` out of the result and you get exactly `q[1,2] @ k[1,2].T` — a plain 2-D matmul. Nothing is mixed across batch or head. Ever.

### Why `transpose(-2, -1)` and not `transpose(2, 3)`

Negative indices count from the right. `-2, -1` means "the last two axes", **regardless of how many batch axes there are.**

So the same line works on `(T, hs)`, on `(nh, T, hs)`, and on `(B, nh, T, hs)`. Writing `transpose(2, 3)` hardcodes the rank and breaks the moment you add or drop a batch axis. **Prefer negative indices for the matrix axes** — it's what makes attention code rank-agnostic.

### Batch-axis broadcasting is where GQA comes from

The batch axes broadcast, following the same rules as Part 2.9:

```
(B, nh, T, hs) @ (1, nh, hs, T)  -> (B, nh, T, T)     the same K for every batch row
(B, nh, T, hs) @ (B, 1,  hs, T)  -> (B, nh, T, T)     ONE K head shared by ALL query heads
```

That second line is **Multi-Query Attention** (MQA). The `(B, n_kv, ...)` generalisation — where `n_kv` query heads share each KV head — is **Grouped-Query Attention** (GQA).

When you read that GQA "shares KV heads across query heads", *this line is the entire mechanism*. There's no clever algorithm. You just let the head axis broadcast. The KV cache shrinks by the group factor, which is the whole point (Part 10.3 has the numbers).

### The memory bill

`att` is `(B, nh, T, T)`. Concretely, at `B=64, nh=6, T=256`, in bf16 (2 bytes):

```
64 * 6 * 256 * 256 * 2 bytes = 50.3 MB      per attention matrix, per layer
```

And autograd must **keep the post-softmax `att` alive for the backward pass** — the gradient of `att @ v` with respect to `v` needs `att`. So `L=6` layers hold ~300 MB, and the naive implementation materialises both the pre- and post-softmax versions.

Now push `T` to 2048, everything else fixed:

```
64 * 6 * 2048 * 2048 * 2 bytes = 3.2 GB     per layer.  x6 layers = 19 GB.
```

**That is the whole story of long context.** The FLOPs grow 64x too — but the FLOPs were never the problem. The `(B,nh,T,T)` *materialisation* is.

FlashAttention's contribution is not a faster matmul. It is **never writing this tensor to HBM at all**. It tiles the computation, keeps blocks in fast on-chip SRAM, and uses an online-softmax trick to accumulate the output without ever forming a full row of `att` in main memory. Memory drops from `O(T²)` to `O(T)` — and it's *faster* despite recomputing during the backward pass, because attention is bound by memory bandwidth, not by arithmetic.

In [ ]:
B_, nh_, T_, hs_ = 2, 3, 5, 4
q = torch.randn(B_, nh_, T_, hs_)
k = torch.randn(B_, nh_, T_, hs_)

sh("q", q)
sh("k.transpose(-2, -1)", k.transpose(-2, -1))
att = q @ k.transpose(-2, -1)
sh("q @ k^T", att)
print(f"\n{B_} x {nh_} = {B_*nh_} independent ({T_},{hs_}) @ ({hs_},{T_}) matmuls, batched.")
print("The last two axes are 'the matrix'. Everything before them is 'batch'.")

# It really is per-(b, h) independent -- check one slice against a plain 2-D matmul
manual = q[1, 2] @ k[1, 2].T                     # (T, hs) @ (hs, T) -> (T, T)
assert torch.allclose(att[1, 2], manual, atol=1e-5)
print(f"\n[OK] att[1,2] == q[1,2] @ k[1,2].T  -- exactly a plain 2-D matmul.")
print("     Nothing is ever mixed across batch or head.")

print("\n--- why transpose(-2,-1) and not transpose(2,3) ---")
for shape in [(5, 4), (3, 5, 4), (2, 3, 5, 4)]:
    tt = torch.randn(*shape)
    print(f"  {str(shape):<14} .transpose(-2,-1) -> {tuple(tt.transpose(-2,-1).shape)}   works")
try:
    torch.randn(5, 4).transpose(2, 3)
except IndexError as e:
    print(f"  (5, 4)         .transpose(2, 3)   -> IndexError   <- hardcodes the rank")

# --- MQA / GQA fall straight out of batch-axis broadcasting ---
print("\n--- broadcasting the head axis = MQA/GQA ---")
k_mqa = torch.randn(B_, 1, T_, hs_)              # ONE kv head
att_mqa = q @ k_mqa.transpose(-2, -1)            # (B,nh,T,hs) @ (B,1,hs,T)
sh("q @ k_mqa^T", att_mqa)
print(f"  {nh_} query heads, 1 KV head -> the KV cache is {nh_}x smaller.")
print("  That is MQA. The mechanism is: let axis 1 broadcast. That's all it is.")

NameError: name 'torch' is not defined

In [ ]:
def att_memory_mb(B, nh, T, bytes_per=2):
    return B * nh * T * T * bytes_per / 1e6

print("Memory for ONE (B, nh, T, T) attention matrix in bf16:\n")
print(f"{'B':>4} {'nh':>4} {'T':>6} {'per layer':>12} {'x6 layers':>12}")
print("-" * 42)
for B, nh, T in [(64, 6, 256), (64, 6, 512), (64, 6, 1024), (64, 6, 2048), (8, 12, 8192)]:
    m = att_memory_mb(B, nh, T)
    print(f"{B:>4} {nh:>4} {T:>6} {m:>10.1f} MB {m*6/1000:>10.2f} GB")

print("\nT: 256 -> 2048 is 8x the tokens and 64x the attention memory (it goes as T^2).")
print("The FLOPs also grow 64x -- but FLOPs were never the bottleneck.")
print("Writing and re-reading this tensor from HBM is. Hence FlashAttention:")
print("tile it, keep tiles in on-chip SRAM, use online softmax, and NEVER")
print("materialise (B,nh,T,T) in main memory at all.")

## 4.6 The mask: `tril`, `masked_fill`, `-inf`, `register_buffer`

### Why `-inf` and not `0`

Mask **before** the softmax, using `-inf`:

```
softmax([2.0, -inf, -inf])  ->  [1.000, 0.000, 0.000]   exp(-inf) = 0. Exactly zero mass.
softmax([2.0,  0.0,  0.0])  ->  [0.787, 0.107, 0.107]   21% of the attention on the FUTURE.
```

The reason is simple once you see it: softmax starts with `exp`. And `exp(0) = 1`, not `0`. **`0` is not a neutral score — it's a perfectly ordinary logit**, and a fairly strong one. Only `-inf` maps to zero probability, because `exp(-inf) = 0`.

Zeroing *after* the softmax is also wrong: the rows would no longer sum to 1, and you'd have to renormalise — which is just softmax again, more slowly.

### `torch.tril`

"tril" = **tri**angular, **l**ower. It keeps the lower triangle and zeros the upper.

```
tril(ones(4,4)) =
[[1, 0, 0, 0],      row 0 -> may see column 0
 [1, 1, 0, 0],      row 1 -> may see columns 0, 1
 [1, 1, 1, 0],      row 2 -> may see columns 0, 1, 2
 [1, 1, 1, 1]]      row 3 -> may see everything
```

Row = query position. Column = key position. `1` = allowed. That's exactly the causal structure: position `i` may look at position `j` only if `j <= i`.

### `masked_fill`

`tensor.masked_fill(bool_mask, value)` sets entries to `value` **wherever the mask is `True`**. The mask broadcasts.

Watch the polarity: our `tril` uses `1` = *keep*. So we must fill where it's `0`, hence `masked_fill(mask == 0, -inf)`.

Get this backwards and you mask the **past** while leaving the **future** visible. You'll spot it instantly — the loss falls off a cliff and the samples are gibberish — but it's worth knowing the failure mode.

### `register_buffer`: state that isn't a parameter

The mask must live on the GPU, next to the tensor it masks. But it must never be trained. Three options; one is right.

| | moves with `.to(device)` | in `state_dict()` | gets gradients |
|---|---|---|---|
| plain `self.mask = tensor` | **no** — silent CPU/CUDA crash | no | no |
| `nn.Parameter(...)` | yes | yes | **yes** — the optimizer will "train" your mask |
| `register_buffer(...)` | **yes** | **yes** | **no** |

`register_buffer` is the exact tool for this: persistent, device-tracked, non-trainable state. It's the same mechanism BatchNorm uses for `running_mean`. Pass `persistent=False` if you'd rather not ship it in the checkpoint (it's recomputable).

### The `[:, :, :T, :T]` slice

We allocate the mask **once** at `block_size` (the maximum) and **slice it** to whatever `T` the current batch has.

Slicing a tensor is a free view — no allocation, no kernel, no copy. This is what lets one model handle `T=1` at decode time and `T=256` at training time with identical code.

In [ ]:
print("torch.tril(torch.ones(4, 4)):")
print(torch.tril(torch.ones(4, 4)))
print("  row = query position, col = key position, 1 = allowed")
print("  row i has 1s in columns 0..i. That IS causality.\n")

scores = torch.tensor([[2.0, 1.0, 3.0],
                       [0.5, 2.0, 1.0],
                       [1.0, 1.0, 2.0]])
m = torch.tril(torch.ones(3, 3))

print("raw scores:\n", scores)
masked = scores.masked_fill(m == 0, float('-inf'))
print("\nafter masked_fill(m == 0, -inf):   <- fill where the mask is 0, i.e. NOT allowed")
print(masked)
print("\nafter softmax(dim=-1):")
print(F.softmax(masked, dim=-1))
print("\n[OK] upper triangle is EXACTLY 0.0, and every row sums to 1")

print("\n--- why not fill with 0? ---")
bad = scores.masked_fill(m == 0, 0.0)
p_bad = F.softmax(bad, dim=-1)
print(f"softmax of the 0-filled row 0: {[round(v, 4) for v in p_bad[0].tolist()]}")
print(f"  -> {p_bad[0, 1:].sum().item()*100:.0f}% of token 0's attention is on the FUTURE.")
print("     Because softmax starts with exp, and exp(0) = 1, not 0.")
print("     '0' is not a neutral score. It is an ordinary, fairly strong logit.")
print("     Only exp(-inf) = 0 gives exactly zero mass.")

In [ ]:
# --- register_buffer vs Parameter vs plain attribute ---
class Demo(nn.Module):
    def __init__(self):
        super().__init__()
        self.plain = torch.tril(torch.ones(3, 3))                    # naive
        self.param = nn.Parameter(torch.tril(torch.ones(3, 3)))      # wrong: trainable mask
        self.register_buffer("buf", torch.tril(torch.ones(3, 3)))    # right

d = Demo()
print(f"state_dict keys      : {list(d.state_dict().keys())}")
print(f"  -> 'plain' is ABSENT. It will not be saved or restored.")
print(f"trainable parameters : {[n for n, _ in d.named_parameters()]}")
print(f"  -> 'param' WOULD be updated by the optimizer. A mask has no gradient to follow.")
print(f"buffers              : {[n for n, _ in d.named_buffers()]}")
print(f"  -> 'buf' is saved, moves with .to(device), and is never trained. Correct.")

if torch.cuda.is_available():
    d = d.cuda()
    print(f"\nafter .cuda():")
    print(f"  plain.device : {d.plain.device}   <- STAYED ON CPU. RuntimeError at forward.")
    print(f"  buf.device   : {d.buf.device}")
else:
    print("\n(on CUDA, .plain would silently stay on the CPU and blow up at forward time)")

# --- the free slice ---
block_size = 8
bias = torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size)
sh("\nfull mask buffer", bias)
print("  the leading (1, 1) broadcast over B and nh -- one mask serves every head\n")
for T in (1, 3, 8):
    v = bias[:, :, :T, :T]
    print(f"  bias[:, :, :{T}, :{T}] -> {str(tuple(v.shape)):<14} "
          f"same storage: {v.data_ptr() == bias.data_ptr()}")
print("\nAllocate once at block_size; slice for free at every T.")
print("Slicing is a view: no allocation, no kernel, no copy.")
print("T=1 at decode, T=256 at training -- identical code path.")

## 4.7 The scale factor `1/sqrt(hs)` — derived

I previously asserted the softmax Jacobian here without deriving it. Let's fix that, because the derivation is short and it's the actual reason the scale factor exists.

### Part 1: why the scores blow up

Suppose `q` and `k` entries are roughly independent with mean 0 and variance 1 — which is what initialisation is designed to achieve. Look at their dot product:

```
score = sum_{i=1..hs} q_i * k_i
```

**Expected value:** each term `q_i * k_i` has mean `E[q_i]*E[k_i] = 0` (independence), so `E[score] = 0`.

**Variance:** for independent zero-mean variables, `Var[q_i * k_i] = Var[q_i] * Var[k_i] = 1 * 1 = 1`. And variances of independent terms **add**:

```
Var[score] = sum of hs terms, each with variance 1 = hs
std[score] = sqrt(hs)
```

At `hs = 64`, that's `sqrt(64) = 8`. So a typical row of raw scores spans roughly `[-24, +24]`.

Dividing by `sqrt(hs)` divides the standard deviation by `sqrt(hs)`, taking it back to exactly `1`.

### Part 2: why big scores are fatal — the Jacobian, derived

Softmax over a row of `n` scores:

```
p_i = exp(s_i) / S           where  S = sum_j exp(s_j)
```

We want `dp_i / ds_k`. Two cases.

**Case `i == k`** — use the quotient rule. `d(exp(s_i))/ds_i = exp(s_i)`, and `dS/ds_i = exp(s_i)`:

```
dp_i     exp(s_i) * S  -  exp(s_i) * exp(s_i)
----  =  -----------------------------------
ds_i                   S^2

         exp(s_i)     exp(s_i)   exp(s_i)
      =  --------  -  -------- * --------
             S            S          S

      =  p_i  -  p_i^2   =   p_i (1 - p_i)
```

**Case `i != k`** — the numerator `exp(s_i)` doesn't depend on `s_k`, so only `S` does. `dS/ds_k = exp(s_k)`:

```
dp_i        0 * S  -  exp(s_i) * exp(s_k)
----  =  -----------------------------------  =  - p_i * p_k
ds_k                   S^2
```

**Combine both cases** using the Kronecker delta (`d_ik = 1` if `i==k`, else `0`):

```
dp_i / ds_k  =  p_i (d_ik - p_k)
```

Which in matrix form is exactly:

```
J  =  diag(p) - p p^T
```

That's where the formula comes from. Not magic — just the quotient rule, twice.

### Part 3: now watch it die

Suppose softmax saturates — one entry `p_m ≈ 1` and all the rest `≈ 0`. Plug into `J = diag(p) - p p^T`:

- **Diagonal entry `m`:** `p_m(1 - p_m) = 1 * (1 - 1) = 0`
- **Every other diagonal entry `i`:** `p_i(1 - p_i) = 0 * (1 - 0) = 0`
- **Every off-diagonal:** `-p_i * p_k`, and at least one of them is 0, so `= 0`

**Every single entry of the Jacobian is zero. A saturated softmax has no gradient at all.**

So this isn't about the forward pass looking ugly. It's that attention would **freeze at initialisation** and never learn what to attend to. And — here's the sting — it would freeze *harder* the larger `hs` gets, since `std = sqrt(hs)`. That's precisely backwards from what you want as you scale up.

Hence `1/sqrt(hs)`.

### One detail: it's `sqrt(hs)`, not `sqrt(C)`

`hs = C // nh = 64`, not `C = 384`. The dot product is taken **within a head**, over `hs` terms — that's what the variance sum was over. Using `sqrt(C)` would over-shrink by a factor of `sqrt(nh)`.

Worth noticing: `hs = 64` is standard almost everywhere (GPT-2, GPT-3, LLaMA). Scaling up means **more heads, not bigger ones** — partly so this variance argument stays exactly where it is.

In [ ]:
# --- Part 1: the variance argument, measured ---
hs = 64
q = torch.randn(10000, hs)
k = torch.randn(10000, hs)
raw = (q * k).sum(-1)

print(f"hs = {hs},  q and k entries ~ N(0, 1)\n")
print(f"  raw dot product std      : {raw.std():.3f}")
print(f"  theory sqrt(hs)          : {math.sqrt(hs):.3f}   <- variances of hs independent terms ADD")
print(f"  after / sqrt(hs), std    : {(raw/math.sqrt(hs)).std():.3f}")
print(f"  theory                   : 1.000")
print(f"\n  so raw scores span roughly [{-3*raw.std():.0f}, {3*raw.std():.0f}]. Softmax of THAT is a hard argmax.")

In [ ]:
# --- Part 2: verify the derived Jacobian against autograd ---
s = torch.tensor([1.0, 2.0, 3.0, 4.0])
p = F.softmax(s, dim=-1)

J_auto = torch.autograd.functional.jacobian(lambda z: F.softmax(z, -1), s)
J_formula = torch.diag(p) - p.outer(p)                # our derivation

print("scores s :", s.tolist())
print("softmax p:", [round(v, 4) for v in p.tolist()])
print("\nautograd's Jacobian dp_i/ds_k:")
print(J_auto)
print("\nour derived formula  diag(p) - p p^T:")
print(J_formula)
assert torch.allclose(J_auto, J_formula, atol=1e-6)
print("\n[OK] the derivation is correct -- it matches autograd exactly.")
print(f"\nspot-check the diagonal: J[0][0] should be p_0(1-p_0) = "
      f"{p[0].item():.4f} * (1 - {p[0].item():.4f}) = {(p[0]*(1-p[0])).item():.4f}")
print(f"                         autograd says {J_auto[0][0].item():.4f}")
print(f"spot-check off-diagonal: J[0][1] should be -p_0*p_1 = "
      f"-{p[0].item():.4f} * {p[1].item():.4f} = {(-p[0]*p[1]).item():.4f}")
print(f"                         autograd says {J_auto[0][1].item():.4f}")

In [ ]:
# --- Part 3: watch the gradient die as the softmax saturates ---
base = torch.tensor([1.0, 2.0, 3.0, 4.0])
print(f"{'scale':>6}  {'std':>6}  {'softmax':<40} {'max |J|':>10}  gradient?")
print("-" * 84)
for scale in [1, 2, 4, 8, 16]:
    s = base * scale
    p = F.softmax(s, dim=-1)
    J = torch.diag(p) - p.outer(p)
    verdict = "healthy" if J.abs().max() > 0.05 else ("weak" if J.abs().max() > 1e-3 else "DEAD")
    print(f"{scale:>6}  {s.std():>6.2f}  {str([round(v,4) for v in p.tolist()]):<40} "
          f"{J.abs().max():>10.2e}  {verdict}")

print("\nAt scale 8 (which is exactly std = sqrt(64), the UNSCALED case at hs=64),")
print("the Jacobian is ~1e-4 everywhere. Attention cannot learn what to attend to.")
print("\nAnd it gets WORSE as hs grows, since std = sqrt(hs). That is exactly backwards")
print("for scaling up -- which is why the 1/sqrt(hs) is not optional.")

## 4.8 The module

Everything above, assembled. Two paths:

- **`flash=False`** — the explicit version. Every intermediate is a named tensor you can print. **This is the one to read.**
- **`flash=True`** — `F.scaled_dot_product_attention`, which since PyTorch 2.0 dispatches to a fused FlashAttention-2 kernel on A100. It does the scale, mask, softmax, dropout and `@v` in one kernel, and **never materialises `(B, nh, T, T)` in HBM**.

They are mathematically identical (up to floating-point reassociation). Use `flash=False` to learn, `flash=True` to train.

`is_causal=True` tells SDPA to apply the causal mask internally — no mask tensor is allocated at all, the kernel just skips the upper-triangular tiles. Which, incidentally, is close to a **2x FLOP saving** that the naive version never gets: the naive path computes the whole `(T, T)` matrix and then throws half of it away.

In [ ]:
class CausalSelfAttention(nn.Module):

    def __init__(self, config, flash=True):
        super().__init__()
        assert config.n_embd % config.n_head == 0, "C must be divisible by nh"
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.dropout = config.dropout

        # ONE fused projection for Q, K, V.  weight is (3C, C).
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

        self.flash = flash and hasattr(F, 'scaled_dot_product_attention')
        if not self.flash:
            # (1, 1, block, block) -- the leading 1s broadcast over B and nh
            self.register_buffer("bias", torch.tril(
                torch.ones(config.block_size, config.block_size)
            ).view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()
        nh, hs = self.n_head, C // self.n_head

        qkv = self.c_attn(x)                                   # (B, T, 3C)
        q, k, v = qkv.split(self.n_embd, dim=2)                # 3 x (B, T, C)

        # split the CHANNELS into heads, then MOVE the head axis next to the batch axis
        q = q.view(B, T, nh, hs).transpose(1, 2)               # (B, nh, T, hs)
        k = k.view(B, T, nh, hs).transpose(1, 2)               # (B, nh, T, hs)
        v = v.view(B, T, nh, hs).transpose(1, 2)               # (B, nh, T, hs)

        if self.flash:
            # fused: never materialises (B, nh, T, T) in HBM
            y = F.scaled_dot_product_attention(
                q, k, v, attn_mask=None,
                dropout_p=self.dropout if self.training else 0.0,
                is_causal=True)                                # (B, nh, T, hs)
        else:
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(hs))   # (B, nh, T, T)
            att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)                       # (B, nh, T, T)
            att = self.attn_dropout(att)
            y = att @ v                                        # (B, nh, T, hs)

        # move the head axis back, fix the layout, merge (nh, hs) -> C
        y = y.transpose(1, 2).contiguous().view(B, T, C)       # (B, T, C)
        return self.resid_dropout(self.c_proj(y))              # (B, T, C)


# Do the two paths agree?
cfg = GPTConfig(block_size=32, n_head=4, n_embd=64, dropout=0.0, bias=False)
a_slow = CausalSelfAttention(cfg, flash=False)
a_fast = CausalSelfAttention(cfg, flash=True)
a_fast.load_state_dict(a_slow.state_dict(), strict=False)
a_slow.eval(); a_fast.eval()

x = torch.randn(2, 16, 64)
with torch.no_grad():
    y1, y2 = a_slow(x), a_fast(x)
sh("input ", x)
sh("output", y1)
print(f"\nmax |manual - SDPA| = {(y1 - y2).abs().max().item():.2e}")
assert torch.allclose(y1, y2, atol=1e-5)
print("[OK] the explicit implementation and FlashAttention agree to float tolerance")
print(f"\nparams: c_attn {tuple(a_slow.c_attn.weight.shape)} = 3*C*C,  "
      f"c_proj {tuple(a_slow.c_proj.weight.shape)} = C*C   ->  4*C^2 total")

In [ ]:
# Watch every intermediate, one line at a time. This is the cell to come back to.
cfg = GPTConfig(block_size=256, n_head=6, n_embd=384, dropout=0.0, bias=False)
attn = CausalSelfAttention(cfg, flash=False)
B, T, C = 8, 256, 384
nh, hs = cfg.n_head, cfg.n_embd // cfg.n_head
x = torch.randn(B, T, C)

print(f"B={B}  T={T}  C={C}  nh={nh}  hs={hs}\n")
sh("x", x)
qkv = attn.c_attn(x);                                  sh("  c_attn(x)          -> qkv", qkv)
q, k, v = qkv.split(C, dim=2);                         sh("  split(C, dim=2)    -> q", q)
q4 = q.view(B, T, nh, hs);                             sh("  view(B,T,nh,hs)    -> q", q4)
q4 = q4.transpose(1, 2);                               sh("  transpose(1,2)     -> q", q4)
k4 = k.view(B, T, nh, hs).transpose(1, 2)
v4 = v.view(B, T, nh, hs).transpose(1, 2)
att = q4 @ k4.transpose(-2, -1) / math.sqrt(hs);       sh("  q @ k^T / sqrt(hs) -> att", att)
att = att.masked_fill(attn.bias[:, :, :T, :T] == 0, float('-inf'))
att = F.softmax(att, dim=-1);                          sh("  softmax(dim=-1)    -> att", att)
y = att @ v4;                                          sh("  att @ v            -> y", y)
y = y.transpose(1, 2).contiguous().view(B, T, C);      sh("  merge heads        -> y", y)
out = attn.c_proj(y);                                  sh("  c_proj(y)          -> out", out)

print(f"\nx.shape == out.shape : {x.shape == out.shape}   <- shape-preserving, always")
print(f"att elements         : {att.numel():,} = B*nh*T*T   ({att.numel()*2/1e6:.1f} MB in bf16)")
print(f"x elements           : {x.numel():,} = B*T*C")
print(f"ratio                : {att.numel()/x.numel():.0f}x = nh*T/C, and it grows LINEARLY with T")

---
# Part 5 — MLP: where the parameters actually are

## 5.1 The shape trace

```
x                        (B, T, C)
 |  c_fc = Linear(C, 4C)
 v
h                        (B, T, 4C)        <- expand to 4x width
 |  GELU  (elementwise, shape unchanged)
 v
h                        (B, T, 4C)
 |  c_proj = Linear(4C, C)
 v
out                      (B, T, C)         <- project back down
```

Expand, nonlinearity, contract. That's the whole module — two matrices and one activation.

## 5.2 It is strictly per-token — and that's the architectural point

`nn.Linear` only ever touches the **last** axis. Token `t` in row `b` is transformed with **zero reference to any other token**. There is no mixing here at all — not across the batch, not across time.

Which gives the clean division of labour that defines a transformer:

> **Attention moves information *between* positions. The MLP does the *thinking* at each position.**

They alternate, and neither does the other's job. Check it against Part 3.4's rule: every op in a GPT is either strictly per-token or explicitly masked. The MLP is per-token. Attention is masked. That's the complete list.

### `nn.Linear` on a 3-D input

A `Linear(C, 4C)` applied to `(B, T, C)` **flattens the leading axes**. It's executed as a single `(B*T, C) @ (C, 4C)` matmul, then reshaped back.

That's why batch and time are interchangeable to a `Linear`, and why **`B*T` — not `B` — is the number that determines whether you're saturating the GPU.** A batch of 64 sequences of 256 tokens gives a matmul with 16,384 rows. A single token during decode gives a matmul with **1** row. Same weights. Wildly different hardware efficiency. (Part 10.4 is entirely about this.)

## 5.3 Where the parameters live

| | weights | at `C=384` |
|---|---|---|
| attention (`c_attn` + `c_proj`) | `3C² + C² = 4C²` | 589,824 |
| MLP (`c_fc` + `c_proj`) | `4C² + 4C² = 8C²` | 1,179,648 |

**The MLP is 2/3 of every block's parameters.**

Sit with that for a second. A "12-layer attention model" is really a 12-layer MLP model with attention bolted on for communication. This ratio is fixed by the 4x expansion and holds for every GPT ever built.

It's also why:
- **MoE replaces the MLP**, not attention — that's where the parameters are.
- **Weight-only quantization targets `c_fc` and `c_proj` first** — same reason.

### Why 4x?

Empirical. It comes from "Attention Is All You Need" and it stuck through every GPT. There is nothing deep about the number 4.

Modern models use `8/3 x` with a **gated SwiGLU** — three matrices instead of two, with the width tuned so the parameter count roughly matches the old `4x`. It trades a little more compute for better quality.

## 5.4 GELU — what `Phi` actually is

I previously wrote `GELU(x) = x * Phi(x)` and left `Phi` as a symbol. Let me actually say what it is.

**`Phi(x)` is the cumulative distribution function (CDF) of the standard normal distribution.** Concretely:

> `Phi(x)` = the probability that a random draw from `N(0, 1)` lands **below** `x`.

Some values to make it concrete:
- `Phi(-inf) = 0` — nothing is below negative infinity
- `Phi(-2) ≈ 0.023` — only 2.3% of a standard normal is below -2
- `Phi(0) = 0.5` — exactly half is below 0 (the normal is symmetric)
- `Phi(2) ≈ 0.977`
- `Phi(+inf) = 1`

So `Phi` is a smooth S-curve from 0 to 1. **It looks like a sigmoid, because it is one** — just a different one from the logistic sigmoid.

### Now the formula reads

```
GELU(x) = x * Phi(x)
        = x * P(a standard normal draw is below x)
```

Compare to ReLU:

```
ReLU(x) = x * 1[x > 0]         gate is HARD: 0 or 1, nothing between
GELU(x) = x * Phi(x)           gate is SOFT: the probability that x "wins"
```

Both are "`x` times a gate". ReLU's gate is a step function — a cliff at zero. **GELU's gate is the smooth probability that a normal draw is below `x`.** Instead of asking "is `x` positive?", it asks "how confidently is `x` positive?".

### The three practical consequences

**1. Smooth everywhere.** ReLU's derivative jumps from 0 to 1 at the origin. GELU's is continuous.

**2. Nonzero gradient for negative inputs — no dead units.** A ReLU unit that outputs negative for every example in your dataset receives **exactly zero gradient, forever**. It's dead and can never recover. GELU's small negative lobe keeps a gradient flowing, so the unit can come back.

**3. Non-monotonic.** GELU dips slightly *below* zero around `x ≈ -0.75` before returning to 0. ReLU never does anything but flatten. Whether this matters is debated; it's real either way.

### The tanh approximation

GPT-2 shipped an approximation, because computing `erf` (which `Phi` needs) was slow in 2019:

```
GELU_tanh(x) = 0.5x * (1 + tanh( sqrt(2/pi) * (x + 0.044715 x^3) ))
```

`nn.GELU(approximate='tanh')` reproduces it exactly. Use it when loading real GPT-2 checkpoints — the two differ by ~5e-4, which is invisible in training but shows up immediately if you're diffing against reference logits. For training from scratch it doesn't matter at all.

In [ ]:
# --- What Phi actually is: the standard normal CDF ---
from math import erf, sqrt

def Phi(x):
    # the standard normal CDF, written out. P(Z < x) for Z ~ N(0,1)
    return 0.5 * (1.0 + erf(x / sqrt(2.0)))

print("Phi(x) = P(a draw from N(0,1) lands below x)\n")
print(f"{'x':>6}  {'Phi(x)':>8}   meaning")
for xv in [-3, -2, -1, 0, 1, 2, 3]:
    print(f"{xv:>6}  {Phi(xv):>8.4f}   {Phi(xv)*100:>5.1f}% of a standard normal is below {xv}")
print("\nIt is a smooth S-curve from 0 to 1. It IS a sigmoid -- just not the logistic one.")

# empirical check: draw a million normals and count
z = torch.randn(1_000_000)
print(f"\nempirical check with 1M draws from N(0,1):")
for xv in [-1.0, 0.0, 1.0]:
    frac = (z < xv).float().mean().item()
    print(f"  fraction below {xv:>4} : {frac:.4f}   (Phi says {Phi(xv):.4f})")

In [ ]:
# --- GELU(x) = x * Phi(x). Verify against torch. ---
xs = torch.tensor([-3., -2., -1., -0.5, 0., 0.5, 1., 2., 3.])
hand = torch.tensor([xv.item() * Phi(xv.item()) for xv in xs])   # x * Phi(x), by hand
torch_exact = F.gelu(xs)                                          # torch's erf version

print(f"{'x':>6} {'Phi(x)':>8} {'x*Phi(x)':>10} {'F.gelu(x)':>11} {'ReLU(x)':>9}")
print("-" * 50)
for i, xv in enumerate(xs.tolist()):
    print(f"{xv:>6.1f} {Phi(xv):>8.4f} {hand[i]:>10.4f} {torch_exact[i]:>11.4f} "
          f"{max(xv,0):>9.1f}")
assert torch.allclose(hand, torch_exact, atol=1e-6)
print("\n[OK] x * Phi(x) IS F.gelu(x), exactly. Nothing hidden.")
print("\nRead the table: ReLU multiplies x by a HARD gate (0 or 1).")
print("GELU multiplies x by Phi(x) -- the SOFT probability that x 'wins'.")

In [ ]:
# --- the three consequences, measured ---
xs = torch.tensor([-3., -2., -1., -0.5, 0., 0.5, 1., 2., 3.], requires_grad=True)
g = F.gelu(xs, approximate='tanh'); g.sum().backward()
g_grad = xs.grad.clone(); xs.grad = None
r = F.relu(xs); r.sum().backward()
r_grad = xs.grad.clone()

print(f"{'x':>6} {'GELU grad':>11} {'ReLU grad':>11}   note")
print("-" * 52)
for i, xv in enumerate(xs.tolist()):
    note = "<- ReLU is DEAD here. Forever." if xv < 0 else ""
    print(f"{xv:>6.1f} {g_grad[i].item():>11.4f} {r_grad[i].item():>11.4f}   {note}")

print("\n1. Smooth: GELU's gradient is continuous; ReLU's jumps 0->1 at the origin.")
print("2. No dead units: GELU keeps gradient for x<0. A ReLU unit that goes negative")
print("   for every example gets EXACTLY zero gradient and can never recover.")
fine = torch.linspace(-3, 0, 300)
print(f"3. Non-monotonic: GELU dips to {F.gelu(fine).min().item():.4f} near "
      f"x = {fine[F.gelu(fine).argmin()].item():.2f} before returning to 0.")

d = (F.gelu(torch.linspace(-5,5,1000)) -
     F.gelu(torch.linspace(-5,5,1000), approximate='tanh')).abs().max()
print(f"\nerf vs tanh GELU, max difference: {d:.2e}")
print("  invisible in training; matters only when diffing against real GPT-2 logits.")

In [ ]:
class MLP(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.c_fc   = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu   = nn.GELU(approximate='tanh')      # GPT-2 used the tanh approximation
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.c_fc(x)      # (B, T, C)  -> (B, T, 4C)   expand
        x = self.gelu(x)      # (B, T, 4C) -> (B, T, 4C)   elementwise
        x = self.c_proj(x)    # (B, T, 4C) -> (B, T, C)    contract
        return self.dropout(x)


cfg = GPTConfig(n_embd=384)
mlp = MLP(cfg)
x = torch.randn(2, 5, 384)
sh("x", x)
sh("c_fc(x)", mlp.c_fc(x))
sh("gelu(...)", mlp.gelu(mlp.c_fc(x)))
sh("c_proj(...) = out", mlp(x))

C = cfg.n_embd
attn_params, mlp_params = 4 * C * C, 8 * C * C
print(f"\nC = {C}")
print(f"  attention params : 4*C^2 = {attn_params:,}")
print(f"  MLP params       : 8*C^2 = {mlp_params:,}")
print(f"  MLP share        : {mlp_params/(mlp_params+attn_params)*100:.0f}%  <- two thirds")
print(f"\nactual: c_fc {tuple(mlp.c_fc.weight.shape)} + c_proj {tuple(mlp.c_proj.weight.shape)} "
      f"= {sum(p.numel() for p in mlp.parameters()):,}")
print("\nA '12-layer attention model' is really a 12-layer MLP model with attention")
print("bolted on for communication. That is why MoE replaces the MLP, not attention.")

In [ ]:
# nn.Linear flattens the leading axes: (B,T,C) is really a (B*T, C) matmul
lin = nn.Linear(384, 1536, bias=False)
print("Linear(384, 1536) on various shapes:\n")
for shape in [(384,), (5, 384), (2, 5, 384), (2, 3, 5, 384)]:
    out = lin(torch.randn(*shape))
    rows = int(np.prod(shape[:-1]))
    print(f"  {str(shape):<18} -> {str(tuple(out.shape)):<18} "
          f"executed as ({rows:>3}, 384) @ (384, 1536)")
print("\nOnly the LAST axis is touched. B and T are interchangeable to a Linear.")
print("What decides GPU efficiency is B*T -- the number of matmul ROWS.")
print("  training: B*T = 64*256 = 16,384 rows  -> the GPU is busy")
print("  decoding: B*T = 1*1    =      1 row   -> the GPU is starving (Part 10.4)")

---
# Part 6 — The Block: the residual stream

```python
x = x + self.attn(self.ln_1(x))     # (B, T, C) -> (B, T, C)
x = x + self.mlp(self.ln_2(x))      # (B, T, C) -> (B, T, C)
```

Two lines. Read them carefully, because the structure they describe is the thing actually worth understanding about transformers.

## 6.1 The residual stream

`x`, shape `(B, T, C)`, is a **bus**. It enters the first block and exits the last with the same shape. Every sublayer does three things:

1. reads a **normalised copy** of the stream — `ln_1(x)`
2. computes something
3. **adds** the result back in

Nothing ever *overwrites* the stream. Each block only makes an **additive contribution**.

That's not a stylistic choice — it has a consequence. Because every write into the stream is a `+`, the final logits are a **sum of contributions**, one per sublayer, all living in the same `C`-dimensional space. You can decompose them, ablate them one at a time, and read them off independently.

That's the foundation of mechanistic interpretability work, and it isn't an analysis convenience — it's a direct consequence of these two `+` signs.

## 6.2 Pre-LN vs post-LN — derived, then measured

I previously asserted that pre-LN trains deep and post-LN doesn't. Let's actually show why, then prove it.

```
post-LN (original 2017 Transformer):   x = LayerNorm(x + attn(x))
pre-LN  (GPT-2 and everything since):  x = x + attn(LayerNorm(x))
```

The difference is whether the LayerNorm sits **on** the residual path or **beside** it.

### The derivation

Write one post-LN block as `x_out = LN(x_in + f(x_in))`. Differentiate:

```
d(x_out)/d(x_in)  =  J_LN · (I + J_f)
                     ^^^^
                     the LayerNorm Jacobian, multiplying EVERYTHING
```

Now stack `L` of them. The chain rule multiplies these together:

```
d(x_out)/d(x_0)  =  J_LN^(L) (I + J_f^(L)) · ... · J_LN^(1) (I + J_f^(1))
```

**Every single term has a `J_LN` factor.** So the gradient reaching layer 0 is scaled by a product of `L` LayerNorm Jacobians. Those factors aren't close to `I`, and multiplying `L` of them compounds: for `L = 40`, you're multiplying 40 matrices that each shrink things a bit. The result is exponentially small.

Now pre-LN. Write `x_out = x_in + f(LN(x_in))`:

```
d(x_out)/d(x_in)  =  I  +  J_f · J_LN
                     ^
                     an EXACT identity term, with nothing multiplying it
```

Unroll the whole stack:

```
x_out = x_in + f_1(ln(x_in)) + f_2(ln(...)) + ... + f_2L(...)
        ^^^^
        d(x_out)/d(x_in) contains a clean, exact identity term
```

**The gradient always has an unobstructed `1x` route home**, no matter the depth. That's what makes 96-layer models trainable.

### The cost, and what pays for it

Pre-LN's stream is never normalised, so **its variance grows with depth** (you saw this in Part 6's demo below). Two things pay for that:

1. a **final LayerNorm** `ln_f` after the last block
2. the `1/sqrt(2L)` residual init in Part 7

## 6.3 Why `ln_1` and `ln_2` are separate

Each has its own `(C,)` gamma and beta. Attention and the MLP want to read the stream at different scales, and it's only `2 * 2 * C = 1536` parameters per block — nothing. Sharing them would save nothing and cost quality.

Note also that `x` goes into the `+` **unmodified**. The normalisation exists purely for the sublayer's benefit; the stream itself is never normalised in place.

In [ ]:
class Block(nn.Module):

    def __init__(self, config, flash=True):
        super().__init__()
        self.ln_1 = LayerNorm(config.n_embd, bias=config.bias)
        self.attn = CausalSelfAttention(config, flash=flash)
        self.ln_2 = LayerNorm(config.n_embd, bias=config.bias)
        self.mlp  = MLP(config)

    def forward(self, x):
        # pre-LN: normalise a COPY for the sublayer. The stream itself is never overwritten.
        x = x + self.attn(self.ln_1(x))    # (B, T, C)
        x = x + self.mlp(self.ln_2(x))     # (B, T, C)
        return x


cfg = GPTConfig(block_size=64, n_head=6, n_embd=384, dropout=0.0, bias=False)
blk = Block(cfg)
x = torch.randn(2, 16, 384)
sh("in ", x)
sh("out", blk(x))
print(f"\nblock params: {sum(p.numel() for p in blk.parameters()):,}")
print(f"  theory 12*C^2 = {12*384*384:,}  (+ 2 LayerNorms x {384} = {2*384})")

In [ ]:
# ============================================================================
# THE CLAIM: pre-LN keeps an identity path home; post-LN multiplies by a
# LayerNorm Jacobian at every layer, and those compound. Let's MEASURE it.
# ============================================================================
C_d, L_d, B_d, T_d = 64, 40, 4, 16

class PreLNBlock(nn.Module):
    def __init__(s):
        super().__init__(); s.ln = nn.LayerNorm(C_d); s.f = nn.Linear(C_d, C_d)
    def forward(s, x): return x + s.f(s.ln(x))        # LN beside the path

class PostLNBlock(nn.Module):
    def __init__(s):
        super().__init__(); s.ln = nn.LayerNorm(C_d); s.f = nn.Linear(C_d, C_d)
    def forward(s, x): return s.ln(x + s.f(x))        # LN ON the path

print(f"{L_d} layers, C={C_d}. Push a gradient from the output back to the input.\n")
results = {}
for name, Blk in [("pre-LN ", PreLNBlock), ("post-LN", PostLNBlock)]:
    torch.manual_seed(0)
    blocks = nn.ModuleList([Blk() for _ in range(L_d)])
    x = torch.randn(B_d, T_d, C_d, requires_grad=True)
    h = x
    for b in blocks:
        h = b(h)
    h.sum().backward()
    results[name] = (x.grad.norm().item(),
                     blocks[0].f.weight.grad.norm().item(),
                     blocks[-1].f.weight.grad.norm().item())
    gin, g0, gl = results[name]
    print(f"{name}:")
    print(f"   grad norm reaching the INPUT     : {gin:.4e}")
    print(f"   grad norm at block  0's weights  : {g0:.4e}")
    print(f"   grad norm at block {L_d-1}'s weights  : {gl:.4e}")
    print()

pre_in, post_in = results["pre-LN "][0], results["post-LN"][0]
print(f"pre-LN's gradient reaches the input {pre_in/post_in:.2e}x stronger than post-LN's.")
print(f"post-LN's has vanished to {post_in:.1e} -- about {math.log10(pre_in/post_in):.0f} orders")
print(f"of magnitude of signal, gone.")
print("\nThat is the L LayerNorm Jacobians compounding, exactly as derived.")
print("Pre-LN's  d(out)/d(in) = I + J_f·J_LN  has an EXACT identity term.")
print("Post-LN's d(out)/d(in) = J_LN·(I + J_f) has J_LN multiplying EVERYTHING.")
print("\nThis is why post-LN needs careful LR warmup and gets hard past ~12-18 layers,")
print("and why every model after GPT-2 is pre-LN.")

In [ ]:
# The cost of pre-LN: nothing normalises the stream, so its variance grows with depth.
x = torch.randn(2, 16, 384)
blocks = nn.ModuleList([Block(cfg) for _ in range(6)])
print("residual stream std through 6 pre-LN blocks:")
sh("  input", x)
print(f"  {'':<22} std = {x.std().item():.3f}")
with torch.no_grad():
    for i, b in enumerate(blocks):
        x = b(x)
        print(f"  after block {i}          {tuple(x.shape)}   std = {x.std().item():.3f}")
print("\nShape: constant. Std: GROWING. Nothing normalises the stream in pre-LN --")
print("that is the price of leaving the residual path unobstructed.")
print("\nTwo things pay for it:")
print("  (a) the final ln_f after the last block")
print("  (b) the 1/sqrt(2L) residual-projection init -- Part 7 derives it")

---
# Part 7 — The full model

## 7.1 Assembly

```
idx                              (B, T)      int64
 |
 |-- wte(idx)                    (B, T, C)   token embeddings   "what am I?"
 |-- wpe(arange(T))                 (T, C)   position embeddings "where am I?"
 v      (broadcast add)
x                                (B, T, C)   <-- the residual stream starts here
 |
 |  block 0 .. block L-1         (B, T, C)   shape never changes
 v
x                                (B, T, C)
 |  ln_f                         (B, T, C)
 v
 |  lm_head = Linear(C, V)
 v
logits                           (B, T, V)   <-- the biggest tensor in the model
```

## 7.2 The logits tensor is enormous

At `B=64, T=256, V=50304`:

```
64 * 256 * 50304 = 823,000,000 floats = 3.3 GB in fp32
```

**The model's entire parameter set is 30M. The logits are 27x larger than the model.**

And `F.cross_entropy` internally computes `log_softmax`, which needs another tensor the same size. This is routinely the **peak-memory moment of the whole training step**, and it's why:

- the LM head is the first thing sharded in tensor parallelism,
- fused cross-entropy kernels (Liger, `torch.compile`'s CE fusion) give large wins,
- at inference we compute logits for **only the last position** (Part 10).

## 7.3 Weight tying, and the `nn.Linear` convention that makes it work

```python
self.transformer.wte.weight = self.lm_head.weight    # one tensor, two names
```

The shape argument, which is the part people find surprising:

- `nn.Embedding(V, C).weight` is `(V, C)` — row `i` is token `i`'s vector. Obvious.
- **`nn.Linear(in, out).weight` is `(out, in)`, not `(in, out)`.** Because Linear computes `x @ W.T + b`. So `nn.Linear(C, V).weight` is `(V, C)`.

Both are `(V, C)`. **The same shape.** So the assignment is a plain Python rebinding of two module attributes to one `nn.Parameter`, and autograd accumulates gradients from both uses into it automatically.

Semantically it says: *the vector that means "cat" on the way in is the vector you dot against to score "cat" on the way out.* Which is reasonable, and it saves `V * C = 19.3M` parameters — **64% of this model.** Our GPT is 30M tied and would be 49M untied.

*(If you're porting real GPT-2 weights: the original OpenAI checkpoints store `c_attn`/`c_fc`/`c_proj` **transposed**, because TensorFlow's `Conv1D` used `(in, out)`. Every port has a `transposed = ['attn.c_attn.weight', ...]` list to flip them. That's this convention biting.)*

## 7.4 `vocab_size = 50304`, not `50257`

`50257` is an awkward number. `50304 = 50257` rounded up to the next multiple of 64. The extra 47 rows are dead tokens the tokenizer never emits — they train to a large negative logit and are never sampled.

Why bother: **GEMM kernels tile along output columns**, usually in multiples of 8/16/64 depending on dtype. A `50257`-wide output leaves a **ragged tail tile** that runs at a fraction of peak and, on tensor cores, can prevent the fast path entirely.

Padding to 50304 adds **0.09% more FLOPs** and Karpathy measured **~25% faster steps** in nanoGPT from this one line. Best FLOPs-per-line-of-code trade in the file, and it generalises: **keep every matmul dimension a multiple of 64** (128 for fp8).

## 7.5 Initialisation — the `1/sqrt(2L)`, walked

**`normal(0, 0.02)` everywhere.** The GPT-2 paper's choice. Note `1/sqrt(384) = 0.051` and `1/sqrt(1536) = 0.026`, so `0.02` is in the right neighbourhood for these widths. It's not principled at every scale — it's what the paper did and what everyone copies.

**Residual projections scaled by `1/sqrt(2L)`.** *This* one is principled and it matters. Let's walk it.

Each block adds **two** contributions to the residual stream (attention, then MLP). So `L` blocks add `2L` contributions total.

Suppose each contribution has variance `s²`, and they're roughly independent. **Variances of independent things add:**

```
Var[stream after 1 contribution]   = Var[input] + s^2
Var[stream after 2 contributions]  = Var[input] + 2s^2
...
Var[stream after 2L contributions] = Var[input] + 2L * s^2
```

So the stream's **standard deviation grows like `sqrt(2L)`**. At `L=6` that's `sqrt(12) ≈ 3.5x`. At `L=48` (GPT-2 XL) it's `sqrt(96) ≈ 9.8x`. The later blocks end up shouting into a stream whose scale they can't control.

**The fix:** scale the *output projections* (`attn.c_proj`, `mlp.c_proj`) by `1/sqrt(2L)` at init. Then each contribution has variance `s²/(2L)`, and:

```
Var[stream] = Var[input] + 2L * (s^2 / 2L) = Var[input] + s^2      <- O(1). Depth-independent.
```

Same reasoning as ResNet's zero-init-last-BN. The GPT-2 paper states it directly.

In code it's a name check on `c_proj.weight` inside `_init_weights` — cryptic if you don't know what it's for, load-bearing if you do.

In [ ]:
class GPT(nn.Module):

    def __init__(self, config, flash=True):
        super().__init__()
        assert config.vocab_size is not None and config.block_size is not None
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte  = nn.Embedding(config.vocab_size, config.n_embd),   # (V, C)
            wpe  = nn.Embedding(config.block_size, config.n_embd),   # (block, C)
            drop = nn.Dropout(config.dropout),
            h    = nn.ModuleList([Block(config, flash=flash) for _ in range(config.n_layer)]),
            ln_f = LayerNorm(config.n_embd, bias=config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)  # weight (V, C)

        # WEIGHT TYING: both are (V, C), so this is ONE tensor with two names.
        self.transformer.wte.weight = self.lm_head.weight

        self.apply(self._init_weights)
        # the 1/sqrt(2L) residual init -- see the variance walk above
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def get_num_params(self, non_embedding=True):
        n = sum(p.numel() for p in self.parameters())
        if non_embedding:
            n -= self.transformer.wpe.weight.numel()   # wte is tied -> counted via lm_head
        return n

    def forward(self, idx, targets=None):
        B, T = idx.size()                                                    # (B, T)
        assert T <= self.config.block_size, f"T={T} exceeds block_size"
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)        # (T,)

        tok_emb = self.transformer.wte(idx)      # (B, T)  -> (B, T, C)
        pos_emb = self.transformer.wpe(pos)      # (T,)    -> (T, C)
        x = self.transformer.drop(tok_emb + pos_emb)    # broadcast -> (B, T, C)

        for block in self.transformer.h:
            x = block(x)                         # (B, T, C) -> (B, T, C)
        x = self.transformer.ln_f(x)             # (B, T, C)

        if targets is not None:
            logits = self.lm_head(x)             # (B, T, C) -> (B, T, V)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),   # (B*T, V)
                                   targets.view(-1),                   # (B*T,)
                                   ignore_index=-1)
        else:
            # INFERENCE: only the last position's logits are ever used.
            # x[:, [-1], :] KEEPS the axis -> (B, 1, C). x[:, -1, :] would give (B, C).
            logits = self.lm_head(x[:, [-1], :])  # (B, 1, C) -> (B, 1, V)
            loss = None
        return logits, loss

print("GPT defined.")

In [ ]:
model = GPT(REAL, flash=True)
print(f"parameters (non-embedding) : {model.get_num_params()/1e6:.2f} M")
print(f"parameters (all)           : {model.get_num_params(False)/1e6:.2f} M")

# --- weight tying is literally ONE tensor ---
print(f"\nwte.weight     : {tuple(model.transformer.wte.weight.shape)}   (nn.Embedding(V,C))")
print(f"lm_head.weight : {tuple(model.lm_head.weight.shape)}   (nn.Linear(C,V) -> weight is (V,C)!)")
print(f"same Python object? : {model.transformer.wte.weight is model.lm_head.weight}")
print(f"same memory?        : "
      f"{model.transformer.wte.weight.data_ptr() == model.lm_head.weight.data_ptr()}")
V, C = REAL.vocab_size, REAL.n_embd
print(f"\ntying saves V*C = {V*C/1e6:.1f}M params = "
      f"{V*C/(model.get_num_params(False)+V*C)*100:.0f}% of the untied model")
print(f"  tied   : {model.get_num_params(False)/1e6:.1f}M")
print(f"  untied : {(model.get_num_params(False)+V*C)/1e6:.1f}M")
print("\nIt works because nn.Linear(C,V).weight is (V,C) -- Linear computes x @ W.T --")
print("which is EXACTLY nn.Embedding(V,C).weight's shape. Same shape -> one tensor.")

In [ ]:
# --- parameter budget, computed rather than asserted ---
cfg = REAL
C, L, V, blk = cfg.n_embd, cfg.n_layer, cfg.vocab_size, cfg.block_size
rows = [
    ("wte (tied with lm_head)", V * C,           f"V*C = {V}*{C}"),
    ("wpe",                     blk * C,         f"block*C = {blk}*{C}"),
    (f"attn.c_attn  x{L}",      L * 3 * C * C,   "L*3C^2"),
    (f"attn.c_proj  x{L}",      L * C * C,       "L*C^2"),
    (f"mlp.c_fc     x{L}",      L * 4 * C * C,   "L*4C^2"),
    (f"mlp.c_proj   x{L}",      L * 4 * C * C,   "L*4C^2"),
    (f"layernorms   x{2*L+1}",  (2 * L + 1) * C, "(2L+1)*C"),
]
total = sum(r[1] for r in rows)
print(f"{'component':<26}{'params':>12}  {'formula':<18}{'share':>7}")
print("-" * 68)
for name, n, formula in rows:
    print(f"{name:<26}{n:>12,}  {formula:<18}{n/total*100:>6.1f}%")
print("-" * 68)
print(f"{'TOTAL':<26}{total:>12,}{'':20}{100.0:>6.1f}%")
assert abs(total - model.get_num_params(False)) < 10
print(f"\n[OK] matches model.get_num_params(False) = {model.get_num_params(False):,}")
print(f"\nPer block: 12*C^2, of which 8*C^2 (67%) is the MLP.")
print(f"Embeddings are {(V*C + blk*C)/total*100:.0f}% of THIS model -- at C=384 the vocab")
print("dominates. In GPT-2 XL (C=1600, L=48) the blocks dominate instead.")

In [ ]:
# --- 7.5 verified: does 1/sqrt(2L) actually keep the stream O(1)? ---
print("Residual stream std after L blocks, WITH vs WITHOUT the 1/sqrt(2L) init:\n")
print(f"{'L':>4} {'sqrt(2L)':>10} {'with scaling':>14} {'without':>12}")
print("-" * 44)
for L_test in [2, 6, 12, 24]:
    c = GPTConfig(block_size=32, vocab_size=256, n_layer=L_test, n_head=4,
                  n_embd=64, dropout=0.0, bias=False)
    x0 = torch.randn(2, 16, 64)

    m_with = GPT(c, flash=True)                       # 1/sqrt(2L) applied in __init__
    m_without = GPT(c, flash=True)
    for pn, p in m_without.named_parameters():        # undo it
        if pn.endswith('c_proj.weight'):
            torch.nn.init.normal_(p, mean=0.0, std=0.02)

    stds = []
    for m in (m_with, m_without):
        with torch.no_grad():
            h = x0.clone()
            for b in m.transformer.h:
                h = b(h)
        stds.append(h.std().item())
    print(f"{L_test:>4} {math.sqrt(2*L_test):>10.2f} {stds[0]:>14.3f} {stds[1]:>12.3f}")

print("\nWithout scaling, the stream's std climbs with depth: 2L independent")
print("contributions, variances ADD, so std grows like sqrt(2L).")
print("With 1/sqrt(2L), each contribution has variance s^2/(2L), so 2L of them")
print("sum back to O(1) -- depth-independent. That is the whole argument.")

In [ ]:
# --- 7.4: the vocab padding trick, on real hardware ---
if device_type == 'cuda':
    def bench_head(V, C=384, BT=64*256, iters=30):
        W = torch.randn(V, C, device=device, dtype=torch.bfloat16)
        x = torch.randn(BT, C, device=device, dtype=torch.bfloat16)
        for _ in range(5): _ = x @ W.T
        torch.cuda.synchronize(); t0 = time.time()
        for _ in range(iters): _ = x @ W.T
        torch.cuda.synchronize()
        return (time.time() - t0) / iters * 1000

    t1, t2 = bench_head(50257), bench_head(50304)
    print(f"lm_head matmul, (16384, 384) @ (384, V), bf16:")
    print(f"  V = 50257 (ragged tail tile) : {t1:.3f} ms")
    print(f"  V = 50304 (multiple of 64)   : {t2:.3f} ms   <- {(t1/t2-1)*100:+.1f}%")
    print(f"\n{(50304-50257)/50257*100:.2f}% MORE FLOPs, and measurably faster.")
else:
    print("(CUDA only -- on an A100 the 50304 version is typically faster)")
    print(f"50304 = 50257 rounded up to a multiple of 64.")
    print(f"That is {(50304-50257)/50257*100:.2f}% more FLOPs, for ~25% faster steps (Karpathy's")
    print("nanoGPT measurement). GEMM kernels tile along output columns; a ragged")
    print("tail tile runs at a fraction of peak.")
print("\nGeneral rule: keep EVERY matmul dimension a multiple of 64 (128 for fp8).")

---
# Part 8 — The loss

## 8.1 The shape gymnastics

We have `logits (B, T, V)` and `targets (B, T)`. `F.cross_entropy` wants `(N, V)` and `(N,)`. So:

```python
loss = F.cross_entropy(logits.view(-1, logits.size(-1)),   # (B, T, V) -> (B*T, V)
                       targets.view(-1))                   # (B, T)    -> (B*T,)
```

`B*T` predictions, `B*T` targets, one mean.

**The `(B, T)` structure is irrelevant to the loss.** Every position is an independent classification over `V` classes. All the sequential structure was already imposed by the causal mask, upstream. Sit with that for a second: **the loss function has no idea it's training a language model.** It's just doing 16,384 unrelated multi-class classifications and averaging.

`.view(-1, V)`: the `-1` means "infer this axis from the total element count." Safe here because `logits` comes fresh from `lm_head` and is contiguous. Use `.reshape` if unsure.

## 8.2 What `cross_entropy` actually does

```
cross_entropy(logits, target)  ==  nll_loss(log_softmax(logits, dim=-1), target)
                               ==  mean over n of  -log_softmax(logits[n])[target[n]]
```

In words: turn logits into probabilities, take the probability assigned to the *correct* class, take its negative log, average over all positions.

## 8.3 Hand calculation

`V=4`, `logits = [1, 2, 3, 4]`, `target = 2`:

```
exp:      [e^1, e^2, e^3, e^4] = [2.7183, 7.3891, 20.0855, 54.5982]
sum:      84.7911
softmax:  [0.0321, 0.0871, 0.2369, 0.6439]
pick index 2:  0.2369
loss = -log(0.2369) = 1.4402
```

## 8.4 The log-sum-exp trick — derived

I previously named this without deriving it. Here it is.

**The problem.** Start from the definition:

```
log_softmax(x)_i  =  log( exp(x_i) / sum_j exp(x_j) )
```

Compute that literally and you must evaluate `exp(x_j)`. In fp32, `exp` overflows to `inf` at about `x = 88`. Logits in a trained model routinely exceed that. Then `inf / inf = nan`, and your loss is destroyed.

**The derivation.** Use `log(a/b) = log(a) - log(b)`:

```
log_softmax(x)_i  =  log(exp(x_i))  -  log(sum_j exp(x_j))
                  =  x_i  -  log(sum_j exp(x_j))
                     ^^^
                     the first exp is already gone -- log(exp(x)) = x
```

That `log(sum_j exp(x_j))` term is the **log-sum-exp**. Now the trick. Pick `m = max(x)` and multiply inside by `exp(-m)/exp(-m)` (which is 1):

```
log sum_j exp(x_j)  =  log[ exp(m) * sum_j exp(x_j - m) ]
                    =  log(exp(m))  +  log sum_j exp(x_j - m)
                    =  m  +  log sum_j exp(x_j - m)
```

**Look at what that bought us.** Every exponent is now `x_j - m <= 0`, so every `exp(x_j - m) <= 1`. **Nothing can overflow.** The largest term is exactly `exp(0) = 1`.

The final, safe formula:

```
log_softmax(x)_i  =  x_i  -  m  -  log sum_j exp(x_j - m)         where m = max(x)
```

Mathematically identical to the definition. Numerically bulletproof.

**This is what `F.cross_entropy` does internally, in one fused kernel.** Which is why you should never hand-roll `log(softmax(x))` from parts. (Amusingly, `F.softmax` also does the max-subtraction internally, so `-log(F.softmax(x))` survives too. The formula that dies is the textbook one: `exp(x) / sum(exp(x))`.)

## 8.5 The single most useful sanity check in LLM training

At initialisation, the model knows nothing. So it should predict **uniform** over `V`:

```
p = 1/V for every class
loss = -log(1/V) = log(V) = log(50304) = 10.8258
```

**If your initial loss is not ~10.83, stop and debug before spending a GPU-hour.**

| initial loss | almost certainly |
|---|---|
| `~10.83 = ln(V)` | correct |
| much **higher** (15, 20, 100) | init std too large — logits are already confidently wrong |
| much **lower** (5, 2) | **label leakage.** Your mask is broken, or `x`/`y` are misaligned |
| `nan` | LR too high, or you hand-rolled `log(softmax(...))` |

**That third row is why this check is worth doing every single time.** Remember Part 1.1: a batch of `(64, 256)` is 16,384 supervised examples, and every one of them is bought by the causal mask. A broken mask *lowers* your loss, looks like progress, and never raises an exception.

## 8.6 `ignore_index=-1`

Targets equal to `ignore_index` contribute **nothing** — no loss, no gradient — and are excluded from the mean's denominator.

Unused in this notebook (every TinyStories token is a real target), but it's how you'd mask padding, and how instruction-tuning masks the prompt so the model is only scored on the completion.

Note: it's `-1` in nanoGPT but PyTorch's **default is `-100`**. A mismatch here silently trains on your padding.

In [ ]:
logits_1 = torch.tensor([1.0, 2.0, 3.0, 4.0])
target_1 = torch.tensor(2)

e = logits_1.exp()
probs = e / e.sum()
hand_loss = -torch.log(probs[target_1])
print(f"logits          : {logits_1.tolist()}")
print(f"exp             : {[round(v, 4) for v in e.tolist()]}")
print(f"sum             : {e.sum().item():.4f}")
print(f"softmax         : {[round(v, 4) for v in probs.tolist()]}")
print(f"p[target=2]     : {probs[target_1].item():.4f}")
print(f"loss = -log(p)  : {hand_loss.item():.4f}")

torch_loss = F.cross_entropy(logits_1.unsqueeze(0), target_1.unsqueeze(0))
print(f"F.cross_entropy : {torch_loss.item():.4f}")
assert torch.allclose(hand_loss, torch_loss, atol=1e-5)
print("\n[OK] hand == F.cross_entropy")
print(f"\nequivalent decomposition, nll_loss(log_softmax(x)) : "
      f"{F.nll_loss(F.log_softmax(logits_1.unsqueeze(0), -1), target_1.unsqueeze(0)).item():.4f}")

In [ ]:
# --- 8.4: the log-sum-exp derivation, verified step by step ---
big = torch.tensor([1000.0, 999.0, 998.0])
tgt = torch.tensor([0])

print(f"logits = {big.tolist()}   (a trained model really does produce logits like this)\n")
print("THE TEXTBOOK FORMULA:  exp(x) / sum(exp(x))")
print(f"  x.exp()                    = {big.exp().tolist()}   <- fp32 exp overflows at x~88")
naive = big.exp() / big.exp().sum()
print(f"  exp(x)/sum(exp(x))         = {naive.tolist()}   <- inf/inf")
print(f"  -log(that)[0]              = {-torch.log(naive)[0].item()}   <- nan. Run destroyed.\n")

print("THE DERIVATION, step by step:")
m = big.max()
print(f"  step 1: m = max(x)                       = {m.item()}")
shifted = big - m
print(f"  step 2: x - m                            = {shifted.tolist()}   <- all <= 0")
print(f"  step 3: exp(x - m)                       = {shifted.exp().tolist()}   <- all <= 1. SAFE.")
lse = m + torch.log(shifted.exp().sum())
print(f"  step 4: logsumexp = m + log(sum(exp(x-m))) = {lse.item():.4f}")
ls = big - lse
print(f"  step 5: log_softmax = x - logsumexp      = {[round(v,4) for v in ls.tolist()]}")
print(f"  step 6: loss = -log_softmax[target=0]    = {-ls[0].item():.4f}")

print(f"\n  F.cross_entropy(x, t)                    = {F.cross_entropy(big.unsqueeze(0), tgt).item():.4f}")
assert torch.allclose(-ls[0], F.cross_entropy(big.unsqueeze(0), tgt), atol=1e-4)
print("\n[OK] the derived formula matches F.cross_entropy exactly.")
print("     Mathematically identical to the definition. Numerically bulletproof.")
print(f"\n  (torch's own logsumexp agrees: {torch.logsumexp(big, 0).item():.4f})")
print("  F.cross_entropy does all of this in ONE fused kernel. Never assemble it from parts.")

In [ ]:
# --- the (B,T,V) -> (B*T,V) flatten ---
B_, T_, V_ = 2, 3, 5
logits = torch.randn(B_, T_, V_)
targets = torch.randint(0, V_, (B_, T_))
sh("logits", logits); sh("targets", targets)
sh("logits.view(-1, V)", logits.view(-1, V_))
sh("targets.view(-1)", targets.view(-1))
loss = F.cross_entropy(logits.view(-1, V_), targets.view(-1))
print(f"loss (a scalar): {loss.item():.4f}   -- the mean over {B_*T_} classifications")

# the loss is genuinely position-agnostic
per_pos = torch.stack([F.cross_entropy(logits[b, t].unsqueeze(0), targets[b, t].unsqueeze(0))
                       for b in range(B_) for t in range(T_)])
print(f"\nthe {B_*T_} per-position losses: {[round(v,3) for v in per_pos.tolist()]}")
print(f"their mean                  : {per_pos.mean().item():.4f}")
assert torch.allclose(loss, per_pos.mean(), atol=1e-6)
print("[OK] identical. The loss has NO notion of sequence -- just B*T independent")
print("     classifications. All the sequential structure came from the causal mask.")

# --- ignore_index ---
t2 = targets.clone(); t2[0, 0] = -1
print(f"\nwith targets[0,0] = -1 and ignore_index=-1:")
print(f"  loss over all {B_*T_}      : {loss.item():.4f}")
print(f"  loss ignoring 1 pos  : "
      f"{F.cross_entropy(logits.view(-1,V_), t2.view(-1), ignore_index=-1).item():.4f}")
print("  that position is dropped from BOTH the sum and the denominator.")
print("  (PyTorch's DEFAULT is -100, not -1. A mismatch silently trains on padding.)")

In [ ]:
# ============================================================================
# THE SANITY CHECK. Run this before every training run. No exceptions.
# ============================================================================
m = GPT(REAL, flash=True)
m.eval()
idx = torch.randint(0, REAL.vocab_size, (4, 32))
tgt = torch.randint(0, REAL.vocab_size, (4, 32))
with torch.no_grad():
    logits, loss = m(idx, tgt)

sh("idx    (B, T)", idx)
sh("logits (B, T, V)", logits)
expected = math.log(REAL.vocab_size)
print(f"\ninitial loss : {loss.item():.4f}")
print(f"ln(V)        : {expected:.4f}     (V = {REAL.vocab_size:,})")
print(f"ratio        : {loss.item()/expected:.4f}")
print(f"\nwhy ln(V)? an untrained model should predict UNIFORM: p = 1/V for every class.")
print(f"            loss = -log(1/V) = log(V) = {expected:.4f}")
assert abs(loss.item() - expected) < 0.2, "INIT IS BROKEN -- do not start training"
print("\n[OK] the model starts out predicting ~uniform over the vocab, as it must.")
print()
print("   Diagnosis table:")
print("     ~10.83 = ln(V) ....... correct")
print("     much higher .......... init std too large: confidently wrong at step 0")
print("     much LOWER ........... LABEL LEAKAGE. Broken mask, or x/y misaligned.")
print("                            This one NEVER raises. It just looks like progress.")
print("     nan .................. LR too high, or you hand-rolled log(softmax(x))")

In [ ]:
# --- 7.2: the logits are the memory story ---
B_, T_ = 64, 256
n = B_ * T_ * REAL.vocab_size
params = m.get_num_params(False)
print(f"At B={B_}, T={T_}, V={REAL.vocab_size:,}:\n")
print(f"  logits elements : {n:,}")
print(f"  logits fp32     : {n*4/1e9:.2f} GB")
print(f"  whole model     : {params/1e6:.0f}M params = {params*4/1e9:.2f} GB")
print(f"  ratio           : the logits are {n/params:.0f}x LARGER than the model\n")
print("  and cross_entropy needs a log_softmax buffer of the same size on top.")
print("\nThis is the peak-memory moment of the entire training step. It is why:")
print("  - the LM head is the first thing sharded in tensor parallelism")
print("  - fused CE kernels (Liger, torch.compile) give large wins")
print("  - at inference we compute logits for ONLY the last position (Part 10.1)")

---
# Part 9 — Training

## 9.1 AdamW parameter groups: why the `dim >= 2` rule

Weight decay is applied to **matmul weights only**. LayerNorm gammas/betas and biases are exempt. The rule that implements this is one line:

```python
decay    = [p for p in params if p.dim() >= 2]     # (V,C), (C,C), (4C,C), ...
no_decay = [p for p in params if p.dim() <  2]     # (C,) LayerNorm weights and biases
```

### Why shape is the right proxy

Because **every 2-D parameter in a transformer is a matmul weight, and every 1-D one is a gain or an offset.** Look at what decay would actually *mean* for each:

- **On a matmul weight**, `w -= lr * wd * w` shrinks the function's sensitivity to its input. That's the L2 regularisation you actually want.
- **On a LayerNorm gamma**, whose entire job is to sit around `1.0` (Part 3.2), decay drags it toward `0` — progressively scaling the block's output to nothing. You're not regularising; you're deleting the layer.
- **On a bias**, you're penalising the model for having a preferred output. That regularises nothing and just adds bias to the estimator.

**`wte` is `(V, C)`, so it gets decayed.** People trip on this — surely embeddings are a lookup, not a matmul? But here it's *both*, because of weight tying (Part 7.3): the same tensor **is** the output matmul. nanoGPT decays it and it works fine.

## 9.2 AdamW vs Adam+L2 — not the same algorithm

These have confusingly similar names and are genuinely different optimizers. I asserted this before; let's derive it and then measure it.

**Adam + L2** adds `wd*w` to the **gradient**, before Adam sees it:

```
g  =  g + wd*w                                   the decay is now part of "the gradient"
w  =  w - lr * m_hat / (sqrt(v_hat) + eps)       ...and Adam normalises the whole thing
```

**AdamW** applies the decay **directly to the weight**, after the Adam update, untouched:

```
w  =  w - lr * m_hat / (sqrt(v_hat) + eps)  -  lr * wd * w
                                               ^^^^^^^^^^^
                                               never goes through the 1/sqrt(v)
```

### Why the difference is fatal for Adam+L2

Adam divides every update by `sqrt(v)` — a running estimate of the gradient's magnitude. So in Adam+L2, the decay term `wd*w` **also** gets divided by `sqrt(v)`.

**The effective decay per parameter therefore gets divided by that parameter's gradient magnitude.** Parameters with big gradients get decayed *less*. Parameters with tiny gradients get decayed *more*.

That's incoherent as regularisation — the amount of shrinkage a weight receives now depends on something that has nothing to do with how much shrinkage it needs. That incoherence is the entire content of the AdamW paper.

The clean way to see it: **set the gradient to exactly zero.** Then only decay can move the weight.
- AdamW should move it by exactly `lr * wd * w`.
- Adam+L2 has `g = 0 + wd*w`, so *the decay is the only signal*. Adam's `m/sqrt(v)` ratio for a constant signal is ≈ 1. So the step is ≈ `lr` — **completely independent of `wd`, and 10x too big.**

The cell below measures exactly this.

> `torch.optim.AdamW` is the decoupled one. `torch.optim.Adam(weight_decay=...)` is the mangled one.

## 9.3 Betas and the fused kernel

`betas=(0.9, 0.95)`. Note **`0.95`, not torch's default `0.999`.** Every GPT paper uses `0.95` — a shorter memory for the second moment, which suits the noisy, non-stationary gradients of LM pretraining.

`fused=True` runs the whole update as a single fused CUDA kernel instead of dozens of small elementwise ops. The optimizer step is pure memory-bandwidth work — it touches `4 x n_params` bytes and does almost no arithmetic — so fusing is a straight win. Free performance, one kwarg.

**Adam's memory tax:** `exp_avg` and `exp_avg_sq`, both fp32, both the size of the parameters. Plus fp32 master weights and fp32 gradients. That's the famous **~16 bytes per parameter** for mixed-precision AdamW — **4x the model, just to train it.** For our 30M model, ~0.5 GB: irrelevant. For a 70B model, **1.1 TB**, and that number is the entire reason ZeRO and FSDP exist.

In [ ]:
# ============================================================================
# THE CLAIM: Adam+L2 lets the 1/sqrt(v) normalisation mangle the decay.
# THE TEST: set the gradient to EXACTLY ZERO. Then only decay can move the weight.
# ============================================================================
print("w = 1.0, lr = 0.1, wd = 0.1, gradient = EXACTLY 0.\n")
print("AdamW should move w by exactly lr*wd*w = 0.1*0.1*1.0 = 0.01 per step.\n")

for name, Opt in [("AdamW  (decoupled)", torch.optim.AdamW),
                  ("Adam+L2 (coupled) ", torch.optim.Adam)]:
    w = torch.tensor([1.0], requires_grad=True)
    opt = Opt([w], lr=0.1, betas=(0.9, 0.95), weight_decay=0.1)
    traj = []
    for _ in range(5):
        w.grad = torch.zeros_like(w)               # ZERO gradient
        opt.step(); opt.zero_grad(set_to_none=False)
        traj.append(round(w.item(), 5))
    print(f"  {name}: {traj}")

print("\nAdamW  : 1.00 -> 0.99 -> 0.9801 ...  exactly lr*wd*w per step. As designed.")
print("Adam+L2: 1.00 -> 0.90 -> 0.80   ...  TEN TIMES faster.")
print("\nWhy: Adam+L2 sets g = 0 + wd*w, so the decay IS the only gradient signal.")
print("     Adam's m/sqrt(v) ratio for a constant signal is ~1, so the step is ~lr = 0.1,")
print("     NOT lr*wd*w = 0.01. The wd magnitude has been normalised away entirely.")

In [ ]:
# The deeper problem: in Adam+L2 the effective decay depends on GRADIENT MAGNITUDE.
print("Same wd=0.1, same lr, 50 steps of ZERO-MEAN noise at three magnitudes.\n")
print(f"{'|grad|':>10} {'AdamW':>10} {'Adam+L2':>10}   verdict")
print("-" * 56)
for gmag in [0.001, 1.0, 100.0]:
    ws = {}
    for name, Opt in [("AdamW", torch.optim.AdamW), ("Adam+L2", torch.optim.Adam)]:
        w = torch.tensor([1.0], requires_grad=True)
        opt = Opt([w], lr=0.01, betas=(0.9, 0.95), weight_decay=0.1)
        for i in range(50):
            w.grad = torch.full_like(w, gmag) * (1 if i % 2 else -1)   # zero-mean
            opt.step(); opt.zero_grad(set_to_none=False)
        ws[name] = w.item()
    note = "Adam+L2's weight GREW despite decay!" if ws['Adam+L2'] > 1.0 else ""
    print(f"{gmag:>10} {ws['AdamW']:>10.4f} {ws['Adam+L2']:>10.4f}   {note}")

print("\nAdamW  : 0.9680 every time. The decay is IDENTICAL regardless of gradient size.")
print("         That word -- 'decoupled' -- means exactly this.")
print("Adam+L2: 0.52 / 0.97 / 1.02. Same wd, wildly different shrinkage. At |grad|=100")
print("         the weight actually GREW. That is incoherent as regularisation,")
print("         and it is the entire content of the AdamW paper.")
print("\ntorch.optim.AdamW = decoupled.  torch.optim.Adam(weight_decay=) = mangled.")

In [ ]:
def configure_optimizers(model, weight_decay, learning_rate, betas, device_type):
    param_dict = {pn: p for pn, p in model.named_parameters() if p.requires_grad}

    # dim >= 2  <=>  it is a matmul weight  <=>  decay it.
    # dim <  2  <=>  it is a LayerNorm gain or a bias  <=>  do not.
    decay_params   = [p for n, p in param_dict.items() if p.dim() >= 2]
    nodecay_params = [p for n, p in param_dict.items() if p.dim() <  2]
    optim_groups = [
        {'params': decay_params,   'weight_decay': weight_decay},
        {'params': nodecay_params, 'weight_decay': 0.0},
    ]
    print(f"decayed   : {len(decay_params):>3} tensors, {sum(p.numel() for p in decay_params):>11,} params")
    print(f"un-decayed: {len(nodecay_params):>3} tensors, {sum(p.numel() for p in nodecay_params):>11,} params")

    # fused AdamW: ONE CUDA kernel for the whole update instead of dozens of tiny ones
    fused_available = 'fused' in inspect.signature(torch.optim.AdamW).parameters
    use_fused = fused_available and device_type == 'cuda'
    extra = dict(fused=True) if use_fused else dict()
    print(f"fused AdamW: {use_fused}")
    return torch.optim.AdamW(optim_groups, lr=learning_rate, betas=betas, **extra)


model = GPT(REAL, flash=True).to(device)
optimizer = configure_optimizers(model, weight_decay=0.1, learning_rate=1e-3,
                                 betas=(0.9, 0.95), device_type=device_type)

print("\nwhat landed in the NO-decay group (dim < 2):")
shown = 0
for n, p in model.named_parameters():
    if p.dim() < 2 and shown < 3:
        print(f"   {n:<40} {tuple(p.shape)}")
        shown += 1
print("   ... every LayerNorm gamma. Their job is to sit near 1.0 (Part 3.2);")
print("       decaying them toward 0 would scale the block's output away entirely.")

n_params = sum(p.numel() for p in model.parameters())
print(f"\nAdamW memory tax at {n_params/1e6:.0f}M params:")
for label in ["fp32 master weights", "fp32 gradients", "exp_avg   (m)", "exp_avg_sq(v)"]:
    print(f"  {label:<22} {n_params*4/1e6:>7.1f} MB")
print(f"  {'-'*30}")
print(f"  TOTAL ~16 bytes/param  {n_params*16/1e6:>7.1f} MB   <- 4x the model, just to train it")
print(f"\n  at 70B params that is {70e9*16/1e12:.1f} TB. THAT is why ZeRO/FSDP exist.")

## 9.4 The LR schedule: linear warmup + cosine decay

```
lr(it) = lr_max * it / warmup                                 it < warmup       (linear up)
       = lr_min                                               it > decay_iters  (floor)
       = lr_min + 0.5*(1+cos(pi*progress))*(lr_max - lr_min)  otherwise         (cosine down)
```

### Why warmup exists — and it isn't superstition

Here's the actual mechanism, which I previously named ("bias correction") without explaining.

Adam maintains `v` = a running average of squared gradients, and divides the step by `sqrt(v)`. But `v` starts at **zero**. So early on, `v` is badly underestimated — it's an average that has barely averaged anything yet.

Adam corrects for this with **bias correction**: it divides `v` by `(1 - beta2^t)` to un-bias the running average. At `beta2 = 0.95`:

```
t = 1  ->  1 - 0.95^1  =  0.0500     divide v by 0.05  = multiply by 20
t = 2  ->  1 - 0.95^2  =  0.0975
t = 10 ->  1 - 0.95^10 =  0.4013
t = 100->  1 - 0.95^100=  0.9941     essentially 1 now. Correction is done.
```

So at step 1, `v_hat` is built from a **single gradient sample**, multiplied by 20. It's a wild, high-variance guess. And the step size is `lr * m_hat / sqrt(v_hat)` — so the first few updates are effectively **enormous and randomly directed**. They can move the weights far enough that the model never recovers the careful scaling its init gave it (Part 7.5).

**Warmup keeps `lr` tiny for exactly as long as it takes `v` to become a usable running average** — a few hundred steps at `beta2=0.95`.

This also explains the folklore that *"large-batch training needs longer warmup"*: bigger batches mean fewer steps to accumulate `v`, so you need more wall-clock warmup, not less.

### Why cosine

Empirical, but robustly so. Early training wants a large LR to traverse the loss landscape; late training wants a small one to settle into a minimum. Cosine spends more time at high LR than linear decay does (it's flat at the top), then decays smoothly to near zero.

**The catch worth knowing:** cosine is defined relative to a *fixed total step count*. Stop at 60% of a cosine schedule and you get a model meaningfully worse than one whose cosine was *designed* to end at 60% — you were still at a high LR, mid-traverse. This is why you can't naively extend a run, and why constant-with-cooldown (WSD) schedules are gaining ground: they let you decide the token budget *after* the run has started.

`min_lr = max_lr / 10` is standard, straight from Chinchilla.

In [ ]:
# --- why warmup: Adam's bias correction at beta2 = 0.95 ---
print("Adam divides v by (1 - beta2^t) to un-bias the running average.\n")
print(f"{'step t':>8} {'1 - 0.95^t':>12} {'-> multiplies v_hat by':>24}")
print("-" * 48)
for t in [1, 2, 5, 10, 50, 100, 300]:
    bc = 1 - 0.95**t
    print(f"{t:>8} {bc:>12.4f} {1/bc:>24.1f}x")
print("\nAt t=1, v_hat comes from ONE gradient sample, multiplied by 20.")
print("The step is lr * m_hat/sqrt(v_hat) -- so the first updates are enormous")
print("and randomly directed. They can destroy the init's careful scaling (Part 7.5).")
print("\nWarmup keeps lr tiny for exactly as long as it takes v to become usable.")
print("By t~100 the correction is ~1.0 and Adam is trustworthy.")

In [ ]:
learning_rate = 1e-3      # small models tolerate (and want) a larger LR than GPT-2's 6e-4
min_lr        = 1e-4      # ~ lr/10, per Chinchilla
warmup_iters  = 200
lr_decay_iters = 10000    # should EQUAL max_iters -- see the cosine caveat above
max_iters     = 10000

def get_lr(it):
    if it < warmup_iters:                                  # 1) linear warmup
        return learning_rate * (it + 1) / (warmup_iters + 1)
    if it > lr_decay_iters:                                # 2) floor
        return min_lr
    decay_ratio = (it - warmup_iters) / (lr_decay_iters - warmup_iters)   # 0 -> 1
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))                 # 1 -> 0
    return min_lr + coeff * (learning_rate - min_lr)       # 3) cosine

import matplotlib.pyplot as plt
its = np.arange(0, max_iters + 500)
lrs = [get_lr(i) for i in its]
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].plot(its, lrs); ax[0].set_title("full schedule"); ax[0].set_xlabel("iteration")
ax[0].axvline(warmup_iters, color='r', ls='--', lw=1, label=f'warmup ends @ {warmup_iters}')
ax[0].axhline(min_lr, color='g', ls=':', lw=1, label='min_lr'); ax[0].legend(); ax[0].set_ylabel("lr")
ax[1].plot(its[:600], lrs[:600]); ax[1].set_title("first 600 iters (the warmup ramp)")
ax[1].axvline(warmup_iters, color='r', ls='--', lw=1); ax[1].set_xlabel("iteration")
plt.tight_layout(); plt.show()

for i in [0, 50, warmup_iters, 1000, 5000, 9000, max_iters]:
    print(f"  iter {i:>6} -> lr {get_lr(i):.2e}")

## 9.5 bf16 autocast: why no GradScaler on A100

### The three float formats

| | exponent bits | mantissa bits | max value | smallest normal |
|---|---|---|---|---|
| fp32 | 8 | 23 | 3.4e38 | 1.2e-38 |
| fp16 | 5 | 10 | **65504** | **6.1e-5** |
| **bf16** | **8** | 7 | **3.4e38** | **1.2e-38** |

Read the table as one trade: **bf16 keeps fp32's exponent and pays for it out of the mantissa.** fp16 does the opposite.

- **Exponent bits** decide the *range* — how big and how small a number can be.
- **Mantissa bits** decide the *precision* — how many significant digits.

That one difference decides everything downstream.

fp16's smallest **normal** value is `6.1e-5`. Gradients in a transformer routinely live below that, in fp16's **subnormal** range where precision degrades progressively, and anything under about `6e-8` flushes to **exactly zero** — at which point those weights stop learning and nothing warns you.

The fix is `GradScaler`: multiply the loss by ~`65536` before backward to shift the whole gradient distribution up into fp16's normal range, unscale before the optimizer step, and dynamically back off whenever an `inf` appears. It works. It's a pile of machinery in service of five missing exponent bits.

**bf16 has fp32's exponent range, so gradients never underflow, so no GradScaler.** You lose mantissa precision, but that turns out not to matter: the fp32 master weights hold the precision, and the gradient is a noisy Monte-Carlo estimate anyway — the noise from your minibatch dwarfs the noise from 7-bit mantissas.

A100 (Ampere) and later support bf16 in hardware. On a V100 (Volta) you have no choice but fp16 + GradScaler. This is one of the biggest quality-of-life differences between the two generations.

### What `autocast` actually does

It is **not** "cast everything to bf16". It maintains a **per-op allow-list**:

```
run in bf16   : matmul, addmm, linear, conv, bmm, einsum      (tensor-core ops, error-tolerant)
run in fp32   : softmax, layer_norm, log_softmax, sum, exp,
                cross_entropy, and anything reduction-ish       (accuracy-critical)
```

The distinction is exactly whether the op is a **reduction that accumulates error**. Summing 50,304 logits in bf16 loses real accuracy; a matmul does not, because tensor cores accumulate into fp32 internally anyway.

Meanwhile **the parameters stay fp32.** Autocast casts each op's *inputs* at the op boundary — it never touches your weights. So you get fp32 master weights, fp32 optimizer state, bf16 matmuls. That's what "mixed precision" means.

### The two other one-line wins

```python
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
```

TF32 is Ampere's fp32 tensor-core mode: 10-bit mantissa, fp32 exponent, accumulate in fp32. It makes fp32 matmuls ~8x faster on A100 with no code change and no accuracy impact worth measuring.

## 9.6 Gradient accumulation: the two details people get wrong

You want a batch of 65k tokens. It doesn't fit. So run `N` micro-batches, accumulate gradients, step once.

```
tokens_per_step = B * T * grad_accum_steps = 64 * 256 * 4 = 65,536
```

This works because **`loss.backward()` accumulates into `.grad` by default** rather than overwriting. That behaviour is usually the thing `zero_grad()` exists to undo; here it's exactly the feature.

### Detail 1: you must divide the loss

`F.cross_entropy` returns a **mean** over its `B*T` positions (Part 8.1). Summing `N` such means gives `N x` the correct gradient:

```
WRONG:  sum_i mean(loss_i)            = N * (the mean you wanted)
RIGHT:  sum_i mean(loss_i) / N        = the mean over all N*B*T positions
```

So `loss = loss / grad_accum_steps` before `backward()`. Get this wrong and your effective LR is silently `N` times too large — which usually just looks like "my model diverges at LR 1e-3 and everyone else's doesn't."

*(Strictly, this is only exactly right when every micro-batch has the same number of counted tokens. With `ignore_index` and ragged padding it isn't, and you need a token-weighted sum. Fixed `T` and no padding — Part 1.3 — is one more thing the flat memmap format buys you.)*

### Detail 2: gradient accumulation is not free

The naive claim is "same math, less memory." The math is identical; the **cost** is not.

`N` micro-batches means `N` forward+backward passes, `N` times the kernel launches, and — crucially — `N` times as many weight reads from HBM, over which you amortise `1/N` as much arithmetic each. Small models at small `B` are already launch- and bandwidth-bound (that's Part 5.2's `B*T` point), so `grad_accum=4` at `B=16` is measurably *slower* than `grad_accum=1` at `B=64` for identical token counts.

**Accumulate only when you must fit memory. Never as a default.**

## 9.7 Gradient clipping

```python
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
```

Computes the **global** L2 norm over *all* gradients concatenated (not per-tensor), and if it exceeds `1.0`, scales every gradient by `1.0/norm`. Direction preserved, magnitude capped.

The purpose isn't general regularisation — it's insurance against **individual bad batches**. LM data is heavy-tailed: somewhere in TinyStories is a sequence of repeated characters or a weird unicode run that produces a gradient 100x the typical norm. Without clipping, one such batch can blow the weights out of their basin and you get a loss spike that takes thousands of steps to recover from — or never does. `1.0` is universal and almost never worth tuning.

**Watch the returned norm.** It's the most informative number in your log. Healthy: starts ~1, settles to 0.1–0.5, occasional spikes. A sustained climb means divergence is coming and you should kill the run.

In [ ]:
torch.backends.cuda.matmul.allow_tf32 = True   # ~8x faster fp32 matmuls on Ampere
torch.backends.cudnn.allow_tf32 = True

print("format ranges:")
for name, dt in [("fp32", torch.float32), ("fp16", torch.float16), ("bf16", torch.bfloat16)]:
    fi = torch.finfo(dt)
    print(f"  {name}: max {fi.max:>10.3e}   smallest normal {fi.tiny:>10.3e}   eps {fi.eps:>9.3e}")
print("\n  exponent bits -> RANGE.  mantissa bits -> PRECISION.")
print("  bf16 keeps fp32's exponent (8 bits) and pays out of the mantissa (7 vs 23).")
print("  fp16 does the opposite: 5 exponent bits, 10 mantissa. Range is what it loses.")

print("\n--- the fp16 problem, demonstrated ---")
grad = torch.tensor([1e-4, 1e-6, 1e-8, 1e-10])
print(f"{'value':>10} {'fp16':>14} {'bf16':>14}   verdict")
print("-" * 62)
for g in grad:
    h, b = g.half(), g.bfloat16()
    if h.item() == 0:       note = "fp16 -> EXACTLY ZERO. This weight is dead."
    elif h.item() < 6.1e-5: note = "fp16 subnormal, precision degrading"
    else:                   note = "fp16 fine"
    print(f"{g.item():>10.0e} {h.item():>14.3e} {b.item():>14.3e}   {note}")
n0 = (grad.half() == 0).sum().item()
print(f"\nfp16 flushed {n0}/{len(grad)} to exactly zero; the rest are subnormal.")
print(f"bf16 represented all {len(grad)} as normals -- it has fp32's exponent range.")
print(f"\nGradScaler's fix: scale the loss by 65536 so the distribution moves up:")
print(f"  (grad*65536).half() = {[f'{v:.2e}' for v in (grad*65536).half().tolist()]}")
print(f"  ...then unscale before the step, and back off on any inf.")
print("bf16 needs NONE of that machinery. That is the whole argument.")
if device_type == 'cuda':
    print(f"\nbf16 supported on this GPU: {torch.cuda.is_bf16_supported()}")

In [ ]:
# --- what autocast actually casts: a per-OP allow-list, not a blanket cast ---
if device_type == 'cuda':
    ctx_demo = torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16)
    lin = nn.Linear(64, 64).cuda()
    ln  = nn.LayerNorm(64).cuda()
    xd = torch.randn(8, 64, device='cuda')
    with ctx_demo:
        print(f"  input dtype         : {xd.dtype}")
        print(f"  linear WEIGHT dtype : {lin.weight.dtype}   <- params STAY fp32. Never cast.")
        print(f"  linear(x)  dtype    : {lin(xd).dtype}   <- matmul: allow-listed -> bf16")
        print(f"  layer_norm dtype    : {ln(xd).dtype}   <- reduction -> stays fp32")
        print(f"  softmax    dtype    : {F.softmax(xd, -1).dtype}   <- reduction -> stays fp32")
    print("\nThe rule: tensor-core ops -> bf16. Reductions that ACCUMULATE ERROR -> fp32.")
    print("Autocast casts each op's INPUTS at the op boundary. Your weights are untouched.")
else:
    print("(CUDA only)")
    print("autocast allow-list: matmul/linear/conv/bmm/einsum -> bf16")
    print("                     softmax/layer_norm/sum/cross_entropy -> stay fp32")
    print("Parameters are NEVER cast. autocast casts each op's INPUTS at the boundary.")
    print("That is what 'mixed precision' means: fp32 weights, bf16 matmuls.")

## 9.8 MFU and the `6N` rule — derived

**Model FLOPs Utilisation** = achieved FLOPs / hardware peak FLOPs. The one number that tells you whether you're compute-bound or wasting the machine.

I previously asserted "6 FLOPs per param per token: 2 forward + 4 backward". Let's derive it, because it's the back-of-envelope every scaling calculation starts from.

### Forward: 2 FLOPs per parameter per token

Take one `Linear(in, out)`. Its weight `W` has `in * out` parameters. For **one token**, computing `y = x @ W` means: for each of the `out` outputs, a dot product over `in` terms.

```
multiply-accumulates (MACs) = in * out
FLOPs                       = 2 * in * out        <- each MAC is 1 multiply + 1 add
params                      = in * out
FLOPs per param             = 2
```

### Backward: 4 FLOPs per parameter per token

The backward pass computes **two** different gradients, and they're both matmuls of the same size:

```
1. dL/dx = dL/dy @ W^T      needed to keep propagating backward   -> 2 * in * out FLOPs
2. dL/dW = x^T @ dL/dy      needed to update this layer's weights -> 2 * in * out FLOPs
                                                                   ----------------------
                                                            total   4 * in * out FLOPs
```

**That's why backward is 2x forward** — it's not a fudge factor, it's literally two matmuls instead of one.

### Total: 6

```
forward 2  +  backward 4  =  6 FLOPs per parameter per token
```

Sum over every parameter in the model and you get `6N` per token. That's the rule. (Chinchilla's compute budget is `6 * N * D` for `N` params and `D` tokens — this is where that comes from.)

### The attention term

`6N` counts the parameters. But attention's `q @ k^T` and `att @ v` involve **no parameters at all** — they're matmuls between activations. So they need their own term:

```
flops_per_token = 6*N  +  12 * L * nh * hs * T
                  ^^^^     ^^^^^^^^^^^^^^^^^^^
                  params   attention's quadratic part
```

### A100 bf16 peak is 312 TFLOPS

| MFU | verdict |
|---|---|
| < 10% | something is broken. CPU-bound dataloader, tiny batch, or a sync in the loop. |
| 15–25% | typical for a small model. `B*T` too small to saturate the matmuls. |
| 35–45% | good. What nanoGPT gets on GPT-2 124M on an A100. |
| 50–60% | excellent, large-model territory |
| > 70% | you are probably counting wrong |

Our 30M model will land ~15–25%: at `C=384` the matmuls are simply too skinny to fill an A100's tensor cores, and per-kernel launch overhead is a real fraction of a 100ms step. That's not a bug — it's *why nobody trains 30M-parameter models on A100s for real work*.

## 9.9 `torch.compile`

```python
model = torch.compile(model)
```

TorchDynamo traces your Python into a graph; TorchInductor lowers it to fused Triton kernels. Typical win: **1.3–2x**, essentially free.

**Where the win comes from:** eager PyTorch launches a separate kernel for every op, and each one round-trips its tensors through HBM. Your GELU reads `(B,T,4C)` from HBM, computes one cheap nonlinearity, writes it back. That op is 100% memory-bound and ~0% compute. Inductor fuses chains of these into one kernel that reads once, does all the elementwise work in registers, and writes once. On the pointwise-heavy parts of a transformer that's most of the available speedup.

**Costs:** the first step takes 30–90s to compile. **Every new input shape triggers a recompile** — which is why fixed `T` matters, and why naive KV-cache decode (where `T` grows by 1 every step) is pathological for `torch.compile` unless you pre-allocate a static shape. Keep a reference to the uncompiled model (`raw_model`) for checkpointing.

In [ ]:
# --- 9.8: derive the 6N rule from ONE Linear layer ---
inn, out = 384, 1536
params = inn * out
fwd = 2 * inn * out                    # y = x @ W       : 1 multiply + 1 add per MAC
bwd_x = 2 * inn * out                  # dL/dx = dL/dy @ W^T
bwd_w = 2 * inn * out                  # dL/dW = x^T @ dL/dy

print(f"ONE Linear({inn}, {out}), for ONE token. It has {params:,} parameters.\n")
print(f"  forward   y = x @ W           : {fwd:>10,} FLOPs = {fwd/params:.0f} per param")
print(f"  backward  dL/dx = dL/dy @ W^T : {bwd_x:>10,} FLOPs = {bwd_x/params:.0f} per param")
print(f"  backward  dL/dW = x^T @ dL/dy : {bwd_w:>10,} FLOPs = {bwd_w/params:.0f} per param")
print(f"  {'-'*58}")
print(f"  TOTAL                         : {fwd+bwd_x+bwd_w:>10,} FLOPs = "
      f"{(fwd+bwd_x+bwd_w)/params:.0f} per param per token")
assert (fwd + bwd_x + bwd_w) / params == 6.0
print("\n[OK] exactly 6. Backward is 2x forward because it computes TWO gradients,")
print("     each a matmul the same size as the forward. Not a fudge factor.")
print("\nSum over every parameter -> 6N FLOPs per token. That IS the 6N rule.")
print("Chinchilla's compute budget C = 6*N*D comes from exactly this.")

In [ ]:
def estimate_mfu(model, fwdbwd_per_iter, dt, peak_flops=312e12):
    N = model.get_num_params()
    cfg = model.config
    L, nh, hs, T = cfg.n_layer, cfg.n_head, cfg.n_embd // cfg.n_head, cfg.block_size
    flops_per_token = 6 * N + 12 * L * nh * hs * T
    flops_per_iter = flops_per_token * T * fwdbwd_per_iter
    return (flops_per_iter / dt) / peak_flops

N = model.get_num_params()
cfg = REAL
L, nh, hs, T = cfg.n_layer, cfg.n_head, cfg.n_embd // cfg.n_head, cfg.block_size
param_term = 6 * N
attn_term  = 12 * L * nh * hs * T
print(f"N (non-embedding params) : {N:,}\n")
print(f"  6*N            (the parameters)   : {param_term/1e6:>8.1f} MFLOPs/token   "
      f"({param_term/(param_term+attn_term)*100:.0f}%)")
print(f"  12*L*nh*hs*T   (attention's q@k^T and att@v -- NO parameters involved)")
print(f"                                    : {attn_term/1e6:>8.1f} MFLOPs/token   "
      f"({attn_term/(param_term+attn_term)*100:.0f}%)")
print(f"  total                             : {(param_term+attn_term)/1e6:>8.1f} MFLOPs/token")

print(f"\nHow the balance shifts with context length T:")
for T_ in [256, 1024, 4096, 16384]:
    a = 12 * L * nh * hs * T_
    bar = "#" * int(a/(param_term+a)*40)
    print(f"  T = {T_:>6}: attention is {a/(param_term+a)*100:>5.1f}% of FLOPs/token  {bar}")
print("\nAt short context, attention is a rounding error and the MLPs dominate.")
print("At long context, attention takes over -- and its MEMORY took over long before")
print("its FLOPs did (Part 4.5). That gap is exactly what FlashAttention exploits.")

In [ ]:
# ============================================================================
# TRAINING SETUP
# ============================================================================
batch_size       = 64          # B
block_size       = 256         # T
grad_accum_steps = 4           # -> 64 * 256 * 4 = 65,536 tokens/step
eval_interval    = 500
eval_iters       = 50
log_interval     = 50
compile_model    = True

tokens_per_step = batch_size * block_size * grad_accum_steps
print(f"tokens per step : {tokens_per_step:,}  = B({batch_size}) x T({block_size}) "
      f"x accum({grad_accum_steps})")
print(f"max_iters       : {max_iters:,}")
print(f"total tokens    : {tokens_per_step * max_iters / 1e6:.0f} M")
print(f"TinyStories is ~373M tokens -> ~{tokens_per_step*max_iters/373e6:.2f} epochs")

ctx = (torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16)
       if device_type == 'cuda' else torch.amp.autocast(device_type='cpu', enabled=False))

model = GPT(REAL, flash=True).to(device)
optimizer = configure_optimizers(model, 0.1, learning_rate, (0.9, 0.95), device_type)

raw_model = model      # keep the UNCOMPILED handle for checkpointing
if compile_model and device_type == 'cuda':
    print("\ncompiling (30-90s, one-off)...")
    model = torch.compile(model)

In [ ]:
@torch.no_grad()
def estimate_loss():
    # no_grad + eval(): no autograd graph, no dropout. Averaged over eval_iters batches
    # because a single batch's loss is far too noisy to make decisions on.
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split, batch_size, block_size)
            with ctx:
                _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out
print("estimate_loss ready")

In [ ]:
# ============================================================================
# THE LOOP.  ~25 min on an A100 for 10k iters.
# ============================================================================
best_val = 1e9
history = []
X, Y = get_batch('train', batch_size, block_size)   # prefetch the first batch
t0 = time.time()
running_mfu = -1.0
model.train()

for it in range(max_iters):

    # 1) set the LR for this step, by hand, on every param group
    lr = get_lr(it)
    for g in optimizer.param_groups:
        g['lr'] = lr

    # 2) evaluate + checkpoint
    if it % eval_interval == 0 or it == max_iters - 1:
        losses = estimate_loss()
        print(f"\n>> iter {it:>5} | train {losses['train']:.4f} | val {losses['val']:.4f} "
              f"| ppl {math.exp(losses['val']):.2f} | lr {lr:.2e}")
        history.append((it, losses['train'], losses['val']))
        if losses['val'] < best_val:
            best_val = losses['val']
            torch.save({'model': raw_model.state_dict(),
                        'optimizer': optimizer.state_dict(),
                        'config': REAL, 'iter': it, 'val_loss': best_val}, 'ckpt.pt')

    # 3) forward/backward, grad_accum_steps micro-batches
    for micro in range(grad_accum_steps):
        with ctx:
            logits, loss = model(X, Y)
            # CRITICAL: CE already returned a MEAN. Summing N means = N x the gradient.
            loss = loss / grad_accum_steps
        # prefetch the next batch while the GPU is busy with backward (async H2D)
        X, Y = get_batch('train', batch_size, block_size)
        loss.backward()          # ACCUMULATES into .grad -- does not overwrite

    # 4) clip the GLOBAL grad norm. Insurance against one pathological batch.
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

    # 5) step, then clear. set_to_none=True frees the buffers (faster than zeroing).
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)

    # 6) log
    t1 = time.time(); dt = t1 - t0; t0 = t1
    if it % log_interval == 0:
        lossf = loss.item() * grad_accum_steps     # undo the scaling for display
        if it >= 5:
            mfu = estimate_mfu(raw_model, batch_size * grad_accum_steps, dt)
            running_mfu = mfu if running_mfu < 0 else 0.9 * running_mfu + 0.1 * mfu
        print(f"iter {it:>5} | loss {lossf:.4f} | lr {lr:.2e} | {dt*1000:>6.1f} ms "
              f"| mfu {running_mfu*100:>5.2f}% | gnorm {grad_norm:.3f} "
              f"| {tokens_per_step/dt/1e3:>6.0f} Ktok/s")

print(f"\ndone. best val loss {best_val:.4f} (ppl {math.exp(best_val):.2f})")

In [ ]:
if history:
    its_, tr, va = zip(*history)
    plt.figure(figsize=(7, 4))
    plt.plot(its_, tr, label='train', marker='o', ms=3)
    plt.plot(its_, va, label='val', marker='s', ms=3)
    plt.axhline(math.log(REAL.vocab_size), color='gray', ls='--', lw=1,
                label=f'ln(V) = {math.log(REAL.vocab_size):.2f} (random init)')
    plt.xlabel('iteration'); plt.ylabel('loss'); plt.legend(); plt.grid(alpha=0.3)
    plt.title('TinyStories, 30M params'); plt.tight_layout(); plt.show()
    print(f"Started at ln(V) = {math.log(REAL.vocab_size):.2f} -- exactly as Part 8.5 predicted.")
    print(f"Best val = {min(va):.3f} -> perplexity {math.exp(min(va)):.2f}.")
    print("Anything under ~1.5 val loss on TinyStories generates fluent English.")

---
# Part 10 — Inference

## 10.1 The generation loop, as shapes

```
idx                          (B, T)
  |
  |  forward
  v
logits                       (B, T, V)      but we only want the LAST position
  |
  |  logits[:, -1, :]
  v
logits                       (B, V)
  |
  |  / temperature  ->  top-k / top-p  ->  softmax(dim=-1)
  v
probs                        (B, V)
  |
  |  torch.multinomial(probs, num_samples=1)
  v
idx_next                     (B, 1)
  |
  |  torch.cat((idx, idx_next), dim=1)
  v
idx                          (B, T+1)       <- and around again
```

The sequence grows by one token per iteration. **`T` is a moving number** — which is why every shape in this notebook was written in terms of `T` rather than a constant.

### `x[:, [-1], :]` vs `x[:, -1, :]` — the indexing detail

In `GPT.forward`, when `targets is None`:

```python
logits = self.lm_head(x[:, [-1], :])     # (B, 1, C) -> (B, 1, V)     KEEPS the axis
```

- **`x[:, -1, :]`** — an *integer* index. The axis is **consumed**: `(B, T, C) -> (B, C)`.
- **`x[:, [-1], :]`** — a *list* index. The axis is **kept**: `(B, T, C) -> (B, 1, C)`.

Keeping it means downstream code sees a `(B, T, V)`-shaped tensor exactly as in training, just with `T=1`. Same code path, no special-casing. (`x[:, -1:, :]` — a slice — does the same and is arguably clearer.)

### The saving is not small

The LM head is `(B*T, C) @ (C, V)`. At `T=256, V=50304, C=384`, computing all positions is **256x the FLOPs** of computing one — and we discard 255/256 of the result.

Recall from Part 7.2 that this is already the single most expensive matmul in the forward pass. At generation time it would be spent entirely on tokens whose predictions we already made and already threw away. **One indexing expression removes it.**

### `torch.multinomial`

```python
idx_next = torch.multinomial(probs, num_samples=1)      # (B, V) -> (B, 1)
```

Samples one index per row, with probability proportional to that row's entries. Three things worth knowing:
- it **normalises internally** — the rows need not sum to 1 (handy after a top-k mask)
- it operates **row-wise** on 2-D input: each batch row is sampled independently
- it returns **indices**, not values

Contrast `torch.argmax(probs, -1)`, which returns `(B,)` and is deterministic. That's greedy decoding — i.e. the `temperature -> 0` limit.

## 10.2 Sampling: temperature, top-k, top-p

### Temperature — hand calculation

Divide the logits by `T`, **before** the softmax.

```
logits = [2.0, 1.0, 0.5, 0.1]

temp = 1.0 :  softmax([2.0, 1.0, 0.5, 0.1])       = [0.5745, 0.2114, 0.1282, 0.0859]
temp = 0.5 :  softmax([4.0, 2.0, 1.0, 0.2])       = [0.8282, 0.1121, 0.0412, 0.0185]   sharper
temp = 2.0 :  softmax([1.0, 0.5, 0.25, 0.05])     = [0.4056, 0.2460, 0.1916, 0.1569]   flatter
```

**Why it works:** softmax's output depends only on logit **differences** (add a constant to every logit and nothing changes — that's exactly the `-m` shift from Part 8.4). Dividing by `T` scales every difference by `1/T`. So temperature is a knob on **how much the model's relative preferences matter**.

The limits, exactly:
- **`temp -> 0`**: dividing by ~0 sends the largest logit to `+inf` relative to the rest. Softmax → one-hot. **This is argmax.** (Don't literally pass `0` — you'll divide by zero. Branch to `argmax`.)
- **`temp -> inf`**: all logits → 0, softmax → uniform over `V`. Pure noise.

### Top-k

Keep the `k` highest logits, set the rest to `-inf`, then softmax. `k=200` from a 50k vocab is typical.

**Why truncate at all?** The tail of a language model's distribution is ~50,000 tokens each with tiny probability — but they **sum to a non-trivial mass**. Sample enough times and you *will* draw one. And because the model is autoregressive, it must then condition on its own mistake, which pushes it off-distribution, which makes the next mistake likelier. Truncation is a guard against that compounding failure.

The mechanism:

```python
v, _ = torch.topk(logits, k)                        # (B, V) -> v is (B, k), SORTED descending
logits[logits < v[:, [-1]]] = -float('inf')         # v[:, [-1]] is (B, 1) = the k-th value
```

`v[:, [-1]]` keeps the axis → `(B, 1)`, which broadcasts against `(B, V)` to give a **per-row threshold** (Part 2.9's rules). `v[:, -1]` would give `(B,)`, which right-aligns as `B vs V` — an error, or if `B == V`, something silently insane.

### Top-p (nucleus)

**Top-k's flaw:** `k` is fixed, but the distribution's sharpness isn't. After `"The capital of France is"` the model is (correctly) ~99% sure — `k=200` drags in 199 tokens of noise. After `"He opened the door and saw a"` the true distribution is genuinely broad — `k=200` may truncate good candidates.

**Top-p adapts:** keep the smallest set of tokens whose **cumulative** probability exceeds `p`.

```
sorted probs : [0.5, 0.2, 0.15, 0.1, 0.05]
cumsum       : [0.5, 0.7, 0.85, 0.95, 1.0]
p = 0.9      : keep until cumsum > 0.9  ->  keep the first 4  (nucleus size 4)

sharp: [0.97, 0.01, ...]  ->  cumsum[0] = 0.97 > 0.9  ->  nucleus size 1
```

Same `p`, different nucleus size, decided by the distribution itself. Implementing it needs `sort`, `cumsum`, and — the interesting part — `scatter` to undo the sort.

In [ ]:
logits = torch.tensor([2.0, 1.0, 0.5, 0.1])
print(f"logits: {logits.tolist()}\n")
print(f"{'temp':>7}  {'logits/temp':<28} {'softmax':<42} effect")
print("-" * 108)
for t in [0.01, 0.5, 0.8, 1.0, 2.0, 100.0]:
    scaled = logits / t
    p = F.softmax(scaled, dim=-1)
    eff = ("-> argmax (one-hot)" if t < 0.1 else
           "-> uniform (pure noise)" if t > 10 else
           "sharper" if t < 1 else "the model's own distribution" if t == 1 else "flatter")
    print(f"{t:>7.2f}  {str([round(v,2) for v in scaled.tolist()]):<28} "
          f"{str([round(v,4) for v in p.tolist()]):<42} {eff}")

print(f"\nargmax(logits)                : {logits.argmax().item()}")
print(f"softmax(logits/0.01).argmax() : {F.softmax(logits/0.01, -1).argmax().item()}   <- the same thing")
print("\nWhy it works: softmax depends only on logit DIFFERENCES.")
print("(Adding a constant to every logit changes nothing -- that IS the -m shift")
print(" from Part 8.4's log-sum-exp derivation.) Dividing by T scales every")
print("difference by 1/T. Temperature = how much the model's preferences matter.")

In [ ]:
# --- top-k ---
torch.manual_seed(0)
V_ = 10
logits = torch.randn(2, V_) * 2                     # (B, V)
k = 3
v, ix = torch.topk(logits, k)                       # v: (B, k) SORTED DESC, ix: (B, k)
sh("logits", logits); sh("topk values", v); sh("topk indices", ix)
print(f"\nrow 0 logits      : {[round(x,2) for x in logits[0].tolist()]}")
print(f"row 0 top-{k} values: {[round(x,2) for x in v[0].tolist()]}  <- already sorted descending")
print(f"row 0 top-{k} idx   : {ix[0].tolist()}")

thresh = v[:, [-1]]                                 # (B, 1) -- the k-th largest, axis KEPT
sh("\nv[:, [-1]] (axis KEPT)", thresh)
sh("v[:, -1]   (axis GONE)", v[:, -1])
print("  (B,1) broadcasts against (B,V) -> a per-row threshold. Correct.")
print("  (B,)  right-aligns B against V -> error, or silently insane if B == V.")

masked = logits.clone()
masked[masked < thresh] = -float('inf')             # broadcast (B,V) < (B,1)
probs = F.softmax(masked, dim=-1)
print(f"\nrow 0 after mask  : {[('-inf' if math.isinf(x) else round(x,2)) for x in masked[0].tolist()]}")
print(f"row 0 probs       : {[round(x,3) for x in probs[0].tolist()]}")
print(f"nonzero entries   : {(probs[0] > 0).sum().item()} = k.  Sums to {probs[0].sum().item():.4f}")

nxt = torch.multinomial(probs, num_samples=1)       # (B, V) -> (B, 1)
sh("\nmultinomial(probs, 1)", nxt)
print(f"sampled: {nxt.flatten().tolist()}   <- INDICES, not values. Rows sampled independently.")

### `torch.gather` and `torch.scatter`: undoing a sort

Top-p sorts the probabilities, decides what to keep **in sorted order**, then must map that decision back to the original vocab positions. That round trip is `gather`/`scatter`.

You already met these in Part 2.7 — the forward pass of an embedding is a gather, the backward is a scatter_add. Same operations, different job.

**`gather(dim, index)` — "pull values FROM these positions":**

```python
out[i][j] = src[i][ index[i][j] ]      # for dim=1
```
`index` and `out` have the same shape. Every axis except `dim` indexes normally; only `dim` gets redirected.

**`scatter(dim, index, src)` — "push values TO these positions":** the exact inverse.

```python
out[i][ index[i][j] ] = src[i][j]      # for dim=1
```

**The relationship worth memorising:**

```python
sorted_p, sorted_idx = probs.sort(descending=True)
probs.gather(1, sorted_idx) == sorted_p                             # gather reproduces the sort
torch.zeros_like(probs).scatter(1, sorted_idx, sorted_p) == probs   # scatter undoes it
```

`sorted_idx[i][j]` answers: *"which original column ended up at sorted position `j`?"* `gather` reads along it; `scatter` writes along it.

In top-p we build the removal mask in *sorted* space and `scatter` it back to *vocab* space — one line that would otherwise be a loop.

**One caveat** (visible in the cell below): `scatter` only cleanly inverts `gather` when the index is a **permutation** — no duplicates. Duplicate indices mean some positions get written twice and others never. `sort()` always returns a permutation, so we're safe here. (And where duplicates *are* expected — like Part 2.6's embedding gradient — you want `scatter_add`, which sums them instead of clobbering.)

In [ ]:
src = torch.tensor([[10., 20., 30., 40.],
                    [50., 60., 70., 80.]])
idx = torch.tensor([[3, 0, 1, 2],
                    [1, 1, 3, 0]])
print("src:\n", src)
print("index:\n", idx)
print("\ngather(1, index):  out[i][j] = src[i][ index[i][j] ]")
print(src.gather(1, idx))
print("  row 0: index [3,0,1,2] -> [src[0][3], src[0][0], src[0][1], src[0][2]] = [40,10,20,30]")

print("\nscatter(1, index, src): out[i][ index[i][j] ] = src[i][j]   -- the inverse")
print(torch.zeros_like(src).scatter(1, idx, src))
print("  row 1's index is [1,1,3,0] -- column 1 is written TWICE and column 2 NEVER,")
print("  so column 2 stays 0 and column 1 keeps the last write.")
print("  scatter only inverts gather cleanly when the index is a PERMUTATION.")
print("  sort() always returns one, so top-p is safe.")

# --- the round trip that top-p depends on ---
probs = torch.tensor([[0.1, 0.5, 0.2, 0.2],
                      [0.4, 0.1, 0.4, 0.1]])
sp, si = probs.sort(dim=-1, descending=True)
print(f"\nprobs      :\n{probs}")
print(f"sorted     :\n{sp}")
print(f"sorted_idx :\n{si}    <- 'which original column is at sorted position j'")
assert torch.allclose(probs.gather(1, si), sp)
print("\n[OK] probs.gather(1, sorted_idx) == sorted_probs        (gather reproduces the sort)")
recovered = torch.zeros_like(probs).scatter(1, si, sp)
assert torch.allclose(recovered, probs)
print("[OK] zeros.scatter(1, sorted_idx, sorted_probs) == probs  (scatter undoes it)")
print("\nThat is how top-p decides in SORTED space and applies in VOCAB space.")

In [ ]:
def top_p_filter(logits, p):
    # logits: (B, V) -> (B, V) with everything outside the nucleus set to -inf
    sorted_logits, sorted_idx = torch.sort(logits, descending=True, dim=-1)   # (B, V)
    cumprobs = F.softmax(sorted_logits, dim=-1).cumsum(dim=-1)                # (B, V)

    # remove everything AFTER the cumulative mass first exceeds p
    remove = cumprobs > p                                                     # (B, V) bool
    # shift right by one so the token that TIPS the sum past p is itself kept
    remove[..., 1:] = remove[..., :-1].clone()
    remove[..., 0] = False                                                    # always keep top-1

    # the mask lives in SORTED space -- scatter it back to VOCAB space
    remove_in_vocab_space = remove.scatter(1, sorted_idx, remove)             # (B, V)
    return logits.masked_fill(remove_in_vocab_space, -float('inf'))


demo = torch.log(torch.tensor([[0.5, 0.2, 0.15, 0.1, 0.05]]))    # a BROAD distribution
out = top_p_filter(demo.clone(), p=0.9)
print(f"probs        : {[round(v,3) for v in F.softmax(demo,-1)[0].tolist()]}")
print(f"cumsum       : {[round(v,3) for v in F.softmax(demo,-1).cumsum(-1)[0].tolist()]}")
print(f"kept (p=0.9) : {[('-' if math.isinf(v) else 'KEEP') for v in out[0].tolist()]}")
print(f"nucleus size : {(~torch.isinf(out[0])).sum().item()}")

print("\n--- top-p adapts; top-k cannot ---")
sharp = torch.log(torch.tensor([[0.97, 0.01, 0.008, 0.007, 0.005]]))
o = top_p_filter(sharp.clone(), p=0.9)
print(f"  SHARP  [0.97, 0.01, ...]  p=0.9 -> nucleus size {(~torch.isinf(o[0])).sum().item()}")
print(f"  BROAD  [0.5, 0.2, ...]    p=0.9 -> nucleus size {(~torch.isinf(out[0])).sum().item()}")
print("\nSame p. Different nucleus, decided by the distribution itself.")
print("A fixed k could not do this -- which is the entire point of top-p.")

In [ ]:
@torch.no_grad()
def generate(model, idx, max_new_tokens, temperature=1.0, top_k=None, top_p=None):
    # idx: (B, T) -> (B, T + max_new_tokens).  NO cache: recomputes everything, every step.
    for _ in range(max_new_tokens):
        # crop to block_size: wpe only has block_size rows
        idx_cond = idx if idx.size(1) <= model.config.block_size \
                       else idx[:, -model.config.block_size:]
        logits, _ = model(idx_cond)                  # (B, 1, V) -- last position only
        logits = logits[:, -1, :] / temperature      # (B, 1, V) -> (B, V)

        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))   # (B, k)
            logits[logits < v[:, [-1]]] = -float('inf')              # broadcast (B,V)<(B,1)
        if top_p is not None:
            logits = top_p_filter(logits, top_p)

        probs = F.softmax(logits, dim=-1)                            # (B, V)
        idx_next = torch.multinomial(probs, num_samples=1)           # (B, 1)
        idx = torch.cat((idx, idx_next), dim=1)                      # (B, T) -> (B, T+1)
    return idx

# Load the trained model if we have one, else use the freshly initialised one
if os.path.exists('ckpt.pt'):
    ck = torch.load('ckpt.pt', map_location=device, weights_only=False)
    gen_model = GPT(ck['config'], flash=True).to(device)
    gen_model.load_state_dict(ck['model'])
    print(f"loaded ckpt: iter {ck['iter']}, val loss {ck['val_loss']:.4f}")
else:
    gen_model = GPT(REAL, flash=True).to(device)
    print("no checkpoint -- using a RANDOM model (output will be noise; shapes still work)")
gen_model.eval()

start = "Once upon a time"
idx = torch.tensor(enc.encode(start), dtype=torch.long, device=device)[None, ...]  # (1, T)
sh("prompt idx", idx)
out = generate(gen_model, idx, max_new_tokens=100, temperature=0.8, top_k=200)
sh("after 100 tokens", out)
print(f"\n{enc.decode(out[0].tolist())}")

## 10.3 The KV cache

### The waste, precisely

Look at what `generate` above actually does. To produce token `t+1`, it runs the **whole model over all `t` tokens** — recomputing `k` and `v` for positions `0..t-1` that **have not changed since the step that first computed them**.

Why haven't they changed? Because position 5's key vector is a function of **position 5's input only** (that's Part 3.4's per-token rule again). It was correct at step 5. It is still correct at step 500.

So: cache them.

### The complexity argument, made concrete

Per decode step, the *projection* work (the `c_attn`, `c_proj`, `c_fc` matmuls — where all the parameters are) is:

```
without cache:  step t re-projects all t tokens   ->  O(t) work per step
with cache:     step t projects only the new one  ->  O(1) work per step
```

Sum over a generation of `T` tokens:

```
without cache:  1 + 2 + 3 + ... + T  =  T(T+1)/2  =  O(T^2)
with cache:     1 + 1 + 1 + ... + 1  =  T         =  O(T)
```

That's the `O(T²) -> O(T)` claim, and the cell below measures the quadratic curve directly.

*(The attention `q@k^T` itself is still `O(t)` per step either way, giving `O(T²)` overall. The cache removes the redundant **projections** and the redundant **MLP** work — which is where 100% of the parameters are.)*

### The shapes — this is the part to internalise

**Prefill** (process the whole prompt, `T_p` tokens at once):

```
x                              (B, T_p, C)
k, v                           (B, nh, T_p, hs)      <- store both
attention: q (B,nh,T_p,hs) @ k^T (B,nh,hs,T_p)  ->  (B, nh, T_p, T_p)     causal mask NEEDED
```

**Decode** (one token at a time):

```
x                              (B, 1, C)             <- T is now 1
q, k_new, v_new                (B, nh, 1, hs)        <- ONE position
k = cat([k_cache, k_new], dim=2)                     (B, nh, t+1, hs)     grows along the T axis
v = cat([v_cache, v_new], dim=2)                     (B, nh, t+1, hs)

attention: q (B,nh,1,hs) @ k^T (B,nh,hs,t+1)  ->  (B, nh, 1, t+1)
softmax over the last axis, then @ v (B,nh,t+1,hs)  ->  (B, nh, 1, hs)
```

Three things fall out of that trace:

**1. The `(T, T)` matrix collapses to `(1, t+1)`.** One query row against `t+1` keys. The quadratic term is gone from the *per-step* cost.

**2. No causal mask during decode.** The single query **is** the newest position; everything in the cache is by construction in its past. There is nothing to mask.

This is why `is_causal=True` is **wrong** for decode. SDPA aligns the causal diagonal at the **top-left** of a `(1, t+1)` matrix, which would mask off all but the very first key. Pass `is_causal=False`. This is a real and popular bug: it produces a model that only ever attends to token 0, and **it does not raise.**

**3. `cat(..., dim=2)` grows the `T` axis** — axis 2 in `(B, nh, T, hs)`. Note that `cat` reallocates and copies the whole cache every step, which is `O(t)` memory traffic per token, i.e. `O(T²)` copying over a generation. Real serving stacks pre-allocate to `max_seq_len` and write into a slice. vLLM's **PagedAttention** goes further: fixed-size blocks with an indirection table, so there's no contiguity requirement and no copy ever. We use `cat` here because it makes the shape story legible.

### The memory bill

```
kv_bytes = 2 (k and v) * B * T * C * L * bytes_per_element
```

Note what's **not** in that formula: it's **linear in `T`**, not quadratic — the cache stores keys and values, not the attention matrix. And it's **per sequence**: serve 64 concurrent users and you pay 64x.

In [ ]:
# --- the O(T^2) vs O(T) claim, counted directly ---
print("Total token-projections performed over a generation of T tokens:\n")
print(f"{'T':>6} {'no cache: 1+2+...+T':>22} {'with cache: T':>16} {'ratio':>8}")
print("-" * 56)
for T_gen in [16, 64, 256, 1024, 4096]:
    nocache = T_gen * (T_gen + 1) // 2      # sum 1..T
    cached = T_gen
    print(f"{T_gen:>6} {nocache:>22,} {cached:>16,} {nocache/cached:>7.0f}x")
print("\nwithout cache: step t re-projects all t tokens -> sum(1..T) = T(T+1)/2 = O(T^2)")
print("with cache   : step t projects only the new one -> T             = O(T)")
print("\nThe ratio grows LINEARLY with T. At T=4096 the uncached version does")
print("2048x the projection work for identical output.")

In [ ]:
def kv_cache_mb(B, T, C, L, bytes_per=2):
    return 2 * B * T * C * L * bytes_per / 1e6    # 2 = one for k, one for v

print("KV cache size (bf16):   2 * B * T * C * L * 2 bytes\n")
print(f"{'model':<22}{'L':>4}{'C':>7}{'T':>7}{'B':>5}{'cache':>12}")
print("-" * 60)
rows = [("our TinyStories GPT",  6,  384,  256,  1),
        ("our GPT, 64 users",    6,  384,  256, 64),
        ("GPT-2 XL (1.5B)",     48, 1600, 1024,  1),
        ("GPT-2 XL, 64 users",  48, 1600, 1024, 64),
        ("Llama-70B (MHA)",     80, 8192, 4096,  1),
        ("Llama-70B (GQA 8kv)", 80, 1024, 4096,  1)]
for name, L, C, T, B in rows:
    mb = kv_cache_mb(B, T, C, L)
    s = f"{mb:>9.1f} MB" if mb < 1000 else f"{mb/1000:>9.2f} GB"
    print(f"{name:<22}{L:>4}{C:>7}{T:>7}{B:>5}{s:>12}")

print("\nThe last two rows are the entire argument for GQA:")
print("  Llama-70B with 64 query heads and 64 KV heads : 10.7 GB of cache PER SEQUENCE")
print("  With 8 KV heads (each shared by 8 query heads):  1.34 GB. 8x less.")
print("\nAnd recall Part 4.5: GQA is implemented by NOTHING MORE than letting the")
print("head axis broadcast in the q @ k^T batched matmul. No clever algorithm.")
print("\nNote the formula is LINEAR in T, not quadratic -- the cache stores keys and")
print("values, not the attention matrix. But it is PER SEQUENCE: 64 users = 64x.")

In [ ]:
class CausalSelfAttentionKV(nn.Module):
    # Same module as Part 4.8, plus an optional (k, v) cache.

    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.n_head, self.n_embd = config.n_head, config.n_embd
        self.dropout = config.dropout
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)

    def forward(self, x, kv_cache=None):
        B, T, C = x.size()
        nh, hs = self.n_head, C // self.n_head

        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)      # 3 x (B, T, C)
        q = q.view(B, T, nh, hs).transpose(1, 2)                # (B, nh, T, hs)
        k = k.view(B, T, nh, hs).transpose(1, 2)                # (B, nh, T, hs)
        v = v.view(B, T, nh, hs).transpose(1, 2)                # (B, nh, T, hs)

        if kv_cache is not None:
            past_k, past_v = kv_cache                           # (B, nh, T_past, hs)
            k = torch.cat((past_k, k), dim=2)                   # (B, nh, T_past+T, hs)
            v = torch.cat((past_v, v), dim=2)                   # grow along the T axis
        new_cache = (k, v)

        # is_causal ONLY when q and k have the same length (prefill / training).
        # During decode q has length 1 and IS the newest position: everything in the
        # cache is already in its past, so there is NOTHING to mask.
        # Passing is_causal=True here would align the diagonal at the top-left of a
        # (1, t+1) matrix and mask off every key but the first. Silently. No error.
        is_causal = (q.size(2) == k.size(2)) and q.size(2) > 1

        y = F.scaled_dot_product_attention(q, k, v, dropout_p=0.0, is_causal=is_causal)
        y = y.transpose(1, 2).contiguous().view(B, T, C)        # (B, T, C)
        return self.c_proj(y), new_cache


class BlockKV(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = LayerNorm(config.n_embd, bias=config.bias)
        self.attn = CausalSelfAttentionKV(config)
        self.ln_2 = LayerNorm(config.n_embd, bias=config.bias)
        self.mlp  = MLP(config)

    def forward(self, x, kv_cache=None):
        attn_out, new_cache = self.attn(self.ln_1(x), kv_cache)
        x = x + attn_out
        x = x + self.mlp(self.ln_2(x))
        return x, new_cache

print("KV-cached attention and block defined.")

In [ ]:
class GPTKV(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte  = nn.Embedding(config.vocab_size, config.n_embd),
            wpe  = nn.Embedding(config.block_size, config.n_embd),
            drop = nn.Dropout(config.dropout),
            h    = nn.ModuleList([BlockKV(config) for _ in range(config.n_layer)]),
            ln_f = LayerNorm(config.n_embd, bias=config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight

    def forward(self, idx, kv_caches=None):
        B, T = idx.size()
        # THE POSITION OFFSET. During decode idx is (B, 1) but its TRUE position is
        # T_past, not 0. Get this wrong and every generated token is told it is token 0.
        t_past = 0 if kv_caches is None else kv_caches[0][0].size(2)
        pos = torch.arange(t_past, t_past + T, dtype=torch.long, device=idx.device)  # (T,)

        x = self.transformer.drop(self.transformer.wte(idx) + self.transformer.wpe(pos))
        new_caches = []
        for i, block in enumerate(self.transformer.h):
            cache = None if kv_caches is None else kv_caches[i]
            x, nc = block(x, cache)                     # (B, T, C)
            new_caches.append(nc)
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x[:, [-1], :])            # (B, 1, C) -> (B, 1, V)
        return logits, new_caches


@torch.no_grad()
def generate_cached(model, idx, max_new_tokens, temperature=1.0, top_k=None, greedy=False):
    # PREFILL: the whole prompt in ONE pass, caches populated
    logits, caches = model(idx)                          # (B, T_p) -> (B, 1, V)

    for _ in range(max_new_tokens):
        logits = logits[:, -1, :] / max(temperature, 1e-8)          # (B, V)
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float('inf')
        if greedy:
            idx_next = logits.argmax(dim=-1, keepdim=True)          # (B, 1)
        else:
            idx_next = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)
        idx = torch.cat((idx, idx_next), dim=1)                     # (B, T+1)

        # DECODE: feed ONLY the new token. This is the entire payoff.
        logits, caches = model(idx_next, caches)                    # (B, 1) -> (B, 1, V)
    return idx

print("GPTKV + generate_cached defined.")

In [ ]:
# Port the trained weights into the KV model (identical parameters, different plumbing)
cfg_gen = ck['config'] if os.path.exists('ckpt.pt') else REAL
kv_model = GPTKV(cfg_gen).to(device)
missing, unexpected = kv_model.load_state_dict(gen_model.state_dict(), strict=False)
kv_model.eval()
print(f"loaded. unexpected keys (the non-cached model's mask buffers): {len(unexpected)}")

# --- watch the cache grow, one decode step at a time ---
idx = torch.tensor(enc.encode("Once upon a time"), dtype=torch.long, device=device)[None, ...]
print(f"\nPREFILL: idx {tuple(idx.shape)}  -- the whole prompt in one pass")
with torch.no_grad():
    logits, caches = kv_model(idx)
print(f"  logits           {tuple(logits.shape)}   <- (B, 1, V): ONLY the last position")
print(f"  caches           {len(caches)} layers, each a (k, v) tuple")
print(f"  cache[0] k       {tuple(caches[0][0].shape)}   <- (B, nh, T_prompt, hs)")

print(f"\nDECODE, 3 steps -- watch axis 2 grow by exactly 1 each time:")
with torch.no_grad():
    for step in range(3):
        nxt = logits[:, -1, :].argmax(-1, keepdim=True)              # (B, 1)
        logits, caches = kv_model(nxt, caches)
        print(f"  step {step}: input {tuple(nxt.shape)} -> logits {tuple(logits.shape)} "
              f"| cache k {tuple(caches[0][0].shape)}")
print("\n  The input is (B, 1) FOREVER. The cache's T axis is the only thing that grows.")
print("  There is no (T, T) attention matrix during decode -- it is (1, t+1).")

In [ ]:
# ============================================================================
# CORRECTNESS FIRST. Greedy decoding must give the SAME tokens with and without
# the cache. If the cache were wrong (position offset, mask polarity, concat axis),
# the sequences would diverge -- usually a few tokens in, and NEVER with an exception.
# ============================================================================
prompt = torch.tensor(enc.encode("Once upon a time"), dtype=torch.long, device=device)[None, ...]
N_NEW = 40

@torch.no_grad()
def generate_greedy_nocache(model, idx, n):
    for _ in range(n):
        cond = idx[:, -model.config.block_size:]
        logits, _ = model(cond)
        idx = torch.cat((idx, logits[:, -1, :].argmax(-1, keepdim=True)), dim=1)
    return idx

out_slow = generate_greedy_nocache(gen_model, prompt.clone(), N_NEW)
out_fast = generate_cached(kv_model, prompt.clone(), N_NEW, greedy=True)

print(f"no cache : {tuple(out_slow.shape)}")
print(f"cache    : {tuple(out_fast.shape)}")
match = torch.equal(out_slow, out_fast)
print(f"\nidentical tokens: {match}")
if not match:
    d = (out_slow != out_fast).nonzero()
    print(f"first divergence at position {d[0,1].item()}")
assert match, "KV cache is WRONG -- check the position offset and is_causal"
print("\n[OK] BIT-IDENTICAL greedy output.")
print("     The cache is a pure optimisation, not an approximation.")
print(f"\n{enc.decode(out_fast[0].tolist())}")

In [ ]:
# ============================================================================
# And now the speed.
# ============================================================================
def bench(fn, *a, **kw):
    for _ in range(2): fn(*a, **kw)                       # warmup
    if device_type == 'cuda': torch.cuda.synchronize()
    t0 = time.time()
    r = fn(*a, **kw)
    if device_type == 'cuda': torch.cuda.synchronize()
    return time.time() - t0, r

print(f"{'new tokens':>11} {'no cache':>11} {'KV cache':>11} {'speedup':>9} {'tok/s (cached)':>16}")
print("-" * 64)
for n in [32, 64, 128, 256]:
    if prompt.size(1) + n > cfg_gen.block_size: break
    t_slow, _ = bench(generate_greedy_nocache, gen_model, prompt.clone(), n)
    t_fast, _ = bench(generate_cached, kv_model, prompt.clone(), n, greedy=True)
    print(f"{n:>11} {t_slow*1000:>9.1f} ms {t_fast*1000:>9.1f} ms "
          f"{t_slow/t_fast:>8.2f}x {n/t_fast:>15.0f}")
print("\nThe gap WIDENS with sequence length, exactly as the O(T^2) vs O(T) count predicted.")
print("At T=1024+ the uncached version is simply unusable.")

## 10.4 Prefill vs decode: two completely different workloads

This is the insight that organises all of LLM inference, and it falls straight out of the shapes above.

|  | prefill | decode |
|---|---|---|
| tokens per forward | `T_p` (e.g. 512) | **1** |
| matmul shape | `(B*T_p, C) @ (C, ...)` — tall | `(B*1, C) @ (C, ...)` — **one row** |
| bound by | compute (tensor cores) | **memory bandwidth** |
| arithmetic intensity | high | `~B` |
| latency metric | TTFT (time to first token) | TPOT (time per output token) |

That "one row" is Part 5.2's point coming home: `nn.Linear` flattens to a `(B*T, C)` matmul, and `B*T` is what decides whether the GPU is busy. At decode, `B*T = B`.

### The arithmetic-intensity argument

First, the term. **Arithmetic intensity** = FLOPs performed per byte moved from memory. It's the single number that decides whether you're waiting on math or waiting on memory.

Decode reads **every weight in the model from HBM** to compute **one token**:

```
bytes moved  = N params * 2 bytes               (the whole model, every single step)
FLOPs done   = 2 * N * B                        (2 FLOPs/param/token -- Part 9.8's forward term,
                                                 times B tokens in the batch)

arithmetic intensity = 2*N*B / (2*N) = B        FLOPs per byte
```

**The intensity is `B`.** Not `N`, not `T` — just the batch size. The model size cancels out entirely.

Now the roofline. An A100-80GB has 312 TFLOPS bf16 and 2039 GB/s of HBM bandwidth:

```
ridge point = 312e12 FLOPs/s / 2039e9 bytes/s = 153 FLOPs per byte
```

Below the ridge, you're memory-bound. Above it, compute-bound.

**So decoding is memory-bound until `B ≈ 153`.** At `B=1` you're running an A100 at roughly `1/153` of its compute — **under 1% MFU** — and no kernel optimisation can fix that, because you aren't waiting on math. You're waiting on HBM to deliver weights you'll use exactly once.

### Everything in modern inference serving is a response to that one number

- **Continuous batching** (vLLM, TGI): the only way to raise `B` is to batch across *users*. Since sequences finish at different times, you must be able to add and evict them mid-flight. This is worth more than any kernel.
- **PagedAttention**: batching is capped by cache memory (10.3), and naive contiguous per-sequence allocation to `max_len` wastes 60–80% of it to fragmentation. Fixed-size blocks + an indirection table recover it → `B` goes up → throughput goes up.
- **Speculative decoding**: a draft model proposes `k` tokens; the target verifies all `k` **in one forward pass**. It's the only trick that raises intensity *without* more users — you spend your idle FLOPs to buy back latency. It works precisely *because* those FLOPs were idle.
- **GQA / MQA**: shrink the cache → fit more sequences → raise `B`.
- **Weight-only quantization (INT8/INT4/FP8)**: if you're bandwidth-bound on weight loads, halving the bytes per weight nearly halves decode latency. The compute was free anyway. **This is why INT4 helps decode enormously and prefill barely at all.**

Every one of those is the same move: **the machine is starved for arithmetic intensity, so raise it.** Once you see that decode's intensity is exactly `B`, the whole field reorganises itself around a single number — and you can predict which optimisations will help *before* benchmarking any of them.

In [ ]:
N = sum(p.numel() for p in kv_model.parameters())
A100_FLOPS, A100_BW = 312e12, 2039e9          # 80GB SXM: bf16 peak, HBM bandwidth
ridge = A100_FLOPS / A100_BW

print(f"Arithmetic intensity = FLOPs performed per BYTE moved from memory.\n")
print(f"During decode:")
print(f"  bytes moved = N * 2         (every weight, from HBM, every step)")
print(f"  FLOPs done  = 2 * N * B     (2 FLOPs/param/token x B tokens)")
print(f"  intensity   = 2NB / 2N = B  <- the model size CANCELS. It is just B.\n")
print(f"A100-80GB: {A100_FLOPS/1e12:.0f} TFLOPS bf16, {A100_BW/1e9:.0f} GB/s HBM")
print(f"ridge point = {A100_FLOPS/1e12:.0f}e12 / {A100_BW/1e9:.0f}e9 = {ridge:.0f} FLOPs/byte")
print(f"  below the ridge -> memory-bound.  above -> compute-bound.\n")
print(f"{'B':>6} {'intensity':>11} {'regime':<16} {'peak achievable'}")
print("-" * 62)
for B in [1, 8, 32, 64, 128, 153, 256, 512]:
    frac = min(B / ridge, 1.0)
    regime = "memory-bound" if B < ridge else "compute-bound"
    print(f"{B:>6} {B:>8} F/B  {regime:<16} {frac*100:>5.1f}%  {'#' * int(frac*22)}")

print(f"\nAt B=1 an A100 runs at ~{1/ridge*100:.1f}% of peak compute. Not a kernel problem:")
print(f"you are waiting on HBM to ship {N*2/1e6:.0f} MB of weights to produce ONE token.")
print("\nThis single number explains continuous batching, PagedAttention, speculative")
print("decoding, GQA and INT4. Every one of them raises arithmetic intensity.")

if device_type == 'cuda':
    print(f"\n--- prefill vs decode, same model, measured ---")
    with torch.no_grad():
        long_prompt = torch.randint(0, 50000, (1, 256), device=device)
        t_pre, (lg, cs) = bench(kv_model, long_prompt)
        one = torch.randint(0, 50000, (1, 1), device=device)
        t_dec, _ = bench(kv_model, one, cs)
    print(f"  prefill 256 tokens : {t_pre*1000:>7.2f} ms  -> {256/t_pre:>8.0f} tok/s")
    print(f"  decode  1 token    : {t_dec*1000:>7.2f} ms  -> {1/t_dec:>8.0f} tok/s")
    print(f"  prefill is {(256/t_pre)/(1/t_dec):.0f}x more token-efficient.")
    print(f"  Same weights. Same FLOPs per token. The ONLY difference is how many")
    print(f"  rows the matmul has -- which is Part 5.2's B*T point, exactly.")

---
# Part 11 — Cheat sheet

## The complete shape trace, training

```
idx                                        (B, T)          int64
targets                                    (B, T)          int64

wte(idx)                                   (B, T, C)
wpe(arange(T))                                (T, C)
x = tok + pos            [broadcast]       (B, T, C)   <-- residual stream begins

  for each of L blocks:
    ln_1(x)                                (B, T, C)
      c_attn                               (B, T, 3C)
      split(C, dim=2)                3 x   (B, T, C)
      view(B,T,nh,hs).transpose(1,2) 3 x   (B, nh, T, hs)
      q @ k.transpose(-2,-1)/sqrt(hs)      (B, nh, T, T)   <-- the only T^2 tensor
      masked_fill(tril==0, -inf)           (B, nh, T, T)
      softmax(dim=-1)                      (B, nh, T, T)
      @ v                                  (B, nh, T, hs)
      transpose(1,2).contiguous().view     (B, T, C)
      c_proj                               (B, T, C)
    x = x + attn_out                       (B, T, C)

    ln_2(x)                                (B, T, C)
      c_fc                                 (B, T, 4C)
      gelu                                 (B, T, 4C)
      c_proj                               (B, T, C)
    x = x + mlp_out                        (B, T, C)

ln_f(x)                                    (B, T, C)
lm_head(x)                                 (B, T, V)   <-- the biggest tensor, by far
logits.view(-1, V)                         (B*T, V)
targets.view(-1)                           (B*T,)
cross_entropy                              ()          scalar
```

## Inference deltas

```
TRAIN                                      DECODE (with cache)
------------------------------------------------------------------------
idx           (B, T)                       idx        (B, 1)
pos = arange(0, T)                         pos = arange(t_past, t_past+1)   <-- OFFSET
k, v          (B, nh, T, hs)               k, v = cat(cache, new) (B,nh,t+1,hs)
att           (B, nh, T, T)                att        (B, nh, 1, t+1)
causal mask   REQUIRED                     mask       NOT NEEDED (is_causal=False)
lm_head on    all T positions              lm_head on x[:, [-1], :] only
compute-bound                              MEMORY-bound (intensity = B)
```

## Formulas

| | formula | at `C=384, L=6, V=50304` |
|---|---|---|
| per block: attn | `4C²` | 590K |
| per block: MLP | `8C²` | 1.18M |
| **per block total** | **`12C²`** | 1.77M |
| all blocks | `12 L C²` | 10.6M |
| embeddings (tied) | `V*C` | 19.3M |
| **total** | | **30.0M** |
| FLOPs/token (fwd+bwd) | `6N + 12 L nh hs T` | 191M |
| KV cache bytes | `2 * B * T * C * L * 2` | 2.4MB @ B=1 |
| decode intensity | `B` | — |
| A100 ridge point | `312e12 / 2039e9` | 153 FLOPs/byte |

## PyTorch functions, by shape effect

| function | shape effect | the thing to remember |
|---|---|---|
| `nn.Embedding(V,C)(x)` | `S -> S + (C,)` | it's `one_hot(x) @ W`; grad is a scatter_add |
| `nn.Linear(i,o)(x)` | `(..., i) -> (..., o)` | **`.weight` is `(o, i)`**; computes `x @ W.T` |
| `view(...)` | metadata only | **only splits/merges ADJACENT axes**; needs contiguous |
| `reshape(...)` | same, may copy | `view` if it can, else `contiguous().view()` |
| `transpose(a,b)` | swaps 2 axes | no copy, **breaks contiguity** |
| `permute(...)` | reorders all axes | no copy, breaks contiguity |
| `contiguous()` | none | **copies** to fix layout |
| `split(n, dim)` | `k*n -> k tensors of n` | views, no copy |
| `stack(list, d)` | **new** axis | all shapes must match exactly |
| `cat(list, d)` | existing axis | all match except `dim` |
| `x[:, -1, :]` | `(B,T,C) -> (B,C)` | axis **consumed** |
| `x[:, [-1], :]` | `(B,T,C) -> (B,1,C)` | axis **kept** |
| `masked_fill(m, v)` | unchanged | fills where mask is **True** |
| `softmax(dim=-1)` | unchanged | over **keys**; needs `-inf`, not `0` |
| `topk(k)` | `(B,V) -> (B,k)` | returns `(values, indices)`, **sorted desc** |
| `gather(d, i)` | `-> i.shape` | `out[i][j] = src[i][ index[i][j] ]` |
| `scatter(d, i, s)` | `-> self.shape` | inverse of gather; needs a permutation |
| `scatter_add` | `-> self.shape` | scatter, but duplicates **sum** (the chain rule) |
| `multinomial(p, 1)` | `(B,V) -> (B,1)` | normalises internally; returns indices |
| `cross_entropy` | `(N,V),(N,) -> ()` | fused stable `log_softmax` + nll |
| `register_buffer` | — | device-tracked, saved, **not trained** |

## The bugs, and what each looks like

| symptom | cause |
|---|---|
| initial loss `~10.83 = ln(V)` | nothing. This is correct. |
| initial loss much **lower** | **label leakage.** Mask broken, or `x`/`y` misaligned. Never raises. |
| initial loss much higher | init std too large |
| loss `nan` at step 0 | LR too high, or hand-rolled `log(softmax(x))` |
| `RuntimeError: view size is not compatible` | missing `.contiguous()` after a `transpose` |
| heads see the wrong things, no error | `.view(B,nh,T,hs)` instead of `.view(B,T,nh,hs).transpose(1,2)` |
| tensor mysteriously `B` times too big | broadcast `(B,1)` against `(1,B)` -> `(B,B)` |
| model attends only to token 0 in decode | `is_causal=True` during single-token decode |
| generation degrades after the prompt | position offset not passed during decode |
| effective LR silently `N`x too high | forgot `loss / grad_accum_steps` |
| LayerNorm off by `sqrt(C/(C-1))` | `var()` defaults to `unbiased=True` |
| weight decay does nothing sensible | used `Adam(weight_decay=)` instead of `AdamW` |
| mask on CPU, tensor on GPU | plain `self.mask = ...` instead of `register_buffer` |
| trains on your padding | `ignore_index=-1` vs PyTorch's default `-100` |
| MFU under 5% | dataloader-bound, or a `.item()` sync inside the loop |

In [ ]:
# ============================================================================
# The whole model, one shape trace. Come back to this cell.
# ============================================================================
cfg = GPTConfig(block_size=256, vocab_size=50304, n_layer=6, n_head=6, n_embd=384,
                dropout=0.0, bias=False)
m = GPT(cfg, flash=False)
B, T = 4, 256
C, nh, hs, V, L = cfg.n_embd, cfg.n_head, cfg.n_embd // cfg.n_head, cfg.vocab_size, cfg.n_layer
idx = torch.randint(0, V, (B, T))
tgt = torch.randint(0, V, (B, T))
print(f"B={B}  T={T}  C={C}  nh={nh}  hs={hs}  V={V}  L={L}\n")

with torch.no_grad():
    sh("idx", idx)
    tok = m.transformer.wte(idx);                          sh("  wte(idx)", tok)
    pos = m.transformer.wpe(torch.arange(T));              sh("  wpe(arange(T))", pos)
    x = tok + pos;                                         sh("  x = tok + pos [broadcast]", x)
    print("  " + "-" * 44 + f"  <-- residual stream, {L} blocks, shape never changes")
    for i, blk in enumerate(m.transformer.h):
        x = blk(x)
        if i < 2 or i == L - 1:
            sh(f"  after block {i}", x)
        elif i == 2:
            print(f"  {'...':<26} (B, T, C)")
    x = m.transformer.ln_f(x);                             sh("  ln_f(x)", x)
    logits = m.lm_head(x);                                 sh("  lm_head(x)", logits)
    sh("  logits.view(-1, V)", logits.view(-1, V))
    sh("  targets.view(-1)", tgt.view(-1))
    loss = F.cross_entropy(logits.view(-1, V), tgt.view(-1))
    print(f"  {'cross_entropy':<26} ()  scalar = {loss.item():.4f}   (ln(V) = {math.log(V):.4f})")

print(f"\nlargest tensors in this forward pass:")
for name, n in sorted([("logits (B,T,V)", B*T*V), ("att (B,nh,T,T) x L", B*nh*T*T*L),
                       ("mlp hidden (B,T,4C) x L", B*T*4*C*L), ("x (B,T,C)", B*T*C),
                       ("params", m.get_num_params(False))], key=lambda r: -r[1]):
    print(f"  {name:<26} {n:>12,}  ({n*4/1e9:.2f} GB fp32)")

---
# Exercises

Roughly ordered by difficulty. Each is a real thing you'd do at work, and each fails in a way that teaches something.

**1. Break the mask on purpose.** Comment out the `masked_fill` and train 200 steps. **Predict the initial loss before you run it** (Part 8.5 tells you what to expect and what a break looks like). Explain the number you get.

**2. Break the head split on purpose.** Swap `.view(B,T,nh,hs).transpose(1,2)` for `.view(B,nh,T,hs)`. Train 500 steps. It won't crash and the loss will go down. How far down? Now explain *why it still learns anything at all.*

**3. Untie the weights.** Give `lm_head` its own `(V, C)`. Param count goes 30M → 49M. Does val loss improve enough to justify 63% more parameters? Try it at a fixed step count *and* at a fixed wall-clock budget — the answers differ, and that difference is the whole art of scaling.

**4. RoPE.** Replace `wpe` with rotary embeddings applied to `q` and `k` inside attention. `wpe` disappears entirely. **Trace the shapes first**: where does the `(T, hs/2)` frequency table broadcast against `(B, nh, T, hs)`? Then explain why RoPE extrapolates past `block_size` and learned absolute embeddings cannot.

**5. GQA.** Make `c_attn` output `C + 2*n_kv*hs` instead of `3C`. Use `repeat_interleave` on the KV head axis — or better, *don't*, and let the batched matmul broadcast it as in Part 4.5. Measure the KV cache shrink at `n_kv = 1, 2, 6` against the table in 10.3.

**6. Static KV cache.** Replace `torch.cat` with a pre-allocated `(B, nh, block_size, hs)` buffer and a write index. Removes the `O(T²)` copying. Then `torch.compile` the decode step — it should work now that the shape is static, and it should *not* have worked before. Explain why (Part 9.9).

**7. Speculative decoding.** Train a 2-layer draft model on the same data. Draft `k=4` tokens, verify all 4 in one target forward, accept the longest matching prefix. Measure acceptance rate and speedup. Then work out the break-even acceptance rate from Part 10.4's intensity argument — **you should be able to predict the speedup before measuring it.**

**8. Write the attention backward by hand.** Given `dL/dy`, derive `dL/dq`, `dL/dk`, `dL/dv` through the softmax. You already have the Jacobian from Part 4.7 — `diag(p) - p p^T` — so this is assembling pieces you've derived. Check against `torch.autograd.gradcheck` in float64. This is the prerequisite for reading FlashAttention, whose entire backward pass is about recomputing `att` from `q,k,v` in SRAM instead of storing it.

**9. Profile it.** `torch.profiler` on one training step. Which kernel dominates? Now run at `T=1024` and answer again. Explain the change using Part 9.8's FLOP breakdown.

**10. Reproduce a number.** Load real GPT-2 124M weights from HuggingFace into this code (you'll need `bias=True`, `n_embd=768`, `n_layer=12`, `n_head=12`, and the `transposed` list from Part 7.3's note about OpenAI's `Conv1D` convention). Match its logits on a fixed input to 1e-4. This is the single best test that you understand the architecture — every convention mismatch shows up as a wrong number and nothing else.

---

# Where to go next

You now have the shape intuition. The next layer down is *why the shapes cost what they cost*:

- **FlashAttention** (Dao et al.) — reread Part 4.5's memory table, then the paper. The whole contribution is refusing to materialise `(B, nh, T, T)` in HBM. Online softmax is the trick that makes it possible.
- **PagedAttention / vLLM** — reread Part 10.3's cache table and Part 10.4's ridge point. The paper is an OS virtual-memory design applied to the KV cache, and it exists because `B` is capped by cache fragmentation.
- **The `6N` rule** — Part 9.8. Every scaling-law and training-budget calculation starts there. Chinchilla is `6N * D` with `D ≈ 20N`.
- **Write the CUDA.** Fused softmax, then a tiled matmul, then a tiled attention kernel. The shapes in this notebook become tile indices; `(B, nh, T, hs)` becomes a grid launch. That's when the two halves meet.

**The through-line worth carrying away:** every architecture decision in this notebook is legible as a shape.

- Multi-head is a **reshape**.
- GQA is a **broadcast**.
- The KV cache is a **`cat` along axis 2**.
- FlashAttention is **refusing to allocate one tensor**.
- Speculative decoding is turning `(B, 1)` into `(B, k)` to buy arithmetic intensity.

Once the shapes are automatic, the papers read like commentary on tensors you can already see.